# Extended Pipeline v260430

In [ ]:
#!/usr/bin/env python3
"""
=============================================================================
 PSYCHIATRIC GENE-SET DIFFERENTIAL & PROXIMITY PIPELINE (TWAS / S-PrediXcan)
=============================================================================
 User-supplied axis space: any number of diseases / conditions.
 User supplies ONE custom gene set (txt file or comma-separated list),
 OR MULTIPLE gene sets.

 Analyses:
   1. Per-disease enrichment (Stouffer Z, mean |Z|, Wilcoxon vs 0,
      permutation p, bootstrap CIs)
   2. Differential tests across diseases (Kruskal-Wallis + pairwise
      Mann-Whitney + FDR)
   3. Profile proximity / distances between diseases (restricted to
      gene-set genes) + hierarchical clustering dendrogram
   4. Per-gene detail table with Z-scores across all diseases
   5. Pairwise disease statistical tests (gene-set-restricted):
        Unpaired : Mann-Whitney U, KS 2-sample, Welch's t, Levene,
                   Brunner-Munzel, Cohen's d
        Paired   : paired t, Wilcoxon signed-rank, Cohen's d, sign-flip
                   permutation
        Concordance: sign concordance rate + binomial test, Lin's CCC,
                     Kendall's tau, concordance/discordance counts
      with BH-FDR correction across all pairs.
   6. Gene-level leave-one-out (LOO) influence analysis to flag
      results driven by 1-2 outlier genes.
   7. Z-score QQ-plot diagnostics per disease.

 Output: CSVs, summary, boxplot, enrichment bar, proximity heatmap,
         scatter matrix, pairwise-stats heatmap, QQ-plot, dendrogram,
         gene-influence plot, input-provenance manifest.

 CLI usage:
   python geneset_pipeline.py \
       --dirs data/mdd/ data/bip/ data/ocd/ \
       --labels MDD BIP OCD \
       --geneset my_genes.txt --label "Dopamine_Receptors" --out results/

   python geneset_pipeline.py \
       --dirs data/mdd/ data/bip/ data/ocd/ \
       --labels MDD BIP OCD \
       --genesets_file genesets.py --out results/ \
       --random_seed 123 --robust --bootstrap

 Programmatic usage:
   from geneset_pipeline import run_pipeline, run_multi_geneset_pipeline

   results = run_pipeline(
       dirs=["data/mdd/", "data/bip/", "data/ocd/"],
       labels=["MDD", "BIP", "OCD"],
       geneset="DRD1,DRD2,DRD3,DRD4,DRD5,COMT,SLC6A3",
       label="Dopamine_System",
       out="results/",
       random_seed=42,
       robust=False,
       bootstrap=True,
       n_boot=2000,
   )
=============================================================================
"""

import pandas as pd
import numpy as np
from pathlib import Path
from scipy import stats
from itertools import combinations
import argparse, warnings, os, sys, textwrap, traceback
import hashlib
import json
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict
from datetime import datetime
import ast

warnings.filterwarnings("ignore")


# ===================================================================
#  0.  SMALL UTILITIES (provenance, validation, robust stats)
# ===================================================================

def _sha256_file(path, blocksize=65536):
    """Compute SHA256 of a file (short-form, first 16 hex chars)."""
    try:
        h = hashlib.sha256()
        with open(path, "rb") as fh:
            for chunk in iter(lambda: fh.read(blocksize), b""):
                h.update(chunk)
        return h.hexdigest()
    except Exception:
        return "ERROR"


def compute_input_provenance(dirs, geneset_input=None):
    """
    Build a provenance manifest containing SHA256 hashes for every
    input file plus timestamps.  Used for auditability/reproducibility.
    """
    manifest = {
        "generated_at": datetime.now().isoformat(timespec="seconds"),
        "python_version": sys.version.split()[0],
        "platform": sys.platform,
        "input_dirs": {},
        "geneset_input": None,
    }

    for d in dirs:
        d_path = Path(d)
        entry = {"path": str(d_path.resolve()), "exists": d_path.exists(), "files": {}}
        if d_path.is_dir():
            for f in sorted(d_path.iterdir()):
                if f.is_file():
                    entry["files"][f.name] = {
                        "sha256": _sha256_file(f),
                        "size_bytes": f.stat().st_size,
                        "mtime": datetime.fromtimestamp(f.stat().st_mtime).isoformat(timespec="seconds"),
                    }
        manifest["input_dirs"][str(d)] = entry

    if geneset_input is not None:
        try:
            if isinstance(geneset_input, (str, Path)) and Path(geneset_input).is_file():
                p = Path(geneset_input)
                manifest["geneset_input"] = {
                    "type": "file",
                    "path": str(p.resolve()),
                    "sha256": _sha256_file(p),
                    "size_bytes": p.stat().st_size,
                    "mtime": datetime.fromtimestamp(p.stat().st_mtime).isoformat(timespec="seconds"),
                }
            elif isinstance(geneset_input, (list, tuple, set)):
                manifest["geneset_input"] = {
                    "type": "iterable",
                    "n_items": len(list(geneset_input)),
                }
            elif isinstance(geneset_input, dict):
                manifest["geneset_input"] = {
                    "type": "dict",
                    "n_sets": len(geneset_input),
                    "labels": list(map(str, geneset_input.keys())),
                }
            else:
                manifest["geneset_input"] = {"type": "string", "preview": str(geneset_input)[:120]}
        except Exception as e:
            manifest["geneset_input"] = {"type": "unknown", "error": str(e)}
    return manifest


def validate_zscores(meta_z_dict, threshold_extreme=8.0, pct_warn=30.0):
    """
    Sanity check Z-score distributions per disease.

    Flags:
      * very high proportion of |Z| > threshold_extreme
      * duplicated keys (already deduped on load – noted for completeness)

    Returns list of warning strings.
    """
    warnings_list = []
    for d, mz in meta_z_dict.items():
        if not mz:
            warnings_list.append(f"{d}: empty meta-Z dictionary")
            continue
        zs = np.asarray(list(mz.values()), dtype=float)
        zs = zs[np.isfinite(zs)]
        if len(zs) == 0:
            warnings_list.append(f"{d}: no finite Z-scores")
            continue
        n_extreme = int(np.sum(np.abs(zs) > threshold_extreme))
        pct_extreme = 100 * n_extreme / len(zs)
        if pct_extreme > pct_warn:
            warnings_list.append(
                f"{d}: {pct_extreme:.1f}% of |Z| > {threshold_extreme} "
                f"(n={n_extreme}/{len(zs)}) — distribution may be inflated"
            )
        n_huge = int(np.sum(np.abs(zs) > 50))
        if n_huge > 0:
            warnings_list.append(
                f"{d}: {n_huge} extremely large |Z| > 50 detected — verify input"
            )
    return warnings_list


def _winsorize(arr, lower_pct=1.0, upper_pct=99.0):
    """Clip values to the [lower_pct, upper_pct] percentile range."""
    arr = np.asarray(arr, dtype=float)
    if len(arr) < 3:
        return arr
    lo = np.percentile(arr, lower_pct)
    hi = np.percentile(arr, upper_pct)
    return np.clip(arr, lo, hi)


def _trimmed_mean(arr, proportion=0.1):
    """Symmetric trimmed mean."""
    return float(stats.trim_mean(arr, proportion))


# ===================================================================
#  1.  GENERALISED LOADER  (returns meta_z dict, Ensembl↔symbol map,
#                            AND per-tissue Z dicts)
# ===================================================================

def load_spredixcan_folder(folder_path):
    """
    Load S-PrediXcan results from a folder.  Handles CSV / TSV / TXT.
    Reads BOTH gene-symbol and Ensembl-ID columns when available.
    Computes meta-Z across tissue files via Stouffer's method.

    Returns
    -------
    meta_z        : dict  {GENE_SYMBOL: meta_z_score}
    id_map        : dict  {ENSEMBL_ID: GENE_SYMBOL, ENSEMBL_NO_VER: GENE_SYMBOL, …}
    tissue_gene_z : dict  {tissue_filename_stem: {GENE_SYMBOL: z_score, …}, …}
    """
    folder = Path(folder_path)
    files = []
    for ext in ("*.csv", "*.txt", "*.tsv", "*.dat"):
        files.extend(folder.glob(ext))
    files = sorted(set(files))
    if not files:
        raise FileNotFoundError(f"No result files found in {folder_path}")

    print(f"    {len(files)} file(s) in {folder.name}/")

    gene_z = defaultdict(list)   # canonical_name → [z, z, …]
    tissue_gene_z = {}           # tissue_stem   → {canonical_name: z}
    id_map = {}                  # alternative_id  → canonical_name

    for f in files:
        tissue_name = f.stem
        tissue_dict = {}
        parsed = False
        for sep in [",", "\t", r"\s+"]:
            try:
                engine = "python" if sep == r"\s+" else "c"
                df = pd.read_csv(f, sep=sep, engine=engine)
                cols_lower = {c.lower(): c for c in df.columns}

                symbol_col = None
                for candidate in ["gene_name", "genename", "symbol", "gene_symbol"]:
                    if candidate in cols_lower:
                        symbol_col = cols_lower[candidate]
                        break

                id_col = None
                for candidate in ["gene", "gene_id", "geneid", "ensembl_id", "ensembl"]:
                    if candidate in cols_lower:
                        actual = cols_lower[candidate]
                        if actual != symbol_col:
                            id_col = actual
                            break

                primary_col = symbol_col or id_col

                z_col = None
                for candidate in ["zscore", "z_score", "z", "zstat", "z_stat"]:
                    if candidate in cols_lower:
                        z_col = cols_lower[candidate]
                        break

                if not (primary_col and z_col):
                    continue

                keep = list(dict.fromkeys(
                    [c for c in [primary_col, z_col, symbol_col, id_col] if c is not None]
                ))
                sub = df[keep].dropna(subset=[primary_col, z_col])

                # Detect duplicates BEFORE deduping
                n_dup = int(sub.duplicated(subset=primary_col).sum())
                if n_dup > 0:
                    print(f"    ⚠ {f.name}: {n_dup} duplicate '{primary_col}' rows "
                          f"(kept first occurrence)")
                sub = sub.drop_duplicates(subset=primary_col, keep="first")

                for _, row in sub.iterrows():
                    sym, ens = None, None
                    if symbol_col:
                        s = str(row[symbol_col]).strip().upper()
                        if s and s not in ("NAN", "NA", "NONE", ""):
                            sym = s
                    if id_col:
                        e = str(row[id_col]).strip().upper()
                        if e and e not in ("NAN", "NA", "NONE", ""):
                            ens = e

                    canonical = sym or ens
                    if not canonical:
                        continue
                    try:
                        z_val = float(row[z_col])
                        gene_z[canonical].append(z_val)
                        tissue_dict[canonical] = z_val
                    except (ValueError, TypeError):
                        continue

                    if ens and sym:
                        id_map[ens] = sym
                        if "." in ens:
                            id_map[ens.split(".")[0]] = sym

                parsed = True
                break
            except Exception:
                continue

        if parsed and tissue_dict:
            tissue_gene_z[tissue_name] = tissue_dict

        if not parsed:
            print(f"    ⚠ Could not parse {f.name}")

    meta = {}
    for g, zs in gene_z.items():
        if len(zs) > 0:
            meta[g] = np.sum(zs) / np.sqrt(len(zs))

    print(f"    → meta-Z computed for {len(meta):,} genes")
    print(f"    → Ensembl aliases mapped: {len(id_map):,}")
    print(f"    → {len(tissue_gene_z)} tissue file(s) indexed")
    return meta, id_map, tissue_gene_z


# ===================================================================
#  2.  GENE-SET I/O  (tabular-aware loader + Ensembl↔symbol harmoniser)
# ===================================================================

def _try_load_tabular_geneset(filepath):
    for sep in ["\t", ",", r"\s+"]:
        try:
            engine = "python" if sep == r"\s+" else "c"
            df = pd.read_csv(filepath, sep=sep, engine=engine)

            if len(df.columns) < 2:
                continue

            cols_lower = {c.lower(): c for c in df.columns}

            known_cols = {
                "gene_name", "genename", "symbol", "gene_symbol",
                "gene", "gene_id", "zscore", "z_score", "pvalue",
                "effect_size", "n_snps_used", "pred_perf_r2",
            }
            if not any(k in cols_lower for k in known_cols):
                continue

            symbol_col = None
            for c in ["gene_name", "genename", "symbol", "gene_symbol"]:
                if c in cols_lower:
                    symbol_col = cols_lower[c]
                    break

            id_col = None
            for c in ["gene", "gene_id", "geneid", "ensembl_id"]:
                if c in cols_lower:
                    if cols_lower[c] != symbol_col:
                        id_col = cols_lower[c]
                        break

            primary = symbol_col or id_col
            if primary is None:
                continue

            genes = []
            fallback_col = id_col if (symbol_col and id_col) else None

            for _, row in df.iterrows():
                val = str(row[primary]).strip().upper() if pd.notna(row[primary]) else ""
                if val and val not in ("NAN", "NA", "NONE", ""):
                    genes.append(val)
                elif fallback_col is not None:
                    val = str(row[fallback_col]).strip().upper() if pd.notna(row[fallback_col]) else ""
                    if val and val not in ("NAN", "NA", "NONE", ""):
                        genes.append(val)

            if genes:
                print(f"  Loaded {len(genes)} genes from tabular file "
                      f"(column: '{primary}'): {Path(filepath).name}")
                return genes

        except Exception:
            continue
    return None


def load_geneset(geneset_input):
    if isinstance(geneset_input, (list, tuple, set)):
        raw = [str(g).strip().upper() for g in geneset_input if str(g).strip()]
        print(f"  Loaded {len(raw)} genes from Python iterable")

    elif Path(geneset_input).is_file():
        raw = _try_load_tabular_geneset(geneset_input)
        if raw is None:
            with open(geneset_input) as fh:
                raw = [line.strip().upper() for line in fh
                       if line.strip() and not line.startswith("#")]
            print(f"  Loaded {len(raw)} genes from plain-text file: "
                  f"{Path(geneset_input).name}")
    else:
        raw = [g.strip().upper() for g in geneset_input.split(",") if g.strip()]
        print(f"  Loaded {len(raw)} genes from inline string")

    seen = set()
    unique = []
    for g in raw:
        if g not in seen:
            seen.add(g)
            unique.append(g)
    return unique


def load_multiple_genesets(genesets_input):
    if isinstance(genesets_input, dict):
        out = {}
        for label, gs in genesets_input.items():
            out[str(label)] = load_geneset(gs)
        return out

    path = Path(genesets_input)
    if not path.is_file():
        raise FileNotFoundError(f"Gene-set collection file not found: {genesets_input}")

    for sep in ["\t", ","]:
        try:
            df = pd.read_csv(path, sep=sep)
            cols = {c.lower(): c for c in df.columns}
            if "label" in cols and "genes" in cols:
                out = {}
                for _, row in df.iterrows():
                    label = str(row[cols["label"]]).strip()
                    genes = str(row[cols["genes"]]).strip()
                    if label and genes and label.upper() != "NAN" and genes.upper() != "NAN":
                        out[label] = load_geneset(genes)
                if out:
                    return out
        except Exception:
            pass

    text = path.read_text()

    try:
        tree = ast.parse(text, filename=str(path))
        for node in tree.body:
            if isinstance(node, ast.Assign):
                for target in node.targets:
                    if isinstance(target, ast.Name) and target.id == "gene_sets":
                        value = ast.literal_eval(node.value)
                        if isinstance(value, dict):
                            out = {}
                            for label, gs in value.items():
                                out[str(label)] = load_geneset(gs)
                            return out
    except Exception:
        pass

    out = {}
    namespace = {}
    try:
        exec(text, {}, namespace)
        for k, v in namespace.items():
            if k.startswith("_"):
                continue
            if isinstance(v, (list, tuple, set, str)):
                try:
                    out[k] = load_geneset(v)
                except Exception:
                    pass
        if out:
            return out
    except Exception:
        pass

    raise ValueError(
        "Could not parse multiple gene sets. "
        "Use a dict, a Python file with assignments, or a TSV/CSV with columns: label, genes."
    )


def harmonize_geneset(gene_list, meta_z_dict, id_maps, verbose=True):
    _print = print if verbose else lambda *a, **k: None

    all_canonical = set()
    for mz in meta_z_dict.values():
        all_canonical.update(mz.keys())

    merged = {}
    for im in id_maps.values():
        merged.update(im)

    harmonized = []
    direct = 0
    aliased = 0
    missed = []

    for g in gene_list:
        if g in all_canonical:
            harmonized.append(g)
            direct += 1
            continue
        if g in merged and merged[g] in all_canonical:
            harmonized.append(merged[g])
            aliased += 1
            continue
        if "." in g:
            base = g.split(".")[0]
            if base in all_canonical:
                harmonized.append(base)
                aliased += 1
                continue
            if base in merged and merged[base] in all_canonical:
                harmonized.append(merged[base])
                aliased += 1
                continue
        harmonized.append(g)
        missed.append(g)

    seen = set()
    unique = []
    for g in harmonized:
        if g not in seen:
            seen.add(g)
            unique.append(g)

    n_dup = len(harmonized) - len(unique)
    _print(f"  Gene-name harmonisation:")
    _print(f"    direct matches   : {direct}")
    _print(f"    alias-resolved   : {aliased}")
    _print(f"    still unmatched  : {len(missed)}")
    if n_dup:
        _print(f"    duplicates removed after aliasing: {n_dup}")
    if missed and len(missed) <= 20:
        _print(f"    unmatched genes  : {', '.join(missed)}")

    return unique


# ===================================================================
#  3.  ENRICHMENT ANALYSIS  (with bootstrap CIs + robust option)
# ===================================================================

def bootstrap_enrichment(zv, n_boot=2000, rng=None):
    """
    Bootstrap percentile 95% CIs for Stouffer Z, mean Z, mean |Z|.

    Returns dict with *_ci_low / *_ci_high keys (rounded to 4 d.p.).
    Returns NaNs for samples smaller than 3.
    """
    out = {
        "stouffer_ci_low":   np.nan, "stouffer_ci_high":   np.nan,
        "mean_z_ci_low":     np.nan, "mean_z_ci_high":     np.nan,
        "mean_abs_z_ci_low": np.nan, "mean_abs_z_ci_high": np.nan,
        "n_boot": n_boot,
    }
    zv = np.asarray(zv, dtype=float)
    zv = zv[np.isfinite(zv)]
    n = len(zv)
    if n < 3 or n_boot <= 0:
        return out

    if rng is None:
        rng = np.random.default_rng(42)

    boot_stouffer = np.empty(n_boot)
    boot_mean     = np.empty(n_boot)
    boot_mean_abs = np.empty(n_boot)
    sqrtn = np.sqrt(n)
    for i in range(n_boot):
        sample = rng.choice(zv, size=n, replace=True)
        boot_stouffer[i] = np.sum(sample) / sqrtn
        boot_mean[i]     = np.mean(sample)
        boot_mean_abs[i] = np.mean(np.abs(sample))

    out["stouffer_ci_low"]   = round(float(np.percentile(boot_stouffer, 2.5)), 4)
    out["stouffer_ci_high"]  = round(float(np.percentile(boot_stouffer, 97.5)), 4)
    out["mean_z_ci_low"]     = round(float(np.percentile(boot_mean, 2.5)), 4)
    out["mean_z_ci_high"]    = round(float(np.percentile(boot_mean, 97.5)), 4)
    out["mean_abs_z_ci_low"] = round(float(np.percentile(boot_mean_abs, 2.5)), 4)
    out["mean_abs_z_ci_high"]= round(float(np.percentile(boot_mean_abs, 97.5)), 4)
    return out


def gene_set_enrichment(meta_z, gene_list, n_perm=10000,
                        random_seed=42, robust=False,
                        bootstrap=False, n_boot=2000):
    """
    Enrichment statistics for one disease:
      - Stouffer Z
      - Mean |Z|, mean Z, median Z
      - Trimmed mean (always reported), winsorized Stouffer (if robust=True)
      - Wilcoxon signed-rank test vs 0
      - Permutation p-value (skipped if n_perm == 0)
      - Bootstrap 95% CIs (if bootstrap=True)
      - Reliability flags (low_n_warning, low_coverage_warning)
    """
    zv = np.array([meta_z.get(g, np.nan) for g in gene_list])
    zv = zv[~np.isnan(zv)]
    n = len(zv)
    n_total = len(gene_list)
    coverage = round(100 * n / n_total, 1) if n_total else 0

    result = {
        "n_genes_found": n,
        "n_genes_total": n_total,
        "coverage_pct": coverage,
        "stouffer_z": np.nan,
        "stouffer_z_winsorized": np.nan,
        "mean_abs_z": np.nan,
        "mean_z": np.nan,
        "trimmed_mean_z": np.nan,
        "median_z": np.nan,
        "wilcoxon_p": np.nan,
        "permutation_p": np.nan,
        "z_scores": [],
        # Bootstrap fields
        "stouffer_ci_low":   np.nan, "stouffer_ci_high":   np.nan,
        "mean_z_ci_low":     np.nan, "mean_z_ci_high":     np.nan,
        "mean_abs_z_ci_low": np.nan, "mean_abs_z_ci_high": np.nan,
        # Reliability flags
        "low_n_warning":         bool(n < 5),
        "low_coverage_warning":  bool(coverage < 60.0 and n_total > 0),
        "wilcoxon_unreliable":   bool(n < 6),
        "robust_used":           bool(robust),
    }

    if n < 3:
        return result

    # Optional winsorization for robust mode
    zv_for_stats = _winsorize(zv) if robust else zv

    stouffer = np.sum(zv_for_stats) / np.sqrt(n)
    result["stouffer_z"] = round(float(stouffer), 4)

    # Always compute winsorized Stouffer for reference
    zv_w = _winsorize(zv)
    result["stouffer_z_winsorized"] = round(float(np.sum(zv_w) / np.sqrt(n)), 4)

    result["mean_abs_z"]    = round(float(np.mean(np.abs(zv_for_stats))), 4)
    result["mean_z"]        = round(float(np.mean(zv_for_stats)), 4)
    result["median_z"]      = round(float(np.median(zv)), 4)
    try:
        result["trimmed_mean_z"] = round(_trimmed_mean(zv, proportion=0.1), 4)
    except Exception:
        pass
    result["z_scores"] = zv.tolist()

    if n >= 6:
        try:
            _, wil_p = stats.wilcoxon(zv, alternative="two-sided")
            result["wilcoxon_p"] = round(float(wil_p), 6)
        except Exception:
            pass

    all_z = np.array(list(meta_z.values()))
    if len(all_z) > n and n_perm > 0:
        obs_stat = abs(stouffer)
        rng_perm = np.random.default_rng(random_seed)
        count = 0
        for _ in range(n_perm):
            perm_z = rng_perm.choice(all_z, size=n, replace=False)
            if robust:
                perm_z = _winsorize(perm_z)
            if abs(np.sum(perm_z) / np.sqrt(n)) >= obs_stat:
                count += 1
        result["permutation_p"] = round((count + 1) / (n_perm + 1), 6)

    if bootstrap and n_boot > 0:
        rng_boot = np.random.default_rng(random_seed + 1)
        ci = bootstrap_enrichment(zv_for_stats, n_boot=n_boot, rng=rng_boot)
        for k in ("stouffer_ci_low", "stouffer_ci_high",
                  "mean_z_ci_low", "mean_z_ci_high",
                  "mean_abs_z_ci_low", "mean_abs_z_ci_high"):
            result[k] = ci[k]

    return result


# ===================================================================
#  3b.  GENE-LEVEL INFLUENCE ANALYSIS (LOO / jackknife)
# ===================================================================

def gene_influence_analysis(meta_z_dict, gene_list, threshold_pct=20.0):
    """
    Leave-one-out (LOO) influence analysis on the per-disease Stouffer Z
    and mean |Z|.

    For each disease and each gene present in the data, recompute the
    enrichment statistic without that gene and record the delta.

    A gene is flagged as "Influential" if removing it:
      - flips the sign of the Stouffer Z, OR
      - changes |Stouffer Z| by more than `threshold_pct` percent.

    Returns a tidy DataFrame (one row per gene × disease).
    """
    rows = []
    diseases = list(meta_z_dict.keys())
    for d in diseases:
        # Build the in-data Z vector for this disease
        zv = []
        gene_in_data = []
        for g in gene_list:
            z = meta_z_dict[d].get(g, np.nan)
            if not np.isnan(z):
                zv.append(z)
                gene_in_data.append(g)
        zv = np.asarray(zv, dtype=float)
        n = len(zv)
        if n < 4:
            continue

        full_stouffer = np.sum(zv) / np.sqrt(n)
        full_mean_abs = np.mean(np.abs(zv))

        for i, g in enumerate(gene_in_data):
            zv_loo = np.delete(zv, i)
            n_loo = len(zv_loo)
            loo_stouffer = np.sum(zv_loo) / np.sqrt(n_loo)
            loo_mean_abs = np.mean(np.abs(zv_loo))

            d_stouffer = full_stouffer - loo_stouffer
            d_mean_abs = full_mean_abs - loo_mean_abs

            sign_change = (np.sign(full_stouffer) != np.sign(loo_stouffer)
                           and full_stouffer != 0 and loo_stouffer != 0)
            pct_change = (abs(d_stouffer) / abs(full_stouffer) * 100
                          if full_stouffer != 0 else np.nan)
            influential = bool(sign_change or
                               (not np.isnan(pct_change) and pct_change > threshold_pct))

            rows.append({
                "Disease": d,
                "Gene": g,
                "Z_score": round(float(zv[i]), 4),
                "Full_Stouffer_Z": round(float(full_stouffer), 4),
                "LOO_Stouffer_Z":  round(float(loo_stouffer), 4),
                "Delta_Stouffer":  round(float(d_stouffer), 4),
                "Pct_Change_Stouffer": (round(float(pct_change), 2)
                                        if not np.isnan(pct_change) else np.nan),
                "Sign_Change": bool(sign_change),
                "Full_Mean_AbsZ":  round(float(full_mean_abs), 4),
                "LOO_Mean_AbsZ":   round(float(loo_mean_abs), 4),
                "Delta_Mean_AbsZ": round(float(d_mean_abs), 4),
                "Influential": influential,
            })

    df = pd.DataFrame(rows)
    if not df.empty:
        df = df.sort_values(["Disease", "Pct_Change_Stouffer"],
                            ascending=[True, False]).reset_index(drop=True)
    return df


# ===================================================================
#  4.  DIFFERENTIAL TESTS
# ===================================================================

def benjamini_hochberg(pvals):
    pvals = np.asarray(pvals, dtype=float)
    n = len(pvals)
    if n == 0:
        return np.array([])
    order = np.argsort(pvals)
    sorted_p = pvals[order]
    adjusted = np.empty(n)
    cum_min = 1.0
    for i in range(n - 1, -1, -1):
        adj = sorted_p[i] * n / (i + 1)
        cum_min = min(cum_min, adj)
        adjusted[order[i]] = min(cum_min, 1.0)
    return adjusted


def differential_tests(meta_z_dict, gene_list):
    diseases = list(meta_z_dict.keys())
    common = [g for g in gene_list if all(g in meta_z_dict[d] for d in diseases)]

    result = {
        "n_common_genes": len(common),
        "common_genes": common,
        "kruskal_h": np.nan,
        "kruskal_p": np.nan,
        "pairwise_df": pd.DataFrame()
    }

    if len(common) < 3:
        print("  ⚠ Fewer than 3 genes overlap across all diseases – skipping differential tests")
        return result

    z_by_disease = {d: np.array([meta_z_dict[d][g] for g in common]) for d in diseases}

    h_stat, kw_p = stats.kruskal(*z_by_disease.values())
    result["kruskal_h"] = round(h_stat, 4)
    result["kruskal_p"] = round(kw_p, 6)

    rows = []
    for d1, d2 in combinations(diseases, 2):
        v1, v2 = z_by_disease[d1], z_by_disease[d2]
        u_stat, mwu_p = stats.mannwhitneyu(v1, v2, alternative="two-sided")
        n1, n2 = len(v1), len(v2)
        rbc = 1 - (2 * u_stat) / (n1 * n2)

        try:
            _, paired_p = stats.wilcoxon(v1 - v2, alternative="two-sided")
            paired_p = round(paired_p, 6)
        except Exception:
            paired_p = np.nan

        rows.append({
            "Disease_1": d1,
            "Disease_2": d2,
            "MWU_U": int(u_stat),
            "MWU_p": round(mwu_p, 6),
            "Rank_Biserial_r": round(rbc, 4),
            "Paired_Wilcoxon_p": paired_p,
            "Mean_Z_diff": round(np.mean(v1) - np.mean(v2), 4)
        })

    pw_df = pd.DataFrame(rows)
    if len(pw_df) > 0:
        pw_df["MWU_FDR"] = benjamini_hochberg(pw_df["MWU_p"].values)
        pw_p = pw_df["Paired_Wilcoxon_p"].values
        if not np.any(np.isnan(pw_p)):
            pw_df["Paired_FDR"] = benjamini_hochberg(pw_p)
    result["pairwise_df"] = pw_df
    return result


# ===================================================================
#  4b.  PAIRWISE DISEASE STATISTICAL TESTS  (gene-set-restricted)
# ===================================================================

def _lins_ccc(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    if len(x) < 3:
        return np.nan
    mx, my = np.mean(x), np.mean(y)
    sx, sy = np.std(x, ddof=1), np.std(y, ddof=1)
    if sx == 0 or sy == 0:
        return np.nan
    r, _ = stats.pearsonr(x, y)
    numerator = 2 * r * sx * sy
    denominator = sx ** 2 + sy ** 2 + (mx - my) ** 2
    if denominator == 0:
        return np.nan
    return numerator / denominator


def pairwise_disease_geneset_tests(meta_z_dict, gene_list,
                                   n_perm=5000, random_seed=42, robust=False):
    """
    Comprehensive pairwise statistical tests between diseases,
    restricted to gene-set genes.

    With `robust=True`, Z-vectors are winsorized (1st/99th percentile)
    before parametric (t-test, Cohen's d, Levene) computations.
    Non-parametric tests are unaffected by winsorization.
    """
    diseases = list(meta_z_dict.keys())
    if len(diseases) < 2:
        return pd.DataFrame()

    rows = []
    for d1, d2 in combinations(diseases, 2):
        z1_all = np.array([meta_z_dict[d1][g] for g in gene_list
                           if g in meta_z_dict[d1]
                           and np.isfinite(meta_z_dict[d1][g])])
        z2_all = np.array([meta_z_dict[d2][g] for g in gene_list
                           if g in meta_z_dict[d2]
                           and np.isfinite(meta_z_dict[d2][g])])

        common = [g for g in gene_list
                  if g in meta_z_dict[d1] and g in meta_z_dict[d2]
                  and np.isfinite(meta_z_dict[d1][g])
                  and np.isfinite(meta_z_dict[d2][g])]
        z1_p = np.array([meta_z_dict[d1][g] for g in common])
        z2_p = np.array([meta_z_dict[d2][g] for g in common])

        # Optional robust transformation for parametric tests
        z1_all_r = _winsorize(z1_all) if robust else z1_all
        z2_all_r = _winsorize(z2_all) if robust else z2_all
        z1_p_r   = _winsorize(z1_p)   if robust else z1_p
        z2_p_r   = _winsorize(z2_p)   if robust else z2_p

        row = {
            "Disease_1": d1,
            "Disease_2": d2,
            "n_genes_D1": len(z1_all),
            "n_genes_D2": len(z2_all),
            "n_genes_paired": len(common),
            "mean_Z_D1":   round(float(np.mean(z1_all)),   4) if len(z1_all) else np.nan,
            "mean_Z_D2":   round(float(np.mean(z2_all)),   4) if len(z2_all) else np.nan,
            "sd_Z_D1":     round(float(np.std(z1_all, ddof=1)), 4) if len(z1_all) > 1 else np.nan,
            "sd_Z_D2":     round(float(np.std(z2_all, ddof=1)), 4) if len(z2_all) > 1 else np.nan,
            "median_Z_D1": round(float(np.median(z1_all)), 4) if len(z1_all) else np.nan,
            "median_Z_D2": round(float(np.median(z2_all)), 4) if len(z2_all) else np.nan,
        }

        if len(common) >= 1:
            diffs = z1_p - z2_p
            diffs_r = z1_p_r - z2_p_r
            row["mean_diff"]   = round(float(np.mean(diffs)),   4)
            row["median_diff"] = round(float(np.median(diffs)), 4)
            row["sd_diff"]     = round(float(np.std(diffs, ddof=1)), 4) if len(common) > 1 else np.nan
        else:
            diffs = np.array([])
            diffs_r = np.array([])
            row["mean_diff"]   = np.nan
            row["median_diff"] = np.nan
            row["sd_diff"]     = np.nan

        _enough_unpaired = len(z1_all) >= 3 and len(z2_all) >= 3

        row["MWU_U"] = np.nan; row["MWU_p"] = np.nan; row["Rank_Biserial_r"] = np.nan
        if _enough_unpaired:
            try:
                u, p = stats.mannwhitneyu(z1_all, z2_all, alternative="two-sided")
                n1, n2 = len(z1_all), len(z2_all)
                row["MWU_U"] = int(u)
                row["MWU_p"] = round(p, 6)
                row["Rank_Biserial_r"] = round(1 - 2 * u / (n1 * n2), 4)
            except Exception:
                pass

        row["KS_stat"] = np.nan; row["KS_p"] = np.nan
        if _enough_unpaired:
            try:
                ks, p = stats.ks_2samp(z1_all, z2_all)
                row["KS_stat"] = round(ks, 4)
                row["KS_p"]    = round(p, 6)
            except Exception:
                pass

        row["Welch_t"] = np.nan; row["Welch_p"] = np.nan
        if _enough_unpaired:
            try:
                t, p = stats.ttest_ind(z1_all_r, z2_all_r, equal_var=False)
                row["Welch_t"] = round(t, 4)
                row["Welch_p"] = round(p, 6)
            except Exception:
                pass

        row["Levene_W"] = np.nan; row["Levene_p"] = np.nan
        if _enough_unpaired:
            try:
                w, p = stats.levene(z1_all_r, z2_all_r)
                row["Levene_W"] = round(w, 4)
                row["Levene_p"] = round(p, 6)
            except Exception:
                pass

        row["BM_W"] = np.nan; row["BM_p"] = np.nan
        if _enough_unpaired:
            try:
                bm, p = stats.brunnermunzel(z1_all, z2_all)
                row["BM_W"] = round(bm, 4)
                row["BM_p"] = round(p, 6)
            except Exception:
                pass

        row["Cohens_d_unpaired"] = np.nan
        if _enough_unpaired:
            try:
                s1 = np.var(z1_all_r, ddof=1)
                s2 = np.var(z2_all_r, ddof=1)
                pooled = np.sqrt((s1 + s2) / 2)
                if pooled > 0:
                    row["Cohens_d_unpaired"] = round(
                        float((np.mean(z1_all_r) - np.mean(z2_all_r)) / pooled), 4
                    )
            except Exception:
                pass

        row["Paired_t_stat"] = np.nan; row["Paired_t_p"] = np.nan
        row["Paired_Wilcoxon_W"] = np.nan; row["Paired_Wilcoxon_p"] = np.nan
        row["Cohens_d_paired"] = np.nan
        row["Permutation_p"] = np.nan

        if len(common) >= 3:
            try:
                t, p = stats.ttest_rel(z1_p_r, z2_p_r)
                row["Paired_t_stat"] = round(t, 4)
                row["Paired_t_p"]    = round(p, 6)
            except Exception:
                pass

            if len(common) >= 6:
                try:
                    w, p = stats.wilcoxon(diffs, alternative="two-sided")
                    row["Paired_Wilcoxon_W"] = int(w)
                    row["Paired_Wilcoxon_p"] = round(p, 6)
                except Exception:
                    pass

            try:
                sd_d = np.std(diffs_r, ddof=1)
                if sd_d > 0:
                    row["Cohens_d_paired"] = round(float(np.mean(diffs_r) / sd_d), 4)
            except Exception:
                pass

            if n_perm > 0:
                try:
                    obs = abs(np.mean(diffs))
                    rng = np.random.default_rng(random_seed)
                    count = 0
                    for _ in range(n_perm):
                        signs = rng.choice([-1, 1], size=len(diffs))
                        if abs(np.mean(diffs * signs)) >= obs:
                            count += 1
                    row["Permutation_p"] = round((count + 1) / (n_perm + 1), 6)
                except Exception:
                    pass

        row["n_concordant"]          = np.nan
        row["n_discordant"]          = np.nan
        row["n_zero_either"]         = np.nan
        row["sign_concordance_rate"] = np.nan
        row["sign_concordance_p"]    = np.nan
        row["Lin_CCC"]               = np.nan
        row["Kendall_tau"]           = np.nan
        row["Kendall_p"]             = np.nan

        if len(common) >= 3:
            try:
                signs1 = np.sign(z1_p)
                signs2 = np.sign(z2_p)
                zero_mask = (signs1 == 0) | (signs2 == 0)
                n_zero = int(np.sum(zero_mask))
                row["n_zero_either"] = n_zero
                nz_mask = ~zero_mask
                s1_nz = signs1[nz_mask]
                s2_nz = signs2[nz_mask]
                n_nz = len(s1_nz)
                if n_nz >= 1:
                    concordant_mask = (s1_nz == s2_nz)
                    n_conc = int(np.sum(concordant_mask))
                    n_disc = n_nz - n_conc
                    row["n_concordant"] = n_conc
                    row["n_discordant"] = n_disc
                    row["sign_concordance_rate"] = round(n_conc / n_nz, 4)
                    if n_nz >= 3:
                        try:
                            binom_p = stats.binomtest(n_conc, n_nz, 0.5).pvalue
                        except AttributeError:
                            binom_p = stats.binom_test(n_conc, n_nz, 0.5)
                        row["sign_concordance_p"] = round(float(binom_p), 6)
            except Exception:
                pass

            try:
                ccc = _lins_ccc(z1_p, z2_p)
                if not np.isnan(ccc):
                    row["Lin_CCC"] = round(float(ccc), 4)
            except Exception:
                pass

            try:
                tau, tau_p = stats.kendalltau(z1_p, z2_p)
                row["Kendall_tau"] = round(float(tau), 4)
                row["Kendall_p"]   = round(float(tau_p), 6)
            except Exception:
                pass

        rows.append(row)

    df = pd.DataFrame(rows)
    if df.empty:
        return df

    p_cols = [c for c in df.columns if c.endswith("_p")]
    for pc in p_cols:
        vals = df[pc].values.astype(float)
        mask = ~np.isnan(vals)
        if mask.sum() > 1:
            fdr_vals = np.full(len(vals), np.nan)
            fdr_vals[mask] = benjamini_hochberg(vals[mask])
            df[pc.replace("_p", "_FDR")] = fdr_vals
        elif mask.sum() == 1:
            df[pc.replace("_p", "_FDR")] = vals

    return df


# ===================================================================
#  5.  PROXIMITY / DISTANCE
# ===================================================================

def profile_proximity(meta_z_dict, gene_list):
    diseases = list(meta_z_dict.keys())
    common = [g for g in gene_list if all(g in meta_z_dict[d] for d in diseases)]

    if len(common) < 5:
        print("  ⚠ Fewer than 5 overlapping genes – proximity metrics unreliable")
    if len(common) < 3:
        return pd.DataFrame(), pd.DataFrame()

    profile_matrix = pd.DataFrame(
        {d: [meta_z_dict[d][g] for g in common] for d in diseases},
        index=common
    )

    rows = []
    for d1, d2 in combinations(diseases, 2):
        v1 = profile_matrix[d1].values
        v2 = profile_matrix[d2].values

        pr, pr_p = stats.pearsonr(v1, v2)
        sr, sr_p = stats.spearmanr(v1, v2)
        cos_sim = np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2) + 1e-15)
        eucl = np.linalg.norm(v1 - v2)
        manhattan = np.sum(np.abs(v1 - v2))

        rows.append({
            "Disease_1": d1,
            "Disease_2": d2,
            "Pearson_r": round(pr, 4),
            "Pearson_p": round(pr_p, 6),
            "Spearman_rho": round(sr, 4),
            "Spearman_p": round(sr_p, 6),
            "Cosine_sim": round(cos_sim, 4),
            "Euclidean_dist": round(eucl, 4),
            "Manhattan_dist": round(manhattan, 4),
            "n_genes": len(common)
        })

    return pd.DataFrame(rows), profile_matrix


# ===================================================================
#  6.  PER-GENE DETAIL TABLE
# ===================================================================

def build_gene_detail_table(meta_z_dict, gene_list):
    diseases = list(meta_z_dict.keys())
    rows = []
    for g in gene_list:
        row = {"Gene": g}
        zvals = []
        for d in diseases:
            z = meta_z_dict[d].get(g, np.nan)
            row[f"Z_{d}"] = round(z, 4) if not np.isnan(z) else np.nan
            if not np.isnan(z):
                zvals.append(z)
        row["N_diseases"] = len(zvals)
        if len(zvals) >= 2:
            row["Z_range"] = round(max(zvals) - min(zvals), 4)
            row["Z_mean"] = round(np.mean(zvals), 4)
        else:
            row["Z_range"] = np.nan
            row["Z_mean"] = np.nan
        rows.append(row)

    df = pd.DataFrame(rows)
    sort_cols = [f"Z_{d}" for d in diseases]
    df["_sort"] = df[sort_cols].abs().mean(axis=1)
    df = df.sort_values("_sort", ascending=False).drop(columns="_sort").reset_index(drop=True)
    return df


# ===================================================================
#  7.  VISUALISATION
# ===================================================================

def _make_disease_colors(diseases):
    _LEGACY = {"MDD": "#e74c3c", "BIP": "#3498db", "OCD": "#2ecc71"}
    pal = sns.color_palette("husl", max(len(diseases), 3))
    colors = {}
    pal_idx = 0
    for d in diseases:
        if d in _LEGACY:
            colors[d] = _LEGACY[d]
        else:
            colors[d] = pal[pal_idx]
            pal_idx += 1
    return colors


def plot_zscore_boxplot(enrichment_dict, label, out_dir):
    data = []
    for dis, info in enrichment_dict.items():
        for z in info["z_scores"]:
            data.append({"Disease": dis, "meta-Z": z})
    if not data:
        return
    df = pd.DataFrame(data)

    order = list(enrichment_dict.keys())
    dcol = _make_disease_colors(order)
    colors = [dcol.get(d, "#999") for d in order]

    fig, ax = plt.subplots(figsize=(max(7, len(order) * 2), 5))

    sns.boxplot(data=df, x="Disease", y="meta-Z", order=order,
                palette=colors, width=0.5, ax=ax)
    sns.stripplot(data=df, x="Disease", y="meta-Z", order=order,
                  color=".25", size=3, alpha=0.6, ax=ax)
    ax.axhline(0, color="gray", ls="--", lw=0.8)
    ax.set_title(f"Gene-set meta-Z Distribution – {label}", fontsize=13)
    ax.set_ylabel("meta-Z (TWAS association)")

    for i, d in enumerate(order):
        n = enrichment_dict.get(d, {}).get("n_genes_found", 0)
        ax.text(i, ax.get_ylim()[0], f"n={n}", ha="center", va="top",
                fontsize=9, color="gray")

    plt.tight_layout()
    path = os.path.join(out_dir, f"zscore_boxplot_{label}.png")
    plt.savefig(path, dpi=300)
    plt.close()
    print(f"  → {path}")


def plot_enrichment_bar(enrichment_dict, label, out_dir):
    order = list(enrichment_dict.keys())
    dcol = _make_disease_colors(order)

    rows = []
    for d in order:
        rows.append({
            "Disease": d,
            "Stouffer Z": enrichment_dict[d].get("stouffer_z", 0) or 0,
            "Mean |Z|":   enrichment_dict[d].get("mean_abs_z", 0) or 0,
            "Stouffer_lo": enrichment_dict[d].get("stouffer_ci_low", np.nan),
            "Stouffer_hi": enrichment_dict[d].get("stouffer_ci_high", np.nan),
            "MeanAbs_lo":  enrichment_dict[d].get("mean_abs_z_ci_low", np.nan),
            "MeanAbs_hi":  enrichment_dict[d].get("mean_abs_z_ci_high", np.nan),
        })
    if not rows:
        return
    df = pd.DataFrame(rows)
    colors = [dcol.get(d, "#999") for d in df["Disease"]]

    fig, axes = plt.subplots(1, 2, figsize=(max(10, len(order) * 2.5), 5))

    bars0 = axes[0].bar(df["Disease"], df["Stouffer Z"], color=colors, edgecolor="white")
    # Draw bootstrap CI error bars when available
    if df["Stouffer_lo"].notna().any():
        for b, lo, hi, val in zip(bars0, df["Stouffer_lo"], df["Stouffer_hi"], df["Stouffer Z"]):
            if not (np.isnan(lo) or np.isnan(hi)):
                axes[0].plot([b.get_x() + b.get_width() / 2] * 2, [lo, hi],
                             color="black", lw=1.2)
                axes[0].plot([b.get_x() + b.get_width() / 2 - 0.08,
                              b.get_x() + b.get_width() / 2 + 0.08], [lo, lo],
                             color="black", lw=1.2)
                axes[0].plot([b.get_x() + b.get_width() / 2 - 0.08,
                              b.get_x() + b.get_width() / 2 + 0.08], [hi, hi],
                             color="black", lw=1.2)

    axes[0].axhline(0, color="gray", ls="--", lw=0.8)
    axes[0].axhline(1.96, color="red", ls=":", lw=0.7, alpha=0.5)
    axes[0].axhline(-1.96, color="red", ls=":", lw=0.7, alpha=0.5)
    axes[0].set_title("Stouffer Z (with 95% bootstrap CI if available)")
    axes[0].set_ylabel("Combined Z-score")

    bars1 = axes[1].bar(df["Disease"], df["Mean |Z|"], color=colors, edgecolor="white")
    if df["MeanAbs_lo"].notna().any():
        for b, lo, hi in zip(bars1, df["MeanAbs_lo"], df["MeanAbs_hi"]):
            if not (np.isnan(lo) or np.isnan(hi)):
                axes[1].plot([b.get_x() + b.get_width() / 2] * 2, [lo, hi],
                             color="black", lw=1.2)
    axes[1].set_title("Mean |Z|")
    axes[1].set_ylabel("Mean absolute Z-score")

    fig.suptitle(f"Gene-set Enrichment – {label}", fontsize=14)
    plt.tight_layout()
    path = os.path.join(out_dir, f"enrichment_bar_{label}.png")
    plt.savefig(path, dpi=300)
    plt.close()
    print(f"  → {path}")


def plot_proximity_heatmap(prox_df, diseases, label, out_dir):
    if prox_df.empty:
        return

    fig, axes = plt.subplots(1, 2, figsize=(max(12, len(diseases) * 3), max(5, len(diseases) * 2)))
    for ax, metric, title in zip(axes,
                                  ["Pearson_r", "Cosine_sim"],
                                  ["Pearson r", "Cosine Similarity"]):
        mat = pd.DataFrame(np.eye(len(diseases)), index=diseases, columns=diseases)
        for _, row in prox_df.iterrows():
            d1, d2 = row["Disease_1"], row["Disease_2"]
            val = row[metric]
            if d1 in diseases and d2 in diseases:
                mat.loc[d1, d2] = val
                mat.loc[d2, d1] = val

        sns.heatmap(mat.astype(float), annot=True, fmt=".3f", cmap="RdBu_r",
                    center=0, vmin=-1, vmax=1, square=True, ax=ax,
                    linewidths=0.5, linecolor="white")
        ax.set_title(title)

    fig.suptitle(f"Profile Proximity – {label}", fontsize=14)
    plt.tight_layout()
    path = os.path.join(out_dir, f"proximity_heatmap_{label}.png")
    plt.savefig(path, dpi=300)
    plt.close()
    print(f"  → {path}")


def plot_proximity_dendrogram(prox_df, diseases, label, out_dir):
    """
    Hierarchical clustering of diseases based on (1 - Pearson r) distance.
    Requires at least 3 diseases.
    """
    if prox_df.empty or len(diseases) < 3:
        return
    try:
        from scipy.cluster.hierarchy import linkage, dendrogram
        from scipy.spatial.distance import squareform
    except Exception as e:
        print(f"  ⚠ scipy clustering unavailable: {e}")
        return

    n = len(diseases)
    dmat = np.zeros((n, n))
    idx = {d: i for i, d in enumerate(diseases)}
    for _, row in prox_df.iterrows():
        d1, d2 = row["Disease_1"], row["Disease_2"]
        if d1 in idx and d2 in idx:
            dist = max(0.0, 1 - float(row["Pearson_r"]))
            dmat[idx[d1], idx[d2]] = dist
            dmat[idx[d2], idx[d1]] = dist

    try:
        condensed = squareform(dmat, checks=False)
        Z = linkage(condensed, method="average")
        fig, ax = plt.subplots(figsize=(max(6, n * 1.3), 5))
        dendrogram(Z, labels=diseases, ax=ax, color_threshold=0)
        ax.set_ylabel("Distance (1 − Pearson r)")
        ax.set_title(f"Disease Hierarchical Clustering – {label}", fontsize=12, fontweight="bold")
        plt.tight_layout()
        path = os.path.join(out_dir, f"proximity_dendrogram_{label}.png")
        plt.savefig(path, dpi=300)
        plt.close()
        print(f"  → {path}")
    except Exception as e:
        print(f"  ⚠ Dendrogram failed: {e}")


def plot_zscore_qq(meta_z_dict, gene_list, label, out_dir):
    """
    Per-disease QQ-plot: gene-set Z-scores vs N(0,1) theoretical quantiles.
    Useful sanity check for calibration.
    """
    diseases = list(meta_z_dict.keys())
    n = len(diseases)
    if n == 0:
        return
    fig, axes = plt.subplots(1, n, figsize=(max(5 * n, 6), 5))
    if n == 1:
        axes = [axes]
    for ax, d in zip(axes, diseases):
        gs_z = np.array([meta_z_dict[d].get(g, np.nan) for g in gene_list])
        gs_z = gs_z[~np.isnan(gs_z)]
        if len(gs_z) < 3:
            ax.text(0.5, 0.5, f"{d}: n<3", ha="center", va="center",
                    transform=ax.transAxes, fontsize=11)
            ax.set_xticks([]); ax.set_yticks([])
            continue
        observed = np.sort(gs_z)
        theoretical = stats.norm.ppf((np.arange(1, len(observed) + 1) - 0.5) / len(observed))
        ax.scatter(theoretical, observed, alpha=0.7, s=30,
                   c="steelblue", edgecolor="black", lw=0.3)
        lims = [min(theoretical.min(), observed.min()),
                max(theoretical.max(), observed.max())]
        ax.plot(lims, lims, "r--", lw=1, label="y = x (N(0,1))")
        ax.axhline(0, color="gray", lw=0.5, ls=":")
        ax.axvline(0, color="gray", lw=0.5, ls=":")
        ax.set_xlabel("Theoretical N(0,1) quantile")
        ax.set_ylabel("Observed gene-set meta-Z")
        ax.set_title(f"{d} (n={len(observed)})", fontsize=11)
        ax.legend(fontsize=8, loc="upper left")
    fig.suptitle(f"Z-score QQ-plots (gene set vs N(0,1)) – {label}",
                 fontsize=13, fontweight="bold")
    plt.tight_layout()
    path = os.path.join(out_dir, f"qq_plot_{label}.png")
    plt.savefig(path, dpi=300)
    plt.close()
    print(f"  → {path}")


def plot_gene_influence(influence_df, label, out_dir, top_n=15):
    """
    Per-disease lollipop / bar plot of the top-|ΔStouffer| genes.
    Influential genes (sign-flip or > threshold% change) are highlighted in red.
    """
    if influence_df.empty:
        return
    diseases = list(influence_df["Disease"].unique())
    n = len(diseases)
    fig, axes = plt.subplots(1, n,
        figsize=(max(6 * n, 8), max(5, top_n * 0.35)))
    if n == 1:
        axes = [axes]
    for ax, d in zip(axes, diseases):
        sub = influence_df[influence_df["Disease"] == d].copy()
        sub = sub.reindex(sub["Delta_Stouffer"].abs().sort_values(ascending=False).index)
        sub = sub.head(top_n).iloc[::-1]
        if sub.empty:
            ax.text(0.5, 0.5, f"{d}: insufficient data",
                    ha="center", va="center", transform=ax.transAxes)
            continue
        colors = ["#e74c3c" if x else "#3498db" for x in sub["Influential"]]
        ax.barh(sub["Gene"], sub["Delta_Stouffer"], color=colors,
                edgecolor="black", lw=0.3)
        ax.axvline(0, color="gray", ls="--", lw=0.8)
        ax.set_xlabel("Δ Stouffer Z (full − LOO)")
        ax.set_title(f"{d}: top influential genes", fontsize=11)
    fig.suptitle(f"Gene-level Leave-One-Out Influence – {label}",
                 fontsize=13, fontweight="bold")
    plt.tight_layout()
    path = os.path.join(out_dir, f"gene_influence_{label}.png")
    plt.savefig(path, dpi=300)
    plt.close()
    print(f"  → {path}")


def plot_scatter_matrix(profile_matrix, label, out_dir):
    if profile_matrix.empty:
        return

    diseases = list(profile_matrix.columns)
    pairs = list(combinations(diseases, 2))
    n_pairs = len(pairs)
    if n_pairs == 0:
        return

    fig, axes = plt.subplots(1, n_pairs, figsize=(5 * n_pairs, 5))
    if n_pairs == 1:
        axes = [axes]

    for ax, (d1, d2) in zip(axes, pairs):
        v1 = profile_matrix[d1].values
        v2 = profile_matrix[d2].values
        ax.scatter(v1, v2, alpha=0.5, s=20, color="#555")

        slope, intercept = np.polyfit(v1, v2, 1)
        xr = np.linspace(v1.min(), v1.max(), 100)
        ax.plot(xr, slope * xr + intercept, color="red", lw=1.5, ls="--")

        r, _ = stats.pearsonr(v1, v2)
        ax.set_xlabel(f"{d1} meta-Z")
        ax.set_ylabel(f"{d2} meta-Z")
        ax.set_title(f"{d1} vs {d2}  (r = {r:.3f})")
        ax.axhline(0, color="gray", ls=":", lw=0.5)
        ax.axvline(0, color="gray", ls=":", lw=0.5)

    fig.suptitle(f"Cross-Disease Z-score Scatter – {label}", fontsize=14)
    plt.tight_layout()
    path = os.path.join(out_dir, f"scatter_matrix_{label}.png")
    plt.savefig(path, dpi=300)
    plt.close()
    print(f"  → {path}")


def plot_pairwise_stats_heatmap(pw_df, diseases, label, out_dir):
    if pw_df.empty or len(diseases) < 2:
        return

    p_candidates = [
        ("KS_p",               "KS 2-sample"),
        ("Welch_p",            "Welch t"),
        ("Paired_Wilcoxon_p",  "Paired Wilcoxon"),
        ("Paired_t_p",         "Paired t"),
        ("Permutation_p",      "Permutation"),
        ("sign_concordance_p", "Sign Concordance"),
        ("Kendall_p",          "Kendall τ"),
    ]
    p_metrics = [(col, lbl) for col, lbl in p_candidates
                 if col in pw_df.columns and pw_df[col].notna().any()]

    d_candidates = [
        ("Cohens_d_unpaired", "Cohen's d (unpaired)"),
        ("Cohens_d_paired",   "Cohen's d (paired)"),
    ]
    d_metrics = [(col, lbl) for col, lbl in d_candidates
                 if col in pw_df.columns and pw_df[col].notna().any()]

    conc_candidates = [
        ("sign_concordance_rate", "Sign Concordance Rate"),
        ("Lin_CCC",               "Lin's CCC"),
        ("Kendall_tau",            "Kendall τ"),
    ]
    conc_metrics = [(col, lbl) for col, lbl in conc_candidates
                    if col in pw_df.columns and pw_df[col].notna().any()]

    n_panels = len(p_metrics) + len(d_metrics) + len(conc_metrics)
    if n_panels == 0:
        return

    fig, axes = plt.subplots(
        1, n_panels,
        figsize=(max(5 * n_panels, 10), max(4, len(diseases) * 1.5))
    )
    if n_panels == 1:
        axes = [axes]

    panel_idx = 0

    for col, lbl in p_metrics:
        ax = axes[panel_idx]; panel_idx += 1
        mat = pd.DataFrame(np.nan, index=diseases, columns=diseases)
        for _, row in pw_df.iterrows():
            d1, d2 = row["Disease_1"], row["Disease_2"]
            val = row[col]
            if d1 in diseases and d2 in diseases and not np.isnan(val):
                neg_log = -np.log10(max(val, 1e-300))
                mat.loc[d1, d2] = neg_log
                mat.loc[d2, d1] = neg_log
        for d in diseases:
            mat.loc[d, d] = 0.0

        sns.heatmap(mat.astype(float), annot=True, fmt=".2f", cmap="YlOrRd",
                    square=True, ax=ax, linewidths=0.5, linecolor="white",
                    vmin=0)
        ax.set_title(f"−log₁₀(p)\n{lbl}", fontsize=10)

    for col, lbl in d_metrics:
        ax = axes[panel_idx]; panel_idx += 1
        mat = pd.DataFrame(0.0, index=diseases, columns=diseases)
        for _, row in pw_df.iterrows():
            d1, d2 = row["Disease_1"], row["Disease_2"]
            val = row[col]
            if d1 in diseases and d2 in diseases and not np.isnan(val):
                mat.loc[d1, d2] =  val
                mat.loc[d2, d1] = -val

        vabs = max(abs(mat.values.min()), abs(mat.values.max()), 0.01)
        sns.heatmap(mat.astype(float), annot=True, fmt=".2f", cmap="RdBu_r",
                    center=0, vmin=-vabs, vmax=vabs, square=True, ax=ax,
                    linewidths=0.5, linecolor="white")
        ax.set_title(lbl, fontsize=10)

    for col, lbl in conc_metrics:
        ax = axes[panel_idx]; panel_idx += 1
        if col == "sign_concordance_rate":
            diag_val = 1.0
            vmin_val, vmax_val, center_val = 0.0, 1.0, 0.5
            cmap = "YlGn"
        elif col == "Lin_CCC":
            diag_val = 1.0
            vmin_val, vmax_val, center_val = -1.0, 1.0, 0.0
            cmap = "RdBu_r"
        elif col == "Kendall_tau":
            diag_val = 1.0
            vmin_val, vmax_val, center_val = -1.0, 1.0, 0.0
            cmap = "RdBu_r"
        else:
            diag_val = 1.0
            vmin_val, vmax_val, center_val = -1.0, 1.0, 0.0
            cmap = "RdBu_r"

        mat = pd.DataFrame(diag_val, index=diseases, columns=diseases)
        for _, row in pw_df.iterrows():
            d1, d2 = row["Disease_1"], row["Disease_2"]
            val = row[col]
            if d1 in diseases and d2 in diseases and not np.isnan(val):
                mat.loc[d1, d2] = val
                mat.loc[d2, d1] = val

        sns.heatmap(mat.astype(float), annot=True, fmt=".3f", cmap=cmap,
                    center=center_val, vmin=vmin_val, vmax=vmax_val,
                    square=True, ax=ax,
                    linewidths=0.5, linecolor="white")
        ax.set_title(lbl, fontsize=10)

    fig.suptitle(f"Pairwise Disease Tests – {label}", fontsize=14, fontweight="bold")
    plt.tight_layout()
    path = os.path.join(out_dir, f"pairwise_stats_heatmap_{label}.png")
    plt.savefig(path, dpi=300)
    plt.close()
    print(f"  → {path}")


# ===================================================================
#  8.  CORE PIPELINE FUNCTION (callable directly)
# ===================================================================

# Static block appended to every summary file
_LIMITATIONS_BLOCK = textwrap.dedent("""\
    LIMITATIONS & ASSUMPTIONS
    -------------------------
    * Stouffer's method assumes (approximately) independent gene-level
      associations.  Linkage disequilibrium and shared eQTLs in TWAS can
      induce gene–gene correlations, slightly inflating Type I error.
    * Permutation p-values are computed against a random gene-set null
      and capture competitive (vs. genome) enrichment; they do not test
      self-containment vs. biological pathway annotation.
    * Wilcoxon/paired tests with n < 6 are flagged as unreliable.
      Coverage < 60% should be interpreted with caution.
    * Bootstrap CIs reflect resampling uncertainty in the gene-set
      members, not in the upstream TWAS Z-scores themselves.
    * Concordance metrics treat each gene independently and do not
      account for cross-tissue Z-score correlation.

    HOW TO INTERPRET
    ----------------
    * |Stouffer Z| > 1.96 ↔ two-sided p < 0.05 (uncorrected); compare
      against permutation p for a competitive null.
    * Cohen's d benchmarks (between-disease):
        small ≈ 0.2 | medium ≈ 0.5 | large ≈ 0.8
    * Rank-biserial r benchmarks:
        small ≈ 0.1 | medium ≈ 0.3 | large ≈ 0.5
    * Lin's CCC: 0 = no concordance, 1 = perfect agreement on identity line.
    * Sign concordance rate = 0.5 expected under independence; values
      ≫ 0.5 indicate shared directional effects.
""")


def run_pipeline(
    dirs,
    labels,
    geneset,
    label="CustomGeneSet",
    out="results",
    n_perm=10000,
    save_outputs=True,
    plot=True,
    verbose=True,
    # ── New robustness parameters (all backward-compatible) ─────────
    random_seed=42,
    robust=False,
    bootstrap=False,
    n_boot=2000,
    family_wise_fdr=False,
    influence_threshold_pct=20.0,
):
    """
    Run the full gene-set differential & proximity pipeline.

    New parameters
    --------------
    random_seed : int
        Master RNG seed used for permutation, sign-flip, and bootstrap
        steps.  Logged into every summary for reproducibility.
    robust : bool
        If True, winsorize Z-scores at 1st/99th percentile before
        parametric statistics (Stouffer Z, Welch t, Cohen's d, ...).
        Non-parametric tests are unaffected.
    bootstrap : bool
        If True, compute 95% bootstrap percentile CIs for Stouffer Z,
        mean Z, and mean |Z| (added to the enrichment table and plots).
    n_boot : int
        Number of bootstrap resamples.
    family_wise_fdr : bool
        If True, additionally compute a single global BH-FDR across
        ALL p-values produced by enrichment + differential + pairwise
        + proximity tests within this run.  Result saved as
        family_wise_fdr_<label>.csv.
    influence_threshold_pct : float
        Percent change in |Stouffer Z| at which a single gene removal
        is flagged as "influential" in the LOO analysis.

    Returns dict (existing keys preserved, new keys added):
        ... (existing) ...
        "influence_df"  : per-disease LOO influence table
        "provenance"    : input-file SHA256 manifest dict
        "warnings"      : list of pipeline-level warnings
    """
    _print = print if verbose else lambda *a, **k: None
    step_errors = {}
    pipeline_warnings = []

    # ── Validate dirs / labels ────────────────────────────────────
    if isinstance(dirs, (str, Path)):
        dirs = [dirs]
    if isinstance(labels, str):
        labels = [labels]
    dirs = list(dirs)
    labels = list(labels)
    if len(dirs) != len(labels):
        raise ValueError(
            f"dirs (length {len(dirs)}) and labels (length {len(labels)}) "
            f"must have the same length."
        )

    if save_outputs or plot:
        os.makedirs(out, exist_ok=True)

    # Master RNG (children spawned for sub-steps for reproducibility)
    master_rng = np.random.default_rng(random_seed)

    _print(f"\n{'='*72}")
    _print(f"  PSYCHIATRIC GENE-SET DIFFERENTIAL & PROXIMITY PIPELINE")
    _print(f"  Gene set         : {label}")
    _print(f"  Conditions       : {', '.join(labels)}")
    _print(f"  Output           : {out}/")
    _print(f"  Random seed      : {random_seed}")
    _print(f"  Robust mode      : {robust}")
    _print(f"  Bootstrap CIs    : {bootstrap}  (n_boot={n_boot})")
    _print(f"  n_perm           : {n_perm}")
    _print(f"  Family-wise FDR  : {family_wise_fdr}")
    _print(f"{'='*72}\n")

    # ── Compute provenance early ───────────────────────────────────
    try:
        provenance = compute_input_provenance(dirs, geneset)
        if save_outputs:
            with open(os.path.join(out, f"input_manifest_{label}.json"), "w") as fh:
                json.dump(provenance, fh, indent=2)
    except Exception as e:
        provenance = {"error": str(e)}
        pipeline_warnings.append(f"Provenance manifest failed: {e}")

    # ─── Step 1: Load disease data ─────────────────────────────────
    _print("[1/7] Loading S-PrediXcan meta-Z scores …")
    meta_z = {}
    id_maps = {}
    tissue_z = {}
    for tag, folder in zip(labels, dirs):
        _print(f"  {tag}:")
        mz, im, tz = load_spredixcan_folder(folder)
        meta_z[tag] = mz
        id_maps[tag] = im
        tissue_z[tag] = tz

    # Z-score sanity validation
    val_warnings = validate_zscores(meta_z)
    pipeline_warnings.extend(val_warnings)
    for w in val_warnings:
        _print(f"  ⚠ {w}")

    all_genes_union = set()
    for mz in meta_z.values():
        all_genes_union.update(mz.keys())
    _print(f"\n  Total unique genes across all diseases: {len(all_genes_union):,}")

    # ─── Step 2: Load gene set ─────────────────────────────────────
    _print(f"\n[2/7] Loading gene set …")
    gene_list = load_geneset(geneset)
    gene_list = harmonize_geneset(gene_list, meta_z, id_maps, verbose=verbose)

    found_any = sum(1 for g in gene_list if g in all_genes_union)
    _print(f"  {found_any}/{len(gene_list)} genes found in at least one disease dataset")

    if found_any == 0:
        raise ValueError(
            "No genes from the set were found in the data. "
            "Check gene names (should match gene_name column, case-insensitive)."
        )
    if found_any < 5:
        msg = (f"Only {found_any} gene(s) found across diseases — results "
               f"are EXPLORATORY. Differential / proximity tests may be unreliable.")
        pipeline_warnings.append(msg)
        _print(f"  ⚠ {msg}")

    # ─── Initialise default results ────────────────────────────────
    enrichment = {}
    enrich_df = pd.DataFrame()
    diff = {
        "n_common_genes": 0, "common_genes": [],
        "kruskal_h": np.nan, "kruskal_p": np.nan,
        "pairwise_df": pd.DataFrame()
    }
    prox_df = pd.DataFrame()
    profile_matrix = pd.DataFrame()
    pairwise_stats_df = pd.DataFrame()
    gene_detail = pd.DataFrame()
    influence_df = pd.DataFrame()
    family_wise_fdr_df = pd.DataFrame()

    # ─── Step 3: Enrichment ────────────────────────────────────────
    _print(f"\n[3/7] Computing per-disease enrichment "
           f"(perm={n_perm}, boot={'on' if bootstrap else 'off'}, robust={robust}) …")
    try:
        for d in labels:
            # Spawn distinct seed per disease for full reproducibility
            disease_seed = int(master_rng.integers(0, 2**31 - 1))
            enrichment[d] = gene_set_enrichment(
                meta_z[d], gene_list,
                n_perm=n_perm,
                random_seed=disease_seed,
                robust=robust,
                bootstrap=bootstrap,
                n_boot=n_boot,
            )
            info = enrichment[d]

            ci_str = ""
            if bootstrap and not np.isnan(info.get("stouffer_ci_low", np.nan)):
                ci_str = (f" CI95=[{info['stouffer_ci_low']}, "
                          f"{info['stouffer_ci_high']}]")
            _print(f"  {d}: {info['n_genes_found']}/{info['n_genes_total']} "
                   f"(coverage={info['coverage_pct']}%) | "
                   f"Stouffer Z = {info['stouffer_z']}{ci_str} | "
                   f"|Z| = {info['mean_abs_z']} | Perm p = {info['permutation_p']}")
            if info["low_coverage_warning"]:
                pipeline_warnings.append(
                    f"{d}: gene-set coverage {info['coverage_pct']}% < 60% — interpret cautiously")
            if info["low_n_warning"]:
                pipeline_warnings.append(
                    f"{d}: only {info['n_genes_found']} genes found — exploratory")

        enrich_rows = []
        for d in labels:
            row = {"Disease": d}
            row.update({k: v for k, v in enrichment[d].items() if k != "z_scores"})
            enrich_rows.append(row)
        enrich_df = pd.DataFrame(enrich_rows)

        if save_outputs:
            enrich_df.to_csv(os.path.join(out, f"enrichment_{label}.csv"), index=False)
            _print(f"  → {out}/enrichment_{label}.csv")
    except Exception as e:
        step_errors["enrichment"] = str(e)
        _print(f"  ⚠ Step 3 (Enrichment) FAILED: {e}")
        _print(f"    Traceback: {traceback.format_exc().strip()}")

    # ─── Step 4: Differential tests ────────────────────────────────
    _print(f"\n[4/7] Differential tests across diseases …")
    try:
        diff = differential_tests(meta_z, gene_list)
        _print(f"  Common genes in all {len(labels)} diseases: {diff['n_common_genes']}")
        _print(f"  Kruskal-Wallis H = {diff['kruskal_h']}, p = {diff['kruskal_p']}")

        if save_outputs and not diff["pairwise_df"].empty:
            pw_path = os.path.join(out, f"differential_pairwise_{label}.csv")
            diff["pairwise_df"].to_csv(pw_path, index=False)
            _print(f"  → {pw_path}")
        if verbose and not diff["pairwise_df"].empty:
            _print(diff["pairwise_df"].to_string(index=False))
    except Exception as e:
        step_errors["differential"] = str(e)
        _print(f"  ⚠ Step 4 (Differential tests) FAILED: {e}")
        _print(f"    Traceback: {traceback.format_exc().strip()}")

    # ─── Step 5: Proximity / distance ──────────────────────────────
    _print(f"\n[5/7] Profile proximity & distance …")
    try:
        prox_df, profile_matrix = profile_proximity(meta_z, gene_list)

        if save_outputs and not prox_df.empty:
            prox_path = os.path.join(out, f"proximity_{label}.csv")
            prox_df.to_csv(prox_path, index=False)
            _print(f"  → {prox_path}")
        if verbose and not prox_df.empty:
            _print(prox_df.to_string(index=False))
    except Exception as e:
        step_errors["proximity"] = str(e)
        _print(f"  ⚠ Step 5 (Proximity) FAILED: {e}")
        _print(f"    Traceback: {traceback.format_exc().strip()}")

    # ─── Step 6: Pairwise disease statistical tests ────────────────
    _print(f"\n[6/7] Pairwise disease statistical tests "
           f"(gene-set-restricted, robust={robust}) …")
    try:
        pw_seed = int(master_rng.integers(0, 2**31 - 1))
        pairwise_stats_df = pairwise_disease_geneset_tests(
            meta_z, gene_list, n_perm=n_perm,
            random_seed=pw_seed, robust=robust,
        )
        if not pairwise_stats_df.empty:
            _print(f"  {len(pairwise_stats_df)} disease pair(s) tested")
            for _, row in pairwise_stats_df.iterrows():
                conc_rate_str = (f"{row['sign_concordance_rate']:.3f}"
                                 if not pd.isna(row.get('sign_concordance_rate', np.nan))
                                 else "NA")
                conc_p_str = (f"{row['sign_concordance_p']:.4f}"
                              if not pd.isna(row.get('sign_concordance_p', np.nan))
                              else "NA")
                ccc_str = (f"{row['Lin_CCC']:.3f}"
                           if not pd.isna(row.get('Lin_CCC', np.nan)) else "NA")
                tau_str = (f"{row['Kendall_tau']:.3f}"
                           if not pd.isna(row.get('Kendall_tau', np.nan)) else "NA")
                _print(
                    f"    {row['Disease_1']} vs {row['Disease_2']}: "
                    f"n_paired={row['n_genes_paired']}, "
                    f"mean_diff={row.get('mean_diff', 'NA')}, "
                    f"KS p={row.get('KS_p', 'NA')}, "
                    f"Welch p={row.get('Welch_p', 'NA')}, "
                    f"Paired-t p={row.get('Paired_t_p', 'NA')}, "
                    f"Cohen d(paired)={row.get('Cohens_d_paired', 'NA')}, "
                    f"SignConc={conc_rate_str} (p={conc_p_str}), "
                    f"CCC={ccc_str}, τ={tau_str}"
                )
            if save_outputs:
                pw_stats_path = os.path.join(out, f"pairwise_disease_stats_{label}.csv")
                pairwise_stats_df.to_csv(pw_stats_path, index=False)
                _print(f"  → {pw_stats_path}")
        else:
            _print("  (fewer than 2 diseases – skipped)")
    except Exception as e:
        step_errors["pairwise_stats"] = str(e)
        _print(f"  ⚠ Step 6 (Pairwise disease stats) FAILED: {e}")
        _print(f"    Traceback: {traceback.format_exc().strip()}")

    # ─── Step 7: Gene-level LOO influence analysis ─────────────────
    _print(f"\n[7/7] Gene-level leave-one-out influence analysis …")
    try:
        influence_df = gene_influence_analysis(
            meta_z, gene_list, threshold_pct=influence_threshold_pct
        )
        if not influence_df.empty:
            n_inf = int(influence_df["Influential"].sum())
            _print(f"  {n_inf} gene(s) flagged as influential "
                   f"(>|{influence_threshold_pct}%| change or sign-flip)")
            if save_outputs:
                inf_path = os.path.join(out, f"gene_influence_{label}.csv")
                influence_df.to_csv(inf_path, index=False)
                _print(f"  → {inf_path}")
        else:
            _print("  (insufficient data for LOO analysis)")
    except Exception as e:
        step_errors["influence"] = str(e)
        _print(f"  ⚠ Step 7 (Influence) FAILED: {e}")
        _print(f"    Traceback: {traceback.format_exc().strip()}")

    # ─── Per-gene detail ───────────────────────────────────────────
    try:
        gene_detail = build_gene_detail_table(meta_z, gene_list)
        if save_outputs:
            detail_path = os.path.join(out, f"gene_detail_{label}.csv")
            gene_detail.to_csv(detail_path, index=False)
            _print(f"  → {detail_path}")
    except Exception as e:
        step_errors["gene_detail"] = str(e)
        _print(f"  ⚠ Per-gene detail table FAILED: {e}")

    # ─── Optional: family-wise FDR across all p-values ─────────────
    if family_wise_fdr:
        try:
            fw_rows = []
            # Enrichment p's
            if not enrich_df.empty:
                for _, r in enrich_df.iterrows():
                    for col in ("wilcoxon_p", "permutation_p"):
                        v = r.get(col, np.nan)
                        if pd.notna(v):
                            fw_rows.append({
                                "Source": "enrichment", "Test": col,
                                "Comparison": r["Disease"], "p_value": float(v)})
            # Differential pairwise
            if isinstance(diff.get("pairwise_df"), pd.DataFrame) and not diff["pairwise_df"].empty:
                for _, r in diff["pairwise_df"].iterrows():
                    for col in ("MWU_p", "Paired_Wilcoxon_p"):
                        v = r.get(col, np.nan)
                        if pd.notna(v):
                            fw_rows.append({
                                "Source": "differential", "Test": col,
                                "Comparison": f"{r['Disease_1']}__vs__{r['Disease_2']}",
                                "p_value": float(v)})
                # Kruskal-Wallis
                if pd.notna(diff.get("kruskal_p", np.nan)):
                    fw_rows.append({
                        "Source": "differential", "Test": "kruskal_p",
                        "Comparison": "all_diseases",
                        "p_value": float(diff["kruskal_p"])})
            # Proximity p's
            if isinstance(prox_df, pd.DataFrame) and not prox_df.empty:
                for _, r in prox_df.iterrows():
                    for col in ("Pearson_p", "Spearman_p"):
                        v = r.get(col, np.nan)
                        if pd.notna(v):
                            fw_rows.append({
                                "Source": "proximity", "Test": col,
                                "Comparison": f"{r['Disease_1']}__vs__{r['Disease_2']}",
                                "p_value": float(v)})
            # Pairwise stats p's
            if isinstance(pairwise_stats_df, pd.DataFrame) and not pairwise_stats_df.empty:
                p_cols = [c for c in pairwise_stats_df.columns if c.endswith("_p")]
                for _, r in pairwise_stats_df.iterrows():
                    for col in p_cols:
                        v = r.get(col, np.nan)
                        if pd.notna(v):
                            fw_rows.append({
                                "Source": "pairwise_stats", "Test": col,
                                "Comparison": f"{r['Disease_1']}__vs__{r['Disease_2']}",
                                "p_value": float(v)})
            family_wise_fdr_df = pd.DataFrame(fw_rows)
            if not family_wise_fdr_df.empty:
                family_wise_fdr_df["family_wise_FDR"] = benjamini_hochberg(
                    family_wise_fdr_df["p_value"].values
                )
                family_wise_fdr_df = family_wise_fdr_df.sort_values(
                    "p_value").reset_index(drop=True)
                if save_outputs:
                    fw_path = os.path.join(out, f"family_wise_fdr_{label}.csv")
                    family_wise_fdr_df.to_csv(fw_path, index=False)
                    _print(f"  → {fw_path}  ({len(family_wise_fdr_df)} p-values)")
        except Exception as e:
            step_errors["family_wise_fdr"] = str(e)
            _print(f"  ⚠ Family-wise FDR FAILED: {e}")

    # ─── Plots ─────────────────────────────────────────────────────
    if plot:
        _print(f"\nGenerating plots …")
        try:
            if enrichment:
                plot_zscore_boxplot(enrichment, label, out)
        except Exception as e:
            _print(f"  ⚠ Boxplot failed: {e}")

        try:
            if enrichment:
                plot_enrichment_bar(enrichment, label, out)
        except Exception as e:
            _print(f"  ⚠ Enrichment bar plot failed: {e}")

        try:
            if not prox_df.empty:
                plot_proximity_heatmap(prox_df, labels, label, out)
        except Exception as e:
            _print(f"  ⚠ Proximity heatmap failed: {e}")

        try:
            if not prox_df.empty:
                plot_proximity_dendrogram(prox_df, labels, label, out)
        except Exception as e:
            _print(f"  ⚠ Proximity dendrogram failed: {e}")

        try:
            if isinstance(profile_matrix, pd.DataFrame) and not profile_matrix.empty:
                plot_scatter_matrix(profile_matrix, label, out)
        except Exception as e:
            _print(f"  ⚠ Scatter matrix plot failed: {e}")

        try:
            if isinstance(pairwise_stats_df, pd.DataFrame) and not pairwise_stats_df.empty:
                plot_pairwise_stats_heatmap(pairwise_stats_df, labels, label, out)
        except Exception as e:
            _print(f"  ⚠ Pairwise stats heatmap failed: {e}")

        try:
            if gene_list:
                plot_zscore_qq(meta_z, gene_list, label, out)
        except Exception as e:
            _print(f"  ⚠ QQ-plot failed: {e}")

        try:
            if isinstance(influence_df, pd.DataFrame) and not influence_df.empty:
                plot_gene_influence(influence_df, label, out)
        except Exception as e:
            _print(f"  ⚠ Gene-influence plot failed: {e}")

    # ─── Summary ───────────────────────────────────────────────────
    summary_text = ""
    try:
        lines = [
            f"{'='*72}",
            f"  GENE-SET ANALYSIS SUMMARY – {label}",
            f"  {len(gene_list)} genes in set, {found_any} found in data",
            f"  Conditions: {', '.join(labels)}",
            f"  Generated : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}",
            f"{'='*72}",
            "",
            "  RUN PARAMETERS:",
            f"    Random seed         : {random_seed}",
            f"    Robust (winsorized) : {robust}",
            f"    Bootstrap CIs       : {bootstrap}  (n_boot={n_boot})",
            f"    Permutations (n)    : {n_perm}",
            f"    Family-wise FDR     : {family_wise_fdr}",
            f"    Influence threshold : ±{influence_threshold_pct}%",
            "",
            "  INPUT PROVENANCE (SHA256, first 16 hex):"
        ]
        try:
            for dir_key, dir_info in provenance.get("input_dirs", {}).items():
                lines.append(f"    [{dir_key}]")
                for fname, finfo in dir_info.get("files", {}).items():
                    lines.append(f"      {fname}: {finfo.get('sha256', '')[:16]}")
            gs_info = provenance.get("geneset_input")
            if isinstance(gs_info, dict) and "sha256" in gs_info:
                lines.append(f"    [geneset]: {gs_info['sha256'][:16]} "
                             f"({Path(gs_info.get('path','')).name})")
            elif isinstance(gs_info, dict):
                lines.append(f"    [geneset]: type={gs_info.get('type','?')}")
        except Exception:
            lines.append("    (provenance unavailable)")

        lines += ["", "  ENRICHMENT (per disease):"]
        if enrichment:
            for d in labels:
                e = enrichment.get(d, {})
                ci_blurb = ""
                if bootstrap and not pd.isna(e.get("stouffer_ci_low", np.nan)):
                    ci_blurb = (f" [CI95: {e['stouffer_ci_low']}, "
                                f"{e['stouffer_ci_high']}]")
                lines.append(
                    f"    {d}: n={e.get('n_genes_found','?')}/{e.get('n_genes_total','?')} "
                    f"(cov={e.get('coverage_pct','?')}%) | "
                    f"Stouffer Z={e.get('stouffer_z','?')}{ci_blurb} | "
                    f"|Z|={e.get('mean_abs_z','?')} | "
                    f"trimZ={e.get('trimmed_mean_z','?')} | "
                    f"Wilcoxon p={e.get('wilcoxon_p','?')} | "
                    f"Perm p={e.get('permutation_p','?')}"
                )
                flags = []
                if e.get("low_n_warning"): flags.append("LOW_N")
                if e.get("low_coverage_warning"): flags.append("LOW_COVERAGE")
                if e.get("wilcoxon_unreliable"): flags.append("WILCOXON<6")
                if flags:
                    lines.append(f"        ⚠ Flags: {', '.join(flags)}")
        else:
            lines.append("    (enrichment step failed)")

        if "enrichment" in step_errors:
            lines.append(f"    ⚠ STEP FAILED: {step_errors['enrichment']}")

        lines += [
            "",
            "  DIFFERENTIAL TESTS:",
            f"    Common genes (all {len(labels)}): {diff['n_common_genes']}",
            f"    Kruskal-Wallis H={diff['kruskal_h']}, p={diff['kruskal_p']}"
        ]
        if "differential" in step_errors:
            lines.append(f"    ⚠ STEP FAILED: {step_errors['differential']}")
        elif not diff["pairwise_df"].empty:
            for _, row in diff["pairwise_df"].iterrows():
                fdr_val = row.get("MWU_FDR", np.nan)
                fdr_str = f"{fdr_val:.4f}" if not pd.isna(fdr_val) else "NA"
                lines.append(
                    f"    {row['Disease_1']} vs {row['Disease_2']}: "
                    f"MWU p={row['MWU_p']}, FDR={fdr_str}, "
                    f"mean-Z diff={row['Mean_Z_diff']}"
                )

        lines += ["", "  PROFILE PROXIMITY:"]
        if "proximity" in step_errors:
            lines.append(f"    ⚠ STEP FAILED: {step_errors['proximity']}")
        elif not prox_df.empty:
            for _, row in prox_df.iterrows():
                lines.append(
                    f"    {row['Disease_1']} vs {row['Disease_2']}: "
                    f"r={row['Pearson_r']}, rho={row['Spearman_rho']}, "
                    f"cos={row['Cosine_sim']}, Eucl={row['Euclidean_dist']}"
                )
        else:
            lines.append("    (insufficient gene overlap)")

        lines += ["", "  PAIRWISE DISEASE STAT TESTS (gene-set-restricted):"]
        if "pairwise_stats" in step_errors:
            lines.append(f"    ⚠ STEP FAILED: {step_errors['pairwise_stats']}")
        elif isinstance(pairwise_stats_df, pd.DataFrame) and not pairwise_stats_df.empty:
            for _, row in pairwise_stats_df.iterrows():
                cd_str = (f"{row['Cohens_d_paired']:.3f}"
                          if not pd.isna(row.get('Cohens_d_paired', np.nan)) else "NA")
                pt_str = (f"{row['Paired_t_p']:.4f}"
                          if not pd.isna(row.get('Paired_t_p', np.nan)) else "NA")
                ks_str = (f"{row['KS_p']:.4f}"
                          if not pd.isna(row.get('KS_p', np.nan)) else "NA")
                we_str = (f"{row['Welch_p']:.4f}"
                          if not pd.isna(row.get('Welch_p', np.nan)) else "NA")
                pm_str = (f"{row['Permutation_p']:.4f}"
                          if not pd.isna(row.get('Permutation_p', np.nan)) else "NA")
                lines.append(
                    f"    {row['Disease_1']} vs {row['Disease_2']} "
                    f"(n_paired={row['n_genes_paired']}): "
                    f"mean_diff={row.get('mean_diff','NA')}, "
                    f"Cohen_d={cd_str}, "
                    f"KS_p={ks_str}, Welch_p={we_str}, "
                    f"Paired_t_p={pt_str}, Perm_p={pm_str}"
                )
            lines.append("")
            lines.append("  CONCORDANCE (gene-set-restricted, paired genes):")
            for _, row in pairwise_stats_df.iterrows():
                conc_rate = row.get("sign_concordance_rate", np.nan)
                conc_p    = row.get("sign_concordance_p", np.nan)
                n_conc    = row.get("n_concordant", np.nan)
                n_disc    = row.get("n_discordant", np.nan)
                ccc_val   = row.get("Lin_CCC", np.nan)
                tau_val   = row.get("Kendall_tau", np.nan)
                tau_p     = row.get("Kendall_p", np.nan)

                conc_rate_str = f"{conc_rate:.3f}" if not pd.isna(conc_rate) else "NA"
                conc_p_str    = f"{conc_p:.4f}"    if not pd.isna(conc_p)    else "NA"
                n_conc_str    = f"{int(n_conc)}"   if not pd.isna(n_conc)    else "NA"
                n_disc_str    = f"{int(n_disc)}"   if not pd.isna(n_disc)    else "NA"
                ccc_str       = f"{ccc_val:.3f}"   if not pd.isna(ccc_val)   else "NA"
                tau_str       = f"{tau_val:.3f}"   if not pd.isna(tau_val)   else "NA"
                tau_p_str     = f"{tau_p:.4f}"     if not pd.isna(tau_p)     else "NA"
                lines.append(
                    f"    {row['Disease_1']} vs {row['Disease_2']} "
                    f"(n_paired={row['n_genes_paired']}): "
                    f"concordant={n_conc_str}, discordant={n_disc_str}, "
                    f"sign_conc_rate={conc_rate_str} (p={conc_p_str}), "
                    f"Lin_CCC={ccc_str}, "
                    f"Kendall_τ={tau_str} (p={tau_p_str})"
                )
        else:
            lines.append("    (fewer than 2 diseases or insufficient genes)")

        # ── Influence summary ──────────────────────────────────────
        lines += ["", "  GENE INFLUENCE (LOO):"]
        if "influence" in step_errors:
            lines.append(f"    ⚠ STEP FAILED: {step_errors['influence']}")
        elif isinstance(influence_df, pd.DataFrame) and not influence_df.empty:
            n_total_inf = int(influence_df["Influential"].sum())
            lines.append(f"    Genes flagged as influential "
                         f"(|ΔStouffer|>{influence_threshold_pct}% or sign-flip): "
                         f"{n_total_inf}")
            for d in labels:
                sub = influence_df[influence_df["Disease"] == d]
                if sub.empty:
                    continue
                flagged = sub[sub["Influential"]]
                if not flagged.empty:
                    top = flagged.head(5)[["Gene", "Delta_Stouffer",
                                           "Pct_Change_Stouffer", "Sign_Change"]]
                    desc = ", ".join(
                        f"{r.Gene} (Δ={r.Delta_Stouffer:+.2f}, "
                        f"{r.Pct_Change_Stouffer:.1f}%"
                        f"{', SIGN' if r.Sign_Change else ''})"
                        for r in top.itertuples(index=False))
                    lines.append(f"    {d}: {desc}")
                else:
                    lines.append(f"    {d}: no genes exceed influence threshold")
        else:
            lines.append("    (insufficient data)")

        # ── Family-wise FDR section ────────────────────────────────
        if family_wise_fdr:
            lines += ["", "  FAMILY-WISE FDR (across all in-run p-values):"]
            if "family_wise_fdr" in step_errors:
                lines.append(f"    ⚠ STEP FAILED: {step_errors['family_wise_fdr']}")
            elif not family_wise_fdr_df.empty:
                n_total = len(family_wise_fdr_df)
                n_sig = int((family_wise_fdr_df["family_wise_FDR"] < 0.05).sum())
                lines.append(f"    Total p-values pooled: {n_total}")
                lines.append(f"    Significant after global FDR (<0.05): {n_sig}")
                top5 = family_wise_fdr_df.head(5)
                for _, r in top5.iterrows():
                    lines.append(
                        f"    {r['Source']}/{r['Test']} [{r['Comparison']}]: "
                        f"p={r['p_value']:.2e}, FDR={r['family_wise_FDR']:.4f}"
                    )
            else:
                lines.append("    (no p-values aggregated)")

        # ── Pipeline warnings ──────────────────────────────────────
        if pipeline_warnings:
            lines += ["", "  PIPELINE WARNINGS:"]
            for w in pipeline_warnings:
                lines.append(f"    ⚠ {w}")

        if step_errors:
            lines += ["", "  STEP ERRORS:"]
            for step_name, err_msg in step_errors.items():
                lines.append(f"    {step_name}: {err_msg}")

        # ── Limitations & interpretation block ─────────────────────
        lines += ["", _LIMITATIONS_BLOCK.rstrip(), "", f"{'='*72}"]

        summary_text = "\n".join(lines)
        _print(f"\n{summary_text}")

        if save_outputs:
            summary_path = os.path.join(out, f"summary_{label}.txt")
            with open(summary_path, "w") as fh:
                fh.write(summary_text)
    except Exception as e:
        summary_text = f"Summary generation failed: {e}"
        _print(f"  ⚠ Summary generation FAILED: {e}")

    _print(f"\n{'='*72}")
    if step_errors:
        _print(f"  DONE (with {len(step_errors)} step error(s)) – outputs in {out}/")
        for step_name, err_msg in step_errors.items():
            _print(f"    ⚠ {step_name}: {err_msg}")
    else:
        _print(f"  DONE – all outputs in {out}/")
    _print(f"{'='*72}\n")

    return {
        "meta_z": meta_z,
        "tissue_z": tissue_z,
        "gene_list": gene_list,
        "enrichment": enrichment,
        "enrichment_df": enrich_df,
        "differential": diff,
        "proximity_df": prox_df,
        "profile_matrix": profile_matrix,
        "pairwise_stats_df": pairwise_stats_df,
        "gene_detail_df": gene_detail,
        "influence_df": influence_df,
        "family_wise_fdr_df": family_wise_fdr_df,
        "summary_text": summary_text,
        "step_errors": step_errors,
        "warnings": pipeline_warnings,
        "provenance": provenance,
        "run_params": {
            "random_seed": random_seed,
            "robust": robust,
            "bootstrap": bootstrap,
            "n_boot": n_boot,
            "n_perm": n_perm,
            "family_wise_fdr": family_wise_fdr,
            "influence_threshold_pct": influence_threshold_pct,
        },
    }


def run_multi_geneset_pipeline(
    dirs,
    labels,
    gene_sets,
    out="results",
    n_perm=10000,
    tissue_n_perm=0,
    save_outputs=True,
    plot=True,
    verbose=True,
    # ── New robustness parameters (forwarded to run_pipeline) ─────
    random_seed=42,
    robust=False,
    bootstrap=False,
    n_boot=2000,
    family_wise_fdr=False,
    influence_threshold_pct=20.0,
):
    """
    Run the same pipeline over multiple gene sets.

    Same new parameters as run_pipeline; they are forwarded per-set.
    """
    _print = print if verbose else lambda *a, **k: None

    gene_sets = load_multiple_genesets(gene_sets)

    if save_outputs or plot:
        os.makedirs(out, exist_ok=True)

    all_results = {}
    summary_rows = []
    tissue_summary_rows = []
    gene_detail_frames = []
    pairwise_stats_frames = []
    influence_frames = []

    for gs_label, genes in gene_sets.items():
        safe_label = str(gs_label).replace(" ", "_")
        sub_out = os.path.join(out, safe_label)

        _print(f"\n{'#'*72}")
        _print(f"Running gene set: {gs_label}  (n={len(genes)})")
        _print(f"{'#'*72}")

        try:
            res = run_pipeline(
                dirs=dirs,
                labels=labels,
                geneset=genes,
                label=gs_label,
                out=sub_out,
                n_perm=n_perm,
                save_outputs=save_outputs,
                plot=plot,
                verbose=verbose,
                random_seed=random_seed,
                robust=robust,
                bootstrap=bootstrap,
                n_boot=n_boot,
                family_wise_fdr=family_wise_fdr,
                influence_threshold_pct=influence_threshold_pct,
            )
            all_results[gs_label] = res

            if isinstance(res.get("enrichment_df"), pd.DataFrame) and not res["enrichment_df"].empty:
                for _, row in res["enrichment_df"].iterrows():
                    summary_rows.append({
                        "GeneSet": gs_label,
                        "Disease": row["Disease"],
                        "Tissue": "meta_across_tissues",
                        "n_genes_found": row["n_genes_found"],
                        "n_genes_total": row["n_genes_total"],
                        "coverage_pct": row["coverage_pct"],
                        "stouffer_z": row["stouffer_z"],
                        "stouffer_ci_low":  row.get("stouffer_ci_low", np.nan),
                        "stouffer_ci_high": row.get("stouffer_ci_high", np.nan),
                        "mean_abs_z": row["mean_abs_z"],
                        "mean_abs_z_ci_low":  row.get("mean_abs_z_ci_low", np.nan),
                        "mean_abs_z_ci_high": row.get("mean_abs_z_ci_high", np.nan),
                        "mean_z": row["mean_z"],
                        "mean_z_ci_low":  row.get("mean_z_ci_low", np.nan),
                        "mean_z_ci_high": row.get("mean_z_ci_high", np.nan),
                        "trimmed_mean_z": row.get("trimmed_mean_z", np.nan),
                        "median_z": row["median_z"],
                        "wilcoxon_p": row["wilcoxon_p"],
                        "permutation_p": row["permutation_p"],
                        "low_n_warning":        row.get("low_n_warning", False),
                        "low_coverage_warning": row.get("low_coverage_warning", False),
                    })

            res_tissue_z = res.get("tissue_z", {})
            harmonized_genes = res.get("gene_list", [])
            for disease_label in labels:
                disease_tissues = res_tissue_z.get(disease_label, {})
                for tissue_name in sorted(disease_tissues.keys()):
                    tissue_dict = disease_tissues[tissue_name]
                    try:
                        te = gene_set_enrichment(
                            tissue_dict, harmonized_genes,
                            n_perm=tissue_n_perm,
                            random_seed=random_seed,
                            robust=robust,
                            bootstrap=False,  # keep speed for tissues
                        )
                        tissue_summary_rows.append({
                            "GeneSet": gs_label,
                            "Disease": disease_label,
                            "Tissue": tissue_name,
                            "n_genes_found": te["n_genes_found"],
                            "n_genes_total": te["n_genes_total"],
                            "coverage_pct": te["coverage_pct"],
                            "stouffer_z": te["stouffer_z"],
                            "mean_abs_z": te["mean_abs_z"],
                            "mean_z": te["mean_z"],
                            "trimmed_mean_z": te.get("trimmed_mean_z", np.nan),
                            "median_z": te["median_z"],
                            "wilcoxon_p": te["wilcoxon_p"],
                            "permutation_p": te["permutation_p"],
                        })
                    except Exception as te_err:
                        _print(f"  ⚠ Tissue enrichment failed for "
                               f"{disease_label}/{tissue_name}: {te_err}")

            pw_res = res.get("pairwise_stats_df", pd.DataFrame())
            if isinstance(pw_res, pd.DataFrame) and not pw_res.empty:
                pw_copy = pw_res.copy()
                pw_copy.insert(0, "GeneSet", gs_label)
                pairwise_stats_frames.append(pw_copy)

            inf_res = res.get("influence_df", pd.DataFrame())
            if isinstance(inf_res, pd.DataFrame) and not inf_res.empty:
                inf_copy = inf_res.copy()
                inf_copy.insert(0, "GeneSet", gs_label)
                influence_frames.append(inf_copy)

            if ("gene_detail_df" in res
                    and isinstance(res["gene_detail_df"], pd.DataFrame)
                    and not res["gene_detail_df"].empty):
                gdf = res["gene_detail_df"].copy()
                gdf.insert(0, "GeneSet", gs_label)

                for disease_label in labels:
                    disease_tissues = res_tissue_z.get(disease_label, {})
                    for tissue_name in sorted(disease_tissues.keys()):
                        col_name = f"Z_{disease_label}__{tissue_name}"
                        td = disease_tissues[tissue_name]
                        gdf[col_name] = gdf["Gene"].map(
                            lambda g, _td=td: round(_td[g], 4) if g in _td else np.nan
                        )

                gene_detail_frames.append(gdf)

        except Exception as e:
            _print(f"  ⚠ Failed for gene set '{gs_label}': {e}")
            all_results[gs_label] = {"error": str(e)}

    # ── Save enrichment summary (meta + tissue rows) ─────────────────
    if save_outputs and (summary_rows or tissue_summary_rows):
        all_enrich = summary_rows + tissue_summary_rows
        enrich_summary_df = pd.DataFrame(all_enrich)
        front = ["GeneSet", "Disease", "Tissue"]
        rest = [c for c in enrich_summary_df.columns if c not in front]
        enrich_summary_df = enrich_summary_df[front + rest]
        enrich_summary_df.sort_values(
            ["GeneSet", "Disease", "Tissue"], ignore_index=True, inplace=True
        )
        enrich_path = os.path.join(out, "multi_geneset_enrichment_summary.csv")
        enrich_summary_df.to_csv(enrich_path, index=False)
        _print(f"\n  → Enrichment summary (with by-tissue + bootstrap CI cols): {enrich_path}")
        n_meta = len(summary_rows)
        n_tiss = len(tissue_summary_rows)
        _print(f"    ({n_meta} meta-level rows + {n_tiss} tissue-level rows)")

    # ── Save combined pairwise disease stats (with global FDR) ───────
    combined_pairwise_stats_df = pd.DataFrame()
    if pairwise_stats_frames:
        combined_pairwise_stats_df = pd.concat(
            pairwise_stats_frames, ignore_index=True
        )
        p_cols = [c for c in combined_pairwise_stats_df.columns if c.endswith("_p")]
        for pc in p_cols:
            global_fdr_col = pc.replace("_p", "_globalFDR")
            vals = combined_pairwise_stats_df[pc].values.astype(float)
            mask = ~np.isnan(vals)
            if mask.sum() > 1:
                fdr_vals = np.full(len(vals), np.nan)
                fdr_vals[mask] = benjamini_hochberg(vals[mask])
                combined_pairwise_stats_df[global_fdr_col] = fdr_vals
            elif mask.sum() == 1:
                combined_pairwise_stats_df[global_fdr_col] = vals

        if save_outputs:
            pw_path = os.path.join(out, "multi_geneset_pairwise_disease_stats.csv")
            combined_pairwise_stats_df.to_csv(pw_path, index=False)
            _print(f"\n  → Pairwise disease stats (all gene sets): {pw_path}")
            _print(f"    ({len(combined_pairwise_stats_df)} rows across "
                   f"{combined_pairwise_stats_df['GeneSet'].nunique()} gene set(s))")

    all_results["_combined_pairwise_stats_df"] = combined_pairwise_stats_df

    # ── Combined influence table ─────────────────────────────────────
    combined_influence_df = pd.DataFrame()
    if influence_frames:
        combined_influence_df = pd.concat(influence_frames, ignore_index=True)
        if save_outputs:
            inf_path = os.path.join(out, "multi_geneset_gene_influence.csv")
            combined_influence_df.to_csv(inf_path, index=False)
            _print(f"\n  → Combined gene-influence (LOO) table: {inf_path}")
    all_results["_combined_influence_df"] = combined_influence_df

    # ── Combined gene-detail CSV ─────────────────────────────────────
    combined_gene_df = pd.DataFrame()
    if gene_detail_frames:
        combined_gene_df = pd.concat(gene_detail_frames, ignore_index=True)
        front_cols = ["GeneSet", "Gene"]
        meta_z_cols = [c for c in combined_gene_df.columns
                       if c.startswith("Z_") and "__" not in c
                       and c not in front_cols]
        tissue_z_cols = [c for c in combined_gene_df.columns
                         if c.startswith("Z_") and "__" in c]
        other_cols = [c for c in combined_gene_df.columns
                      if c not in front_cols + meta_z_cols + tissue_z_cols]
        combined_gene_df = combined_gene_df[
            front_cols + meta_z_cols + other_cols + tissue_z_cols
        ]
        if save_outputs:
            combined_path = os.path.join(out, "all_genes_all_genesets.csv")
            combined_gene_df.to_csv(combined_path, index=False)
            _print(f"\n  → Combined gene-level CSV (with by-tissue Z): {combined_path}")
            _print(f"    ({len(combined_gene_df)} rows across "
                   f"{combined_gene_df['GeneSet'].nunique()} gene set(s), "
                   f"{len(tissue_z_cols)} tissue Z columns)")

    all_results["_combined_gene_df"] = combined_gene_df

    # ── Top-level run-parameter manifest ─────────────────────────────
    if save_outputs:
        try:
            run_manifest = {
                "generated_at": datetime.now().isoformat(timespec="seconds"),
                "n_gene_sets": len(gene_sets),
                "labels": labels,
                "dirs": [str(d) for d in dirs],
                "params": {
                    "random_seed": random_seed,
                    "robust": robust,
                    "bootstrap": bootstrap,
                    "n_boot": n_boot,
                    "n_perm": n_perm,
                    "tissue_n_perm": tissue_n_perm,
                    "family_wise_fdr": family_wise_fdr,
                    "influence_threshold_pct": influence_threshold_pct,
                },
            }
            with open(os.path.join(out, "multi_geneset_run_manifest.json"), "w") as fh:
                json.dump(run_manifest, fh, indent=2)
        except Exception as e:
            _print(f"  ⚠ Failed to write run manifest: {e}")

    _print(f"\n{'='*72}")
    _print(f"  MULTI-GENESET PIPELINE COMPLETE – all outputs in {out}/")
    _print(f"{'='*72}\n")

    return all_results


# ===================================================================
#  9.  CLI ENTRY POINT
# ===================================================================

def main():
    """Thin CLI wrapper around run_pipeline()."""
    parser = argparse.ArgumentParser(
        description="Psychiatric Gene-Set Differential & Proximity Pipeline",
        formatter_class=argparse.RawDescriptionHelpFormatter,
        epilog=textwrap.dedent("""\
        Examples:
          python geneset_pipeline.py --dirs data/mdd/ data/bip/ data/ocd/ \
              --labels MDD BIP OCD \
              --geneset my_genes.txt --label "Dopamine_Receptors"

          python geneset_pipeline.py --dirs data/mdd/ data/bip/ data/ocd/ \
              --labels MDD BIP OCD \
              --geneset DRD1,DRD2,DRD3,DRD4,DRD5,COMT --label "DA_Genes" \
              --bootstrap --robust --random_seed 1234

          python geneset_pipeline.py --dirs data/mdd/ data/bip/ data/ocd/ \
              --labels MDD BIP OCD \
              --genesets_file genesets.py --out results/ \
              --family_wise_fdr
        """)
    )
    parser.add_argument("--dirs", required=True, nargs="+",
                        help="Folders with S-PrediXcan results (one per condition)")
    parser.add_argument("--labels", required=True, nargs="+",
                        help="Labels for each folder (same order as --dirs)")
    parser.add_argument("--geneset", default=None,
                        help="Path to txt/tsv file OR comma-separated gene list")
    parser.add_argument("--genesets_file", default=None,
                        help="Path to file containing multiple gene sets")
    parser.add_argument("--label", default="CustomGeneSet",
                        help="Label for gene set (used in titles and filenames)")
    parser.add_argument("--n_perm", type=int, default=10000,
                        help="Permutations for enrichment p-value (default: 10000; "
                             "set to 0 to skip permutations entirely)")
    parser.add_argument("--tissue_n_perm", type=int, default=0,
                        help="Permutations for per-tissue enrichment (default: 0)")
    parser.add_argument("--out", default="results", help="Output directory")
    parser.add_argument("--no_plot", action="store_true", help="Skip plot generation")
    # ── New flags ──────────────────────────────────────────────────
    parser.add_argument("--random_seed", type=int, default=42,
                        help="Master RNG seed for reproducibility (default: 42)")
    parser.add_argument("--robust", action="store_true",
                        help="Winsorize Z-scores (1st/99th pct) before parametric stats")
    parser.add_argument("--bootstrap", action="store_true",
                        help="Compute 95% bootstrap CIs for Stouffer Z, mean Z, mean |Z|")
    parser.add_argument("--n_boot", type=int, default=2000,
                        help="Number of bootstrap resamples (default: 2000)")
    parser.add_argument("--family_wise_fdr", action="store_true",
                        help="Compute global FDR across ALL p-values in the run")
    parser.add_argument("--influence_threshold_pct", type=float, default=20.0,
                        help="LOO percent-change threshold for flagging influential genes")
    args = parser.parse_args()

    if len(args.dirs) != len(args.labels):
        parser.error(
            f"--dirs ({len(args.dirs)} items) and --labels ({len(args.labels)} items) "
            f"must have the same number of arguments."
        )

    if not args.geneset and not args.genesets_file:
        parser.error("Provide either --geneset or --genesets_file")
    if args.geneset and args.genesets_file:
        parser.error("Use only one of --geneset or --genesets_file")

    common_kwargs = dict(
        random_seed=args.random_seed,
        robust=args.robust,
        bootstrap=args.bootstrap,
        n_boot=args.n_boot,
        family_wise_fdr=args.family_wise_fdr,
        influence_threshold_pct=args.influence_threshold_pct,
    )

    try:
        if args.genesets_file:
            run_multi_geneset_pipeline(
                dirs=args.dirs,
                labels=args.labels,
                gene_sets=args.genesets_file,
                out=args.out,
                n_perm=args.n_perm,
                tissue_n_perm=args.tissue_n_perm,
                save_outputs=True,
                plot=not args.no_plot,
                verbose=True,
                **common_kwargs,
            )
        else:
            run_pipeline(
                dirs=args.dirs,
                labels=args.labels,
                geneset=args.geneset,
                label=args.label,
                out=args.out,
                n_perm=args.n_perm,
                save_outputs=True,
                plot=not args.no_plot,
                verbose=True,
                **common_kwargs,
            )
    except Exception as e:
        # Clean error report + traceback to file
        os.makedirs(args.out, exist_ok=True)
        err_path = os.path.join(args.out, "PIPELINE_ERROR.txt")
        with open(err_path, "w") as fh:
            fh.write(f"Pipeline failed at {datetime.now().isoformat(timespec='seconds')}\n")
            fh.write(f"Error: {e}\n\nTraceback:\n{traceback.format_exc()}")
        print(f"\n✗ Pipeline FAILED: {e}")
        print(f"  Full traceback written to: {err_path}")
        sys.exit(1)


# ===================================================================
#  10.  REPORT GENERATOR
# ===================================================================

def save_geneset_report(results, gene_set_name=None, output_dir=None):
    """
    Generate and save a full report from gene-set enrichment results.

    Multi-result support: if `results` is a dict of {label: single_run_result},
    reports are generated into subfolders.

    Outputs (additions in this update marked NEW):
    <output_dir>/
    ├── summary.txt
    ├── tables/
    │   ├── enrichment_summary.csv         (now includes bootstrap CI cols)
    │   ├── pairwise_differential.csv
    │   ├── pairwise_disease_stats.csv
    │   ├── profile_proximity.csv
    │   ├── profile_matrix.csv
    │   ├── gene_detail.csv
    │   ├── meta_z_geneset.csv
    │   ├── gene_influence.csv             (NEW)
    │   ├── family_wise_fdr.csv            (NEW; if computed)
    │   └── input_manifest.json            (NEW)
    └── plots/
        ├── 01_enrichment_overview.png/.pdf
        ├── 02_profile_heatmap.png/.pdf
        ├── 03_gene_barplot.png/.pdf
        ├── 04_pairwise_scatter.png/.pdf
        ├── 05_gene_z_range.png/.pdf
        ├── 06_z_distribution_boxplot.png/.pdf
        ├── 07_enrichment_metrics.png/.pdf
        ├── 08_pairwise_stats_heatmap.png/.pdf
        ├── 09_concordance_heatmap.png/.pdf
        ├── 10_qq_plot.png/.pdf            (NEW)
        ├── 11_dendrogram.png/.pdf         (NEW)
        └── 12_gene_influence.png/.pdf     (NEW)
    """
    # Multi-result support
    if "enrichment" not in results and all(isinstance(v, dict) for v in results.values()):
        out_dirs = {}
        base_dir = output_dir
        if base_dir is None:
            ts = datetime.now().strftime("%Y%m%d_%H%M%S")
            base_dir = f"./geneset_reports_{ts}"
        os.makedirs(base_dir, exist_ok=True)

        for label, res in results.items():
            if not isinstance(res, dict) or "enrichment" not in res:
                continue
            subdir = os.path.join(base_dir, str(label).replace(" ", "_"))
            out_dirs[label] = save_geneset_report(
                res, gene_set_name=label, output_dir=subdir
            )
        return out_dirs

    if gene_set_name is None:
        try:
            line = results["summary_text"].split("\n")[1]
            gene_set_name = line.split("–")[-1].strip() if "–" in line else "gene_set"
        except Exception:
            gene_set_name = "gene_set"

    safe_name = gene_set_name.replace(" ", "_")

    if output_dir is None:
        ts = datetime.now().strftime("%Y%m%d_%H%M%S")
        output_dir = f"./geneset_report_{safe_name}_{ts}"

    tables_dir = os.path.join(output_dir, "tables")
    plots_dir  = os.path.join(output_dir, "plots")
    os.makedirs(tables_dir, exist_ok=True)
    os.makedirs(plots_dir,  exist_ok=True)

    diseases = list(results["enrichment"].keys()) if results.get("enrichment") else []
    edf      = results.get("enrichment_df", pd.DataFrame()).copy()

    _pal = sns.color_palette("Set2", max(len(diseases), 3))
    dcol = {d: _pal[i] for i, d in enumerate(diseases)}

    # ── 1. Summary text ───────────────────────────────────────────
    try:
        summary_path = os.path.join(output_dir, "summary.txt")
        with open(summary_path, "w") as fh:
            fh.write(results.get("summary_text", "(no summary available)"))
            fh.write(f"\n\nReport generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
            fh.write(f"Output directory : {os.path.abspath(output_dir)}\n")
        print(f"[✓] summary.txt")
    except Exception as e:
        print(f"[⚠] summary.txt failed: {e}")

    # ── 2. Tables ─────────────────────────────────────────────────
    try:
        if not edf.empty:
            edf.to_csv(os.path.join(tables_dir, "enrichment_summary.csv"), index=False)
    except Exception as e:
        print(f"[⚠] enrichment_summary.csv failed: {e}")

    try:
        if "differential" in results and "pairwise_df" in results["differential"]:
            pw = results["differential"]["pairwise_df"]
            if isinstance(pw, pd.DataFrame) and not pw.empty:
                pw.to_csv(os.path.join(tables_dir, "pairwise_differential.csv"), index=False)
    except Exception as e:
        print(f"[⚠] pairwise_differential.csv failed: {e}")

    try:
        pw_stats = results.get("pairwise_stats_df", pd.DataFrame())
        if isinstance(pw_stats, pd.DataFrame) and not pw_stats.empty:
            pw_stats.to_csv(os.path.join(tables_dir, "pairwise_disease_stats.csv"), index=False)
    except Exception as e:
        print(f"[⚠] pairwise_disease_stats.csv failed: {e}")

    try:
        prox = results.get("proximity_df", pd.DataFrame())
        if isinstance(prox, pd.DataFrame) and not prox.empty:
            prox.to_csv(os.path.join(tables_dir, "profile_proximity.csv"), index=False)
    except Exception as e:
        print(f"[⚠] profile_proximity.csv failed: {e}")

    try:
        pm = results.get("profile_matrix", pd.DataFrame())
        if isinstance(pm, pd.DataFrame) and not pm.empty:
            pm.to_csv(os.path.join(tables_dir, "profile_matrix.csv"))
    except Exception as e:
        print(f"[⚠] profile_matrix.csv failed: {e}")

    try:
        gd = results.get("gene_detail_df", pd.DataFrame())
        if isinstance(gd, pd.DataFrame) and not gd.empty:
            gd.to_csv(os.path.join(tables_dir, "gene_detail.csv"), index=False)
    except Exception as e:
        print(f"[⚠] gene_detail.csv failed: {e}")

    # NEW: gene influence table
    try:
        inf = results.get("influence_df", pd.DataFrame())
        if isinstance(inf, pd.DataFrame) and not inf.empty:
            inf.to_csv(os.path.join(tables_dir, "gene_influence.csv"), index=False)
    except Exception as e:
        print(f"[⚠] gene_influence.csv failed: {e}")

    # NEW: family-wise FDR
    try:
        fw = results.get("family_wise_fdr_df", pd.DataFrame())
        if isinstance(fw, pd.DataFrame) and not fw.empty:
            fw.to_csv(os.path.join(tables_dir, "family_wise_fdr.csv"), index=False)
    except Exception as e:
        print(f"[⚠] family_wise_fdr.csv failed: {e}")

    # NEW: input manifest
    try:
        prov = results.get("provenance", {})
        if prov:
            with open(os.path.join(tables_dir, "input_manifest.json"), "w") as fh:
                json.dump(prov, fh, indent=2)
    except Exception as e:
        print(f"[⚠] input_manifest.json failed: {e}")

    try:
        meta_rows = []
        for gene in results.get("gene_list", []):
            row = {"Gene": gene}
            for d in diseases:
                row[f"meta_z_{d}"] = results.get("meta_z", {}).get(d, {}).get(gene, np.nan)
            meta_rows.append(row)
        if meta_rows:
            pd.DataFrame(meta_rows).to_csv(
                os.path.join(tables_dir, "meta_z_geneset.csv"), index=False
            )
    except Exception as e:
        print(f"[⚠] meta_z_geneset.csv failed: {e}")

    n_table_files = len([f for f in os.listdir(tables_dir)]) if os.path.isdir(tables_dir) else 0
    print(f"[✓] tables/  ({n_table_files} files)")

    def _save(fig, stem):
        for ext in ("png", "pdf"):
            fig.savefig(
                os.path.join(plots_dir, f"{stem}.{ext}"),
                dpi=200, bbox_inches="tight",
            )
        plt.close(fig)

    pm = results.get("profile_matrix", pd.DataFrame())
    if isinstance(pm, pd.DataFrame) and not pm.empty:
        pm = pm.dropna()

    # ---- 01  Enrichment overview (Stouffer Z + −log10 perm-p) ----
    try:
        if not edf.empty and diseases:
            fig, axes = plt.subplots(1, 2, figsize=(max(12, len(diseases) * 3), 5))

            bars = axes[0].bar(
                edf["Disease"], edf["stouffer_z"],
                color=[dcol[d] for d in edf["Disease"]],
                edgecolor="black", linewidth=0.5,
            )
            axes[0].axhline(0, color="grey", lw=0.8, ls="--")
            for thr, c in [(1.96, "red"), (-1.96, "red")]:
                axes[0].axhline(thr, color=c, lw=0.6, ls=":", alpha=0.6)
            axes[0].set_ylabel("Stouffer Z")
            # Add CI error bars when available
            if "stouffer_ci_low" in edf.columns and edf["stouffer_ci_low"].notna().any():
                for b, lo, hi in zip(bars, edf["stouffer_ci_low"], edf["stouffer_ci_high"]):
                    if not (pd.isna(lo) or pd.isna(hi)):
                        axes[0].plot([b.get_x() + b.get_width()/2]*2, [lo, hi],
                                     color="black", lw=1.2)
                axes[0].set_title("Stouffer Z per Disease (95% bootstrap CI)")
            else:
                axes[0].set_title("Stouffer Z per Disease")
            for b, pv in zip(bars, edf["permutation_p"]):
                if pd.isna(pv):
                    continue
                sig = "***" if pv < 0.001 else "**" if pv < 0.01 else "*" if pv < 0.05 else "ns"
                va  = "bottom" if b.get_height() >= 0 else "top"
                axes[0].text(
                    b.get_x() + b.get_width() / 2, b.get_height(),
                    f"p={pv:.4f}\n{sig}", ha="center", va=va, fontsize=9,
                )

            neg_log = -np.log10(edf["permutation_p"].replace(0, 1e-10).fillna(1.0))
            axes[1].bar(
                edf["Disease"], neg_log,
                color=[dcol[d] for d in edf["Disease"]],
                edgecolor="black", linewidth=0.5,
            )
            axes[1].axhline(-np.log10(0.05),  color="red",     lw=0.8, ls="--", label="p = 0.05")
            axes[1].axhline(-np.log10(0.001), color="darkred",  lw=0.8, ls=":",  label="p = 0.001")
            axes[1].set_ylabel("−log₁₀(permutation p)")
            axes[1].set_title("Enrichment Significance")
            axes[1].legend(fontsize=8)

            fig.suptitle(f"{gene_set_name} — Enrichment Overview", fontsize=13, fontweight="bold")
            plt.tight_layout()
            _save(fig, "01_enrichment_overview")
    except Exception as e:
        print(f"[⚠] Plot 01 (enrichment overview) failed: {e}")
        plt.close("all")

    # ---- 02  Heatmap of profile matrix --------------------------
    try:
        if isinstance(pm, pd.DataFrame) and not pm.empty:
            vabs = max(abs(pm.values.min()), abs(pm.values.max()))
            fig, ax = plt.subplots(figsize=(max(6, len(diseases) * 2.5), max(6, len(pm) * 0.38)))
            sns.heatmap(
                pm, cmap="RdBu_r", center=0, vmin=-vabs, vmax=vabs,
                annot=True, fmt=".2f", linewidths=0.5, linecolor="white",
                cbar_kws={"label": "meta-Z"}, ax=ax,
            )
            ax.set_title(f"{gene_set_name} — Gene × Disease Z-score Heatmap",
                         fontsize=12, fontweight="bold")
            ax.set_ylabel("Gene"); ax.set_xlabel("Disease")
            plt.tight_layout()
            _save(fig, "02_profile_heatmap")
    except Exception as e:
        print(f"[⚠] Plot 02 (profile heatmap) failed: {e}")
        plt.close("all")

    # ---- 03  Grouped bar chart ----------------------------------
    try:
        if isinstance(pm, pd.DataFrame) and not pm.empty:
            order = pm.loc[pm.abs().mean(axis=1).sort_values(ascending=True).index]
            fig, ax = plt.subplots(figsize=(max(9, len(order) * 0.55), 6))
            x = np.arange(len(order))
            w = 0.8 / len(diseases)
            for i, d in enumerate(diseases):
                off = (i - len(diseases) / 2 + 0.5) * w
                ax.bar(x + off, order[d], w, label=d,
                       color=dcol[d], edgecolor="black", linewidth=0.3)
            ax.set_xticks(x)
            ax.set_xticklabels(order.index, rotation=45, ha="right", fontsize=8)
            ax.axhline(0, color="grey", lw=0.8)
            ax.set_ylabel("meta-Z score")
            ax.set_title(f"{gene_set_name} — Gene-level Z-scores by Disease",
                         fontsize=12, fontweight="bold")
            ax.legend(title="Disease")
            plt.tight_layout()
            _save(fig, "03_gene_barplot")
    except Exception as e:
        print(f"[⚠] Plot 03 (gene barplot) failed: {e}")
        plt.close("all")

    # ---- 04  Pairwise scatter ------------------------------------
    try:
        if isinstance(pm, pd.DataFrame) and not pm.empty and len(diseases) >= 2:
            pairs  = list(combinations(diseases, 2))
            fig, axes_sc = plt.subplots(1, len(pairs), figsize=(6 * len(pairs), 5.5))
            if len(pairs) == 1:
                axes_sc = [axes_sc]
            prox = results.get("proximity_df", None)

            for idx, (d1, d2) in enumerate(pairs):
                ax = axes_sc[idx]
                xv, yv = pm[d1].values, pm[d2].values
                ax.scatter(xv, yv, c="steelblue", edgecolor="black", lw=0.3, s=50, alpha=0.8)
                for g, xi, yi in zip(pm.index, xv, yv):
                    ax.annotate(g, (xi, yi), fontsize=6, alpha=0.7,
                                xytext=(3, 3), textcoords="offset points")
                m = ~(np.isnan(xv) | np.isnan(yv))
                if m.sum() > 2:
                    coef = np.polyfit(xv[m], yv[m], 1)
                    xl   = np.linspace(xv[m].min(), xv[m].max(), 100)
                    ax.plot(xl, np.polyval(coef, xl), "r--", alpha=0.6, lw=1)
                if prox is not None and isinstance(prox, pd.DataFrame) and not prox.empty:
                    row = prox[((prox["Disease_1"] == d1) & (prox["Disease_2"] == d2)) |
                               ((prox["Disease_1"] == d2) & (prox["Disease_2"] == d1))]
                    if not row.empty:
                        r = row["Pearson_r"].values[0]; p = row["Pearson_p"].values[0]
                        ax.set_title(f"{d1} vs {d2}\nr = {r:.3f}, p = {p:.4f}", fontsize=11)
                    else:
                        ax.set_title(f"{d1} vs {d2}")
                else:
                    ax.set_title(f"{d1} vs {d2}")
                ax.set_xlabel(f"{d1} meta-Z"); ax.set_ylabel(f"{d2} meta-Z")
                ax.axhline(0, color="grey", lw=0.5, ls="--")
                ax.axvline(0, color="grey", lw=0.5, ls="--")

            fig.suptitle(f"{gene_set_name} — Pairwise Z-score Correlations",
                         fontsize=13, fontweight="bold")
            plt.tight_layout()
            _save(fig, "04_pairwise_scatter")
    except Exception as e:
        print(f"[⚠] Plot 04 (pairwise scatter) failed: {e}")
        plt.close("all")

    # ---- 05  Z-range lollipop -----------------------------------
    try:
        gdf = results.get("gene_detail_df", pd.DataFrame())
        if isinstance(gdf, pd.DataFrame) and not gdf.empty and "Z_range" in gdf.columns:
            gdf = gdf.dropna(subset=["Z_range"]).sort_values("Z_range", ascending=False)
            if not gdf.empty:
                fig, ax = plt.subplots(figsize=(8, max(5, len(gdf) * 0.38)))
                y = np.arange(len(gdf))
                ax.barh(y, gdf["Z_range"], color="salmon", edgecolor="black", lw=0.3, height=0.6)
                ax.set_yticks(y)
                ax.set_yticklabels(gdf["Gene"], fontsize=9)
                ax.set_xlabel("Z-score range (max − min across diseases)")
                ax.set_title(f"{gene_set_name} — Cross-Disease Variability per Gene",
                             fontsize=12, fontweight="bold")
                ax.invert_yaxis()
                plt.tight_layout()
                _save(fig, "05_gene_z_range")
    except Exception as e:
        print(f"[⚠] Plot 05 (gene z-range) failed: {e}")
        plt.close("all")

    # ---- 06  Box + strip plot -----------------------------------
    try:
        if isinstance(pm, pd.DataFrame) and not pm.empty:
            fig, ax = plt.subplots(figsize=(max(6, len(diseases) * 2.2), 5.5))
            bp = ax.boxplot(
                [pm[d].values for d in diseases],
                labels=diseases, patch_artist=True, showmeans=True,
                meanprops=dict(marker="D", markerfacecolor="red", markersize=6),
            )
            for patch, d in zip(bp["boxes"], diseases):
                patch.set_facecolor(dcol[d]); patch.set_alpha(0.55)
            for i, d in enumerate(diseases):
                jitter = np.random.default_rng(42).normal(0, 0.04, len(pm))
                ax.scatter(
                    np.full(len(pm), i + 1) + jitter, pm[d].values,
                    c="black", s=18, alpha=0.5, zorder=3,
                )
            ax.axhline(0, color="grey", lw=0.8, ls="--")
            ax.set_ylabel("meta-Z score")
            ax.set_title(f"{gene_set_name} — Z-score Distributions",
                         fontsize=12, fontweight="bold")
            plt.tight_layout()
            _save(fig, "06_z_distribution_boxplot")
    except Exception as e:
        print(f"[⚠] Plot 06 (z-distribution boxplot) failed: {e}")
        plt.close("all")

    # ---- 07  Enrichment metrics comparison -----------------------
    try:
        if len(diseases) >= 2 and not edf.empty:
            metrics = ["stouffer_z", "mean_z", "median_z", "mean_abs_z"]
            metric_labels = ["Stouffer Z", "Mean Z", "Median Z", "Mean |Z|"]
            fig, ax = plt.subplots(figsize=(max(10, len(diseases) * 2.5), 5))
            x = np.arange(len(metrics))
            w = 0.8 / len(diseases)
            for i, d in enumerate(diseases):
                row  = edf[edf["Disease"] == d].iloc[0]
                vals = [float(row[m]) for m in metrics]
                off  = (i - len(diseases) / 2 + 0.5) * w
                ax.bar(x + off, vals, w, label=d,
                       color=dcol[d], edgecolor="black", linewidth=0.3)
            ax.set_xticks(x)
            ax.set_xticklabels(metric_labels, fontsize=10)
            ax.axhline(0, color="grey", lw=0.8, ls="--")
            ax.set_ylabel("Value")
            ax.set_title(f"{gene_set_name} — Enrichment Metrics Comparison",
                         fontsize=12, fontweight="bold")
            ax.legend(title="Disease")
            plt.tight_layout()
            _save(fig, "07_enrichment_metrics")
    except Exception as e:
        print(f"[⚠] Plot 07 (enrichment metrics) failed: {e}")
        plt.close("all")

    # ---- 08  Pairwise stats heatmap -----------------------------
    try:
        pw_stats = results.get("pairwise_stats_df", pd.DataFrame())
        if isinstance(pw_stats, pd.DataFrame) and not pw_stats.empty and len(diseases) >= 2:
            p_candidates = [
                ("KS_p",               "KS 2-sample"),
                ("Welch_p",            "Welch t"),
                ("Paired_Wilcoxon_p",  "Paired Wilcoxon"),
                ("Paired_t_p",         "Paired t"),
                ("Permutation_p",      "Permutation"),
            ]
            p_metrics = [(col, lbl) for col, lbl in p_candidates
                         if col in pw_stats.columns and pw_stats[col].notna().any()]
            d_candidates = [
                ("Cohens_d_unpaired", "Cohen's d (unpaired)"),
                ("Cohens_d_paired",   "Cohen's d (paired)"),
            ]
            d_metrics = [(col, lbl) for col, lbl in d_candidates
                         if col in pw_stats.columns and pw_stats[col].notna().any()]
            n_panels = len(p_metrics) + len(d_metrics)
            if n_panels > 0:
                fig, axes_pw = plt.subplots(
                    1, n_panels,
                    figsize=(max(5 * n_panels, 10), max(4, len(diseases) * 1.5))
                )
                if n_panels == 1:
                    axes_pw = [axes_pw]
                panel_idx = 0
                for col, lbl in p_metrics:
                    ax = axes_pw[panel_idx]; panel_idx += 1
                    mat = pd.DataFrame(np.nan, index=diseases, columns=diseases)
                    for _, rw in pw_stats.iterrows():
                        d1, d2 = rw["Disease_1"], rw["Disease_2"]
                        val = rw[col]
                        if d1 in diseases and d2 in diseases and not np.isnan(val):
                            neg_log = -np.log10(max(val, 1e-300))
                            mat.loc[d1, d2] = neg_log
                            mat.loc[d2, d1] = neg_log
                    for d in diseases:
                        mat.loc[d, d] = 0.0
                    sns.heatmap(mat.astype(float), annot=True, fmt=".2f", cmap="YlOrRd",
                                square=True, ax=ax, linewidths=0.5, linecolor="white", vmin=0)
                    ax.set_title(f"−log₁₀(p)\n{lbl}", fontsize=10)
                for col, lbl in d_metrics:
                    ax = axes_pw[panel_idx]; panel_idx += 1
                    mat = pd.DataFrame(0.0, index=diseases, columns=diseases)
                    for _, rw in pw_stats.iterrows():
                        d1, d2 = rw["Disease_1"], rw["Disease_2"]
                        val = rw[col]
                        if d1 in diseases and d2 in diseases and not np.isnan(val):
                            mat.loc[d1, d2] =  val
                            mat.loc[d2, d1] = -val
                    vabs = max(abs(mat.values.min()), abs(mat.values.max()), 0.01)
                    sns.heatmap(mat.astype(float), annot=True, fmt=".2f", cmap="RdBu_r",
                                center=0, vmin=-vabs, vmax=vabs, square=True, ax=ax,
                                linewidths=0.5, linecolor="white")
                    ax.set_title(lbl, fontsize=10)
                fig.suptitle(f"{gene_set_name} — Pairwise Disease Stat Tests",
                             fontsize=13, fontweight="bold")
                plt.tight_layout()
                _save(fig, "08_pairwise_stats_heatmap")
    except Exception as e:
        print(f"[⚠] Plot 08 (pairwise stats heatmap) failed: {e}")
        plt.close("all")

    # ---- 09  Concordance heatmap --------------------------------
    try:
        pw_stats = results.get("pairwise_stats_df", pd.DataFrame())
        if isinstance(pw_stats, pd.DataFrame) and not pw_stats.empty and len(diseases) >= 2:
            conc_candidates = [
                ("sign_concordance_rate", "Sign Concordance Rate", "YlGn",    0.0, 1.0, 0.5),
                ("Lin_CCC",               "Lin's CCC",            "RdBu_r", -1.0, 1.0, 0.0),
                ("Kendall_tau",            "Kendall τ",            "RdBu_r", -1.0, 1.0, 0.0),
            ]
            conc_metrics_avail = [
                (col, lbl, cmap, vmin, vmax, center)
                for col, lbl, cmap, vmin, vmax, center in conc_candidates
                if col in pw_stats.columns and pw_stats[col].notna().any()
            ]
            conc_p_avail = []
            if "sign_concordance_p" in pw_stats.columns and pw_stats["sign_concordance_p"].notna().any():
                conc_p_avail.append(("sign_concordance_p", "Sign Conc. −log₁₀(p)"))
            if "Kendall_p" in pw_stats.columns and pw_stats["Kendall_p"].notna().any():
                conc_p_avail.append(("Kendall_p", "Kendall τ −log₁₀(p)"))
            n_panels = len(conc_metrics_avail) + len(conc_p_avail)
            if n_panels > 0:
                fig, axes_c = plt.subplots(
                    1, n_panels,
                    figsize=(max(5 * n_panels, 10), max(4, len(diseases) * 1.5))
                )
                if n_panels == 1:
                    axes_c = [axes_c]
                pi = 0
                for col, lbl, cmap, vmin_val, vmax_val, center_val in conc_metrics_avail:
                    ax = axes_c[pi]; pi += 1
                    mat = pd.DataFrame(1.0, index=diseases, columns=diseases)
                    for _, rw in pw_stats.iterrows():
                        d1, d2 = rw["Disease_1"], rw["Disease_2"]
                        val = rw[col]
                        if d1 in diseases and d2 in diseases and not np.isnan(val):
                            mat.loc[d1, d2] = val
                            mat.loc[d2, d1] = val
                    sns.heatmap(
                        mat.astype(float), annot=True, fmt=".3f", cmap=cmap,
                        center=center_val, vmin=vmin_val, vmax=vmax_val,
                        square=True, ax=ax, linewidths=0.5, linecolor="white",
                    )
                    ax.set_title(lbl, fontsize=10)
                for col, lbl in conc_p_avail:
                    ax = axes_c[pi]; pi += 1
                    mat = pd.DataFrame(np.nan, index=diseases, columns=diseases)
                    for _, rw in pw_stats.iterrows():
                        d1, d2 = rw["Disease_1"], rw["Disease_2"]
                        val = rw[col]
                        if d1 in diseases and d2 in diseases and not np.isnan(val):
                            neg_log = -np.log10(max(val, 1e-300))
                            mat.loc[d1, d2] = neg_log
                            mat.loc[d2, d1] = neg_log
                    for d in diseases:
                        mat.loc[d, d] = 0.0
                    sns.heatmap(
                        mat.astype(float), annot=True, fmt=".2f", cmap="YlOrRd",
                        square=True, ax=ax, linewidths=0.5, linecolor="white", vmin=0,
                    )
                    ax.set_title(lbl, fontsize=10)
                fig.suptitle(f"{gene_set_name} — Concordance Metrics",
                             fontsize=13, fontweight="bold")
                plt.tight_layout()
                _save(fig, "09_concordance_heatmap")
    except Exception as e:
        print(f"[⚠] Plot 09 (concordance heatmap) failed: {e}")
        plt.close("all")

    # ---- 10  QQ-plot (NEW) --------------------------------------
    try:
        meta_z = results.get("meta_z", {})
        gene_list = results.get("gene_list", [])
        if meta_z and gene_list:
            diseases_local = list(meta_z.keys())
            n = len(diseases_local)
            fig, axes = plt.subplots(1, n, figsize=(max(5 * n, 6), 5))
            if n == 1:
                axes = [axes]
            for ax, d in zip(axes, diseases_local):
                gs_z = np.array([meta_z[d].get(g, np.nan) for g in gene_list])
                gs_z = gs_z[~np.isnan(gs_z)]
                if len(gs_z) < 3:
                    ax.text(0.5, 0.5, f"{d}: n<3", ha="center", va="center",
                            transform=ax.transAxes); continue
                observed = np.sort(gs_z)
                theoretical = stats.norm.ppf((np.arange(1, len(observed) + 1) - 0.5) / len(observed))
                ax.scatter(theoretical, observed, alpha=0.7, s=30,
                           c=dcol.get(d, "steelblue"), edgecolor="black", lw=0.3)
                lims = [min(theoretical.min(), observed.min()),
                        max(theoretical.max(), observed.max())]
                ax.plot(lims, lims, "r--", lw=1, label="N(0,1) reference")
                ax.axhline(0, color="gray", lw=0.5, ls=":")
                ax.set_xlabel("Theoretical N(0,1) quantile")
                ax.set_ylabel("Observed gene-set Z")
                ax.set_title(f"{d} (n={len(observed)})", fontsize=11)
                ax.legend(fontsize=8)
            fig.suptitle(f"{gene_set_name} — QQ-plot vs N(0,1)",
                         fontsize=13, fontweight="bold")
            plt.tight_layout()
            _save(fig, "10_qq_plot")
    except Exception as e:
        print(f"[⚠] Plot 10 (QQ-plot) failed: {e}")
        plt.close("all")

    # ---- 11  Dendrogram (NEW) ------------------------------------
    try:
        prox = results.get("proximity_df", pd.DataFrame())
        if isinstance(prox, pd.DataFrame) and not prox.empty and len(diseases) >= 3:
            from scipy.cluster.hierarchy import linkage, dendrogram
            from scipy.spatial.distance import squareform
            n = len(diseases)
            dmat = np.zeros((n, n))
            idx = {d: i for i, d in enumerate(diseases)}
            for _, row in prox.iterrows():
                d1, d2 = row["Disease_1"], row["Disease_2"]
                if d1 in idx and d2 in idx:
                    dist = max(0.0, 1 - float(row["Pearson_r"]))
                    dmat[idx[d1], idx[d2]] = dist
                    dmat[idx[d2], idx[d1]] = dist
            condensed = squareform(dmat, checks=False)
            Z = linkage(condensed, method="average")
            fig, ax = plt.subplots(figsize=(max(6, n * 1.3), 5))
            dendrogram(Z, labels=diseases, ax=ax, color_threshold=0)
            ax.set_ylabel("Distance (1 − Pearson r)")
            ax.set_title(f"{gene_set_name} — Disease Hierarchical Clustering",
                         fontsize=12, fontweight="bold")
            plt.tight_layout()
            _save(fig, "11_dendrogram")
    except Exception as e:
        print(f"[⚠] Plot 11 (dendrogram) failed: {e}")
        plt.close("all")

    # ---- 12  Gene influence (NEW) --------------------------------
    try:
        inf = results.get("influence_df", pd.DataFrame())
        if isinstance(inf, pd.DataFrame) and not inf.empty:
            diseases_local = list(inf["Disease"].unique())
            n = len(diseases_local)
            fig, axes = plt.subplots(1, n, figsize=(max(6 * n, 8), max(5, 15 * 0.35)))
            if n == 1:
                axes = [axes]
            for ax, d in zip(axes, diseases_local):
                sub = inf[inf["Disease"] == d].copy()
                sub = sub.reindex(sub["Delta_Stouffer"].abs().sort_values(ascending=False).index)
                sub = sub.head(15).iloc[::-1]
                if sub.empty:
                    continue
                colors = ["#e74c3c" if x else "#3498db" for x in sub["Influential"]]
                ax.barh(sub["Gene"], sub["Delta_Stouffer"], color=colors,
                        edgecolor="black", lw=0.3)
                ax.axvline(0, color="gray", ls="--", lw=0.8)
                ax.set_xlabel("Δ Stouffer Z (full − LOO)")
                ax.set_title(f"{d}: top influential genes", fontsize=11)
            fig.suptitle(f"{gene_set_name} — Gene-level LOO Influence",
                         fontsize=13, fontweight="bold")
            plt.tight_layout()
            _save(fig, "12_gene_influence")
    except Exception as e:
        print(f"[⚠] Plot 12 (gene influence) failed: {e}")
        plt.close("all")

    n_plots = len([f for f in os.listdir(plots_dir) if f.endswith(".png")]) if os.path.isdir(plots_dir) else 0
    print(f"[✓] plots/   ({n_plots} PNGs + {n_plots} PDFs)")
    print(f"\n{'='*60}")
    print(f"  Report saved → {os.path.abspath(output_dir)}/")
    print(f"{'='*60}")

    return output_dir



### Loading

In [ ]:
!cp "/content/drive/MyDrive/Dr Uccello/00_Studies/Z_proximity/proximity/scz2026_eur" /content/ -r
!cp "/content/drive/MyDrive/Dr Uccello/00_Studies/Z_proximity/proximity/scz_cloz" /content/ -r

In [ ]:
!cp "/content/drive/MyDrive/Dr Uccello/00_Studies/Z_GMT/psychedelics_GSE161626" /content -r

# Combining gene sets

In [ ]:
import os
import glob
import re

# Folder containing your GMT and/or TXT files
folder_path = "/content/psychedelics_GSE161626/GOBP"  # <-- change this to your folder

# Dictionary to hold all gene sets {SET_NAME: [genes...]}
gene_sets = {}

def sanitize(name: str) -> str:
    """Make a string safe to use as a Python variable name (UPPER_CASE)."""
    name = re.sub(r'[^0-9a-zA-Z_]', '_', name)   # replace illegal chars
    name = re.sub(r'_+', '_', name).strip('_')   # collapse underscores
    if name and name[0].isdigit():
        name = "_" + name
    return name.upper()

def parse_gmt_file(file_path: str):
    """
    Parse a .gmt file.

    GMT format:
        SET_NAME <tab> DESCRIPTION <tab> GENE1 <tab> GENE2 ...
    """
    parsed_sets = {}

    with open(file_path, "r") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue

            parts = line.split("\t")

            # GMT lines should have at least:
            # set name, description, one gene
            if len(parts) < 3:
                continue

            set_name = sanitize(parts[0])
            genes = [g.strip() for g in parts[2:] if g.strip()]

            if genes:
                parsed_sets[set_name] = genes

    return parsed_sets

def parse_txt_file(file_path: str):
    """
    Parse a .txt file.

    Assumes one gene per line by default.

    Example:
        GeneA
        GeneB
        GeneC

    The gene-set name is taken from the TXT filename.
    """
    set_name = sanitize(os.path.splitext(os.path.basename(file_path))[0])
    genes = []

    with open(file_path, "r") as f:
        for line in f:
            line = line.strip()

            if not line:
                continue

            # Skip common header-like lines if present
            if line.lower() in {"gene", "genes", "symbol", "gene_symbol"}:
                continue

            genes.append(line)

    return {set_name: genes} if genes else {}

# Find all .gmt and .txt files in the folder
gmt_files = glob.glob(os.path.join(folder_path, "*.gmt"))
txt_files = glob.glob(os.path.join(folder_path, "*.txt"))

all_files = gmt_files + txt_files

print(f"Found {len(gmt_files)} GMT files")
print(f"Found {len(txt_files)} TXT files")
print(f"Found {len(all_files)} total files\n")

for file_path in all_files:
    ext = os.path.splitext(file_path)[1].lower()

    if ext == ".gmt":
        parsed = parse_gmt_file(file_path)
    elif ext == ".txt":
        parsed = parse_txt_file(file_path)
    else:
        continue

    gene_sets.update(parsed)

# ---- Inject each gene set as a global Python variable ----
for set_name, genes in gene_sets.items():
    globals()[set_name] = genes

print(f"Loaded {len(gene_sets)} gene sets as Python variables.\n")

for set_name, genes in list(gene_sets.items())[:5]:
    print(f"{set_name}: {len(genes)} genes — first 5: {genes[:5]}")

# ---- Optional: write them out as a .py file in the desired format ----
output_py = "/content/gene_sets.py"

def format_list(genes, per_line=8, indent="    "):
    lines = []
    for i in range(0, len(genes), per_line):
        chunk = ", ".join(f'"{g}"' for g in genes[i:i+per_line])
        lines.append(indent + chunk)
    return ",\n".join(lines)

with open(output_py, "w") as out:
    out.write("# Auto-generated gene sets from GMT and TXT files\n\n")

    for set_name, genes in gene_sets.items():
        out.write(f"{set_name} = [\n{format_list(genes)}\n]\n\n")

print(f"\nAll gene sets written to: {output_py}")

Found 1 GMT files
Found 0 TXT files
Found 1 total files

Loaded 33 gene sets as Python variables.

SYNAPSE_ORGANIZATION_GO_0050808: 56 genes — first 5: ['ADGRL3', 'AGRN', 'ANK3', 'APBB2', 'APP']
CELL_CELL_ADHESION_VIA_PLASMA_MEMBRANE_ADHESION_MOLECULES_GO_0098742: 66 genes — first 5: ['ADGRL1', 'ADGRL3', 'ALCAM', 'AMIGO1', 'CADM1']
NERVOUS_SYSTEM_DEVELOPMENT_GO_0007399: 136 genes — first 5: ['ADAM23', 'ADGRA2', 'ADGRB1', 'ADGRL3', 'ADGRV1']
POSITIVE_REGULATION_OF_CELL_PROJECTION_ORGANIZATION_GO_0031346: 47 genes — first 5: ['ALKAL2', 'AMIGO1', 'ARSB', 'ATP1B2', 'CAMK2B']
NEURON_PROJECTION_MORPHOGENESIS_GO_0048812: 54 genes — first 5: ['ADGRB1', 'ALCAM', 'ALS2', 'ANK3', 'APP']

All gene sets written to: /content/gene_sets.py


# Summary Function

In [ ]:
# ===================================================================
#  11.  SIGNIFICANT-RESULTS SUMMARY  (post-hoc on an existing run)
# ===================================================================

def generate_sig_summary(
    results_dir,
    output_path=None,
    # ── significance thresholds (all optional / tunable) ──────────
    p_threshold=0.05,
    fdr_threshold=0.05,
    stouffer_z_threshold=1.96,
    cohens_d_threshold=0.5,
    ccc_threshold=0.5,
    sign_concordance_threshold=0.7,
    kendall_tau_threshold=0.3,
    pearson_r_threshold=0.3,
    rank_biserial_threshold=0.3,
    use_fdr=True,
    include_borderline=False,
    top_n_per_section=None,
    verbose=True,
):
    """
    Walk a pipeline output folder and write a `sig_summary.txt` listing
    every significant finding across all CSVs the pipeline produced.

    Works for BOTH:
      * single-gene-set runs    (`run_pipeline()` output folder)
      * multi-gene-set runs     (`run_multi_geneset_pipeline()` output
                                 folder — including all sub-folders)

    Sections written:
      1. Per-disease enrichment       (Stouffer Z, permutation/Wilcoxon p)
      2. Differential tests           (Kruskal-Wallis, MWU, paired Wilcoxon)
      3. Profile proximity            (Pearson r, Spearman ρ)
      4. Pairwise disease tests       (KS, Welch, paired t/Wilcoxon, perm,
                                       Brunner-Munzel, Levene, Cohen's d)
      5. Concordance metrics          (sign rate, Lin's CCC, Kendall τ)
      6. Gene-level LOO influence     (genes flagged as Influential)
      7. Family-wise FDR              (if computed in the run)

    Parameters
    ----------
    results_dir : str | Path
        Folder produced by the pipeline.
    output_path : str | Path | None
        Destination text file.  Defaults to `<results_dir>/sig_summary.txt`.
    p_threshold, fdr_threshold : float
        Cut-offs for raw p and BH-FDR/global-FDR.
    stouffer_z_threshold, cohens_d_threshold, ccc_threshold,
    sign_concordance_threshold, kendall_tau_threshold,
    pearson_r_threshold, rank_biserial_threshold : float
        Effect-size thresholds (absolute value) used in addition to p-values.
    use_fdr : bool
        If True (default) and an FDR column exists, gate significance on
        FDR rather than raw p-value.  Raw p is still reported.
    include_borderline : bool
        If True, also include rows that pass *any* effect-size threshold
        even when the corresponding p/FDR is missing.
    top_n_per_section : int | None
        If given, truncate each section to its top-N strongest hits.
    verbose : bool
        Print progress to stdout.

    Returns
    -------
    pathlib.Path
        Absolute path of the written `sig_summary.txt`.
    """
    results_dir = Path(results_dir)
    if not results_dir.is_dir():
        raise FileNotFoundError(f"{results_dir} is not a directory")

    if output_path is None:
        output_path = results_dir / "sig_summary.txt"
    output_path = Path(output_path)

    _print = print if verbose else (lambda *a, **k: None)

    # ── helpers ───────────────────────────────────────────────────
    def _fmt(v, fmt=".4g", na="NA"):
        try:
            if v is None or (isinstance(v, float) and np.isnan(v)):
                return na
            return format(float(v), fmt)
        except Exception:
            return na

    def _safe_read(path):
        try:
            return pd.read_csv(path)
        except Exception as e:
            _print(f"  ⚠ could not read {path.name}: {e}")
            return pd.DataFrame()

    def _gs_from_filename(path, prefix):
        stem = path.stem
        if stem.startswith(prefix):
            return stem[len(prefix):]
        return stem

    def _ensure_geneset_col(df, fallback_label):
        """Make sure the dataframe has a 'GeneSet' column."""
        if "GeneSet" not in df.columns:
            df = df.copy()
            df.insert(0, "GeneSet", fallback_label)
        return df

    def _is_multi_mode():
        return any((results_dir / x).exists() for x in (
            "multi_geneset_enrichment_summary.csv",
            "multi_geneset_pairwise_disease_stats.csv",
            "multi_geneset_gene_influence.csv",
            "multi_geneset_run_manifest.json",
        ))

    multi_mode = _is_multi_mode()

    # ── header ────────────────────────────────────────────────────
    lines = []
    bar = "=" * 78
    lines += [
        bar,
        "  SIGNIFICANT RESULTS SUMMARY",
        f"  Source folder : {results_dir.resolve()}",
        f"  Mode detected : {'multi-gene-set' if multi_mode else 'single-gene-set'}",
        f"  Generated     : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}",
        bar,
        "",
        "  THRESHOLDS:",
        f"    raw p-value             : < {p_threshold}",
        f"    FDR / global FDR        : < {fdr_threshold}",
        f"    Filter on FDR if avail. : {use_fdr}",
        f"    |Stouffer Z|            : > {stouffer_z_threshold}",
        f"    |Cohen's d|             : > {cohens_d_threshold}",
        f"    |Lin's CCC|             : > {ccc_threshold}",
        f"    Sign-concordance rate   : > {sign_concordance_threshold}",
        f"    |Kendall τ|             : > {kendall_tau_threshold}",
        f"    |Pearson r|             : > {pearson_r_threshold}",
        f"    |Rank-biserial r|       : > {rank_biserial_threshold}",
        f"    include borderline      : {include_borderline}",
        f"    top-N per section       : {top_n_per_section}",
        "",
    ]

    section_counts = {}

    # ==============================================================
    # SECTION 1 — Per-disease enrichment
    # ==============================================================
    lines += [bar, "  [1] PER-DISEASE ENRICHMENT", bar]

    enrich_dfs = []
    if multi_mode:
        f = results_dir / "multi_geneset_enrichment_summary.csv"
        if f.exists():
            df = _safe_read(f)
            if not df.empty:
                enrich_dfs.append(df)
    # Always also pick up per-set files (works for single-mode too)
    for f in sorted(results_dir.rglob("enrichment_*.csv")):
        if f.name == "multi_geneset_enrichment_summary.csv":
            continue
        df = _safe_read(f)
        if not df.empty:
            df = _ensure_geneset_col(df, _gs_from_filename(f, "enrichment_"))
            if "Tissue" not in df.columns:
                df["Tissue"] = "meta_across_tissues"
            enrich_dfs.append(df)

    if not enrich_dfs:
        lines += ["  (no enrichment CSVs found)", ""]
    else:
        e_all = pd.concat(enrich_dfs, ignore_index=True, sort=False)
        # Deduplicate
        dedup_cols = [c for c in ("GeneSet", "Disease", "Tissue") if c in e_all.columns]
        if dedup_cols:
            e_all = e_all.drop_duplicates(subset=dedup_cols, keep="first")

        sig_rows = []
        for _, row in e_all.iterrows():
            stouffer = row.get("stouffer_z", np.nan)
            perm_p   = row.get("permutation_p", np.nan)
            wil_p    = row.get("wilcoxon_p", np.nan)

            reasons = []
            if pd.notna(perm_p) and perm_p < p_threshold:
                reasons.append(f"perm_p={_fmt(perm_p)}")
            if pd.notna(wil_p) and wil_p < p_threshold:
                reasons.append(f"wilcoxon_p={_fmt(wil_p)}")
            if pd.notna(stouffer) and abs(stouffer) > stouffer_z_threshold:
                reasons.append(f"|Stouffer|={abs(stouffer):.2f}")
            if not reasons:
                continue

            ci_lo = row.get("stouffer_ci_low", np.nan)
            ci_hi = row.get("stouffer_ci_high", np.nan)
            ci_str = (f" CI95=[{_fmt(ci_lo,'.2f')},{_fmt(ci_hi,'.2f')}]"
                      if pd.notna(ci_lo) and pd.notna(ci_hi) else "")

            sig_rows.append({
                "GeneSet": row.get("GeneSet", "?"),
                "Disease": row.get("Disease", "?"),
                "Tissue":  row.get("Tissue", "meta_across_tissues"),
                "n":       int(row.get("n_genes_found", 0) or 0),
                "stouffer_z": float(stouffer) if pd.notna(stouffer) else np.nan,
                "perm_p":  perm_p,
                "wil_p":   wil_p,
                "ci_str":  ci_str,
                "_sort":   (perm_p if pd.notna(perm_p)
                            else (wil_p if pd.notna(wil_p) else 1.0)),
                "reasons": "; ".join(reasons),
            })

        sig_rows.sort(key=lambda r: r["_sort"])
        if top_n_per_section:
            sig_rows = sig_rows[:top_n_per_section]
        section_counts["enrichment"] = len(sig_rows)

        if sig_rows:
            lines.append(f"  {len(sig_rows)} significant enrichment finding(s):")
            lines.append("")
            for r in sig_rows:
                tissue_blurb = (f" [{r['Tissue']}]"
                                if r['Tissue'] not in ("meta_across_tissues", "meta", "?")
                                else "")
                lines.append(
                    f"    • [{r['GeneSet']}]  {r['Disease']}{tissue_blurb}  "
                    f"n={r['n']}  Stouffer Z={_fmt(r['stouffer_z'],'.3f')}{r['ci_str']}  "
                    f"perm_p={_fmt(r['perm_p'])}  wilcoxon_p={_fmt(r['wil_p'])}"
                )
                lines.append(f"        → {r['reasons']}")
        else:
            lines.append("  (no significant enrichment at given thresholds)")
        lines.append("")

    # ==============================================================
    # SECTION 2 — Differential tests across diseases
    # ==============================================================
    lines += [bar, "  [2] DIFFERENTIAL TESTS (Kruskal-Wallis + pairwise)", bar]

    diff_files = sorted(results_dir.rglob("differential_pairwise_*.csv"))
    if not diff_files:
        lines += ["  (no differential_pairwise_*.csv files found)", ""]
    else:
        sig_rows = []
        for f in diff_files:
            df = _safe_read(f)
            if df.empty:
                continue
            df = _ensure_geneset_col(df, _gs_from_filename(f, "differential_pairwise_"))
            for _, row in df.iterrows():
                mwu_p   = row.get("MWU_p", np.nan)
                mwu_fdr = row.get("MWU_FDR", np.nan)
                pw_p    = row.get("Paired_Wilcoxon_p", np.nan)
                pw_fdr  = row.get("Paired_FDR", np.nan)
                rbc     = row.get("Rank_Biserial_r", np.nan)

                gate_mwu = mwu_fdr if (use_fdr and pd.notna(mwu_fdr)) else mwu_p
                gate_pw  = pw_fdr  if (use_fdr and pd.notna(pw_fdr))  else pw_p
                cutoff_mwu = fdr_threshold if (use_fdr and pd.notna(mwu_fdr)) else p_threshold
                cutoff_pw  = fdr_threshold if (use_fdr and pd.notna(pw_fdr))  else p_threshold

                reasons = []
                if pd.notna(gate_mwu) and gate_mwu < cutoff_mwu:
                    reasons.append(f"MWU_{'FDR' if use_fdr and pd.notna(mwu_fdr) else 'p'}={_fmt(gate_mwu)}")
                if pd.notna(gate_pw) and gate_pw < cutoff_pw:
                    reasons.append(f"PairedW_{'FDR' if use_fdr and pd.notna(pw_fdr) else 'p'}={_fmt(gate_pw)}")
                if pd.notna(rbc) and abs(rbc) > rank_biserial_threshold:
                    reasons.append(f"|RBC|={abs(rbc):.2f}")
                if not reasons:
                    continue

                sig_rows.append({
                    "GeneSet": row.get("GeneSet", "?"),
                    "pair":    f"{row.get('Disease_1','?')} vs {row.get('Disease_2','?')}",
                    "mean_z_diff": row.get("Mean_Z_diff", np.nan),
                    "mwu_p":    mwu_p, "mwu_fdr": mwu_fdr,
                    "pw_p":     pw_p,  "pw_fdr":  pw_fdr,
                    "rbc":      rbc,
                    "_sort":    (gate_mwu if pd.notna(gate_mwu) else 1.0),
                    "reasons":  "; ".join(reasons),
                })

        sig_rows.sort(key=lambda r: r["_sort"])
        if top_n_per_section:
            sig_rows = sig_rows[:top_n_per_section]
        section_counts["differential"] = len(sig_rows)

        if sig_rows:
            lines.append(f"  {len(sig_rows)} significant differential pair(s):")
            lines.append("")
            for r in sig_rows:
                lines.append(
                    f"    • [{r['GeneSet']}]  {r['pair']}  "
                    f"mean_Z_diff={_fmt(r['mean_z_diff'],'.3f')}  "
                    f"MWU p={_fmt(r['mwu_p'])} (FDR={_fmt(r['mwu_fdr'])})  "
                    f"PairedW p={_fmt(r['pw_p'])}  RBC={_fmt(r['rbc'],'.3f')}"
                )
                lines.append(f"        → {r['reasons']}")
        else:
            lines.append("  (no significant differential results at given thresholds)")
        lines.append("")

    # ==============================================================
    # SECTION 3 — Profile proximity
    # ==============================================================
    lines += [bar, "  [3] PROFILE PROXIMITY (Pearson / Spearman)", bar]

    prox_files = sorted(results_dir.rglob("proximity_*.csv"))
    if not prox_files:
        lines += ["  (no proximity_*.csv files found)", ""]
    else:
        sig_rows = []
        for f in prox_files:
            df = _safe_read(f)
            if df.empty:
                continue
            df = _ensure_geneset_col(df, _gs_from_filename(f, "proximity_"))
            for _, row in df.iterrows():
                pr  = row.get("Pearson_r", np.nan)
                prp = row.get("Pearson_p", np.nan)
                sr  = row.get("Spearman_rho", np.nan)
                srp = row.get("Spearman_p", np.nan)
                cos = row.get("Cosine_sim", np.nan)

                reasons = []
                if pd.notna(prp) and prp < p_threshold and pd.notna(pr) and abs(pr) > pearson_r_threshold:
                    reasons.append(f"Pearson r={pr:.2f} (p={_fmt(prp)})")
                if pd.notna(srp) and srp < p_threshold and pd.notna(sr) and abs(sr) > pearson_r_threshold:
                    reasons.append(f"Spearman ρ={sr:.2f} (p={_fmt(srp)})")
                if not reasons:
                    continue
                sig_rows.append({
                    "GeneSet": row.get("GeneSet", "?"),
                    "pair":    f"{row.get('Disease_1','?')} vs {row.get('Disease_2','?')}",
                    "n":       int(row.get("n_genes", 0) or 0),
                    "pr": pr, "prp": prp, "sr": sr, "srp": srp, "cos": cos,
                    "_sort":   (prp if pd.notna(prp) else 1.0),
                    "reasons": "; ".join(reasons),
                })

        sig_rows.sort(key=lambda r: r["_sort"])
        if top_n_per_section:
            sig_rows = sig_rows[:top_n_per_section]
        section_counts["proximity"] = len(sig_rows)

        if sig_rows:
            lines.append(f"  {len(sig_rows)} significant proximity pair(s):")
            lines.append("")
            for r in sig_rows:
                lines.append(
                    f"    • [{r['GeneSet']}]  {r['pair']}  n={r['n']}  "
                    f"r={_fmt(r['pr'],'.3f')} (p={_fmt(r['prp'])})  "
                    f"ρ={_fmt(r['sr'],'.3f')} (p={_fmt(r['srp'])})  "
                    f"cos={_fmt(r['cos'],'.3f')}"
                )
                lines.append(f"        → {r['reasons']}")
        else:
            lines.append("  (no significant proximity results at given thresholds)")
        lines.append("")

    # ==============================================================
    # SECTION 4 — Pairwise disease statistical tests
    # ==============================================================
    lines += [bar, "  [4] PAIRWISE DISEASE STATISTICAL TESTS", bar]

    pw_files = []
    f_multi = results_dir / "multi_geneset_pairwise_disease_stats.csv"
    if f_multi.exists():
        pw_files.append(f_multi)
    pw_files += [f for f in sorted(results_dir.rglob("pairwise_disease_stats_*.csv"))]

    pw_dfs = []
    for f in pw_files:
        df = _safe_read(f)
        if df.empty:
            continue
        df = _ensure_geneset_col(df, _gs_from_filename(f, "pairwise_disease_stats_"))
        pw_dfs.append(df)

    if not pw_dfs:
        lines += ["  (no pairwise_disease_stats_*.csv files found)", ""]
    else:
        pw_all = pd.concat(pw_dfs, ignore_index=True, sort=False)
        # Deduplicate by GeneSet+pair
        dedup_cols = [c for c in ("GeneSet", "Disease_1", "Disease_2") if c in pw_all.columns]
        if dedup_cols:
            pw_all = pw_all.drop_duplicates(subset=dedup_cols, keep="first")

        # Tests considered (skip concordance — handled in Section 5)
        test_p_cols = [
            ("MWU_p",              "MWU_FDR",           "MWU"),
            ("KS_p",               "KS_FDR",            "KS"),
            ("Welch_p",            "Welch_FDR",         "Welch t"),
            ("Levene_p",           "Levene_FDR",        "Levene"),
            ("BM_p",               "BM_FDR",            "Brunner-Munzel"),
            ("Paired_t_p",         "Paired_t_FDR",      "Paired t"),
            ("Paired_Wilcoxon_p",  "Paired_Wilcoxon_FDR","Paired Wilcoxon"),
            ("Permutation_p",      "Permutation_FDR",   "Sign-flip Perm"),
        ]
        # If multi-mode global FDR exists, prefer it
        for p_col, fdr_col, _ in test_p_cols:
            global_fdr = p_col.replace("_p", "_globalFDR")
            if global_fdr not in pw_all.columns and fdr_col in pw_all.columns:
                pass  # use per-run FDR
            # nothing else to do

        sig_rows = []
        for _, row in pw_all.iterrows():
            pair = f"{row.get('Disease_1','?')} vs {row.get('Disease_2','?')}"
            n_paired = row.get("n_genes_paired", np.nan)
            cohen_p  = row.get("Cohens_d_paired", np.nan)
            cohen_u  = row.get("Cohens_d_unpaired", np.nan)
            mean_diff = row.get("mean_diff", np.nan)

            row_reasons = []
            best_p = 1.0

            for p_col, fdr_col, name in test_p_cols:
                p_val   = row.get(p_col, np.nan)
                fdr_val = row.get(fdr_col, np.nan)
                # Prefer multi-mode globalFDR if present
                gfdr = row.get(p_col.replace("_p", "_globalFDR"), np.nan)
                if pd.notna(gfdr):
                    fdr_val = gfdr
                gate = fdr_val if (use_fdr and pd.notna(fdr_val)) else p_val
                cutoff = (fdr_threshold
                          if (use_fdr and pd.notna(fdr_val)) else p_threshold)
                if pd.notna(gate) and gate < cutoff:
                    label = "FDR" if (use_fdr and pd.notna(fdr_val)) else "p"
                    row_reasons.append(f"{name} {label}={_fmt(gate)}")
                    if pd.notna(p_val) and p_val < best_p:
                        best_p = p_val

            # Effect-size triggers
            if pd.notna(cohen_p) and abs(cohen_p) > cohens_d_threshold:
                row_reasons.append(f"|Cohen's d (paired)|={abs(cohen_p):.2f}")
            if pd.notna(cohen_u) and abs(cohen_u) > cohens_d_threshold:
                row_reasons.append(f"|Cohen's d (unpaired)|={abs(cohen_u):.2f}")

            if not row_reasons:
                continue

            sig_rows.append({
                "GeneSet": row.get("GeneSet", "?"),
                "pair":    pair,
                "n_paired": int(n_paired) if pd.notna(n_paired) else "?",
                "mean_diff": mean_diff,
                "cohen_p":  cohen_p,
                "cohen_u":  cohen_u,
                "_sort":    best_p,
                "reasons":  "; ".join(row_reasons),
            })

        sig_rows.sort(key=lambda r: r["_sort"])
        if top_n_per_section:
            sig_rows = sig_rows[:top_n_per_section]
        section_counts["pairwise_stats"] = len(sig_rows)

        if sig_rows:
            lines.append(f"  {len(sig_rows)} significant pairwise comparison(s):")
            lines.append("")
            for r in sig_rows:
                lines.append(
                    f"    • [{r['GeneSet']}]  {r['pair']}  n_paired={r['n_paired']}  "
                    f"mean_diff={_fmt(r['mean_diff'],'.3f')}  "
                    f"Cohen_d(paired)={_fmt(r['cohen_p'],'.2f')}  "
                    f"Cohen_d(unpaired)={_fmt(r['cohen_u'],'.2f')}"
                )
                lines.append(f"        → {r['reasons']}")
        else:
            lines.append("  (no significant pairwise tests at given thresholds)")
        lines.append("")

    # ==============================================================
    # SECTION 5 — Concordance metrics
    # ==============================================================
    lines += [bar, "  [5] CONCORDANCE METRICS (sign / CCC / Kendall τ)", bar]

    if not pw_dfs:
        lines += ["  (no pairwise files; concordance unavailable)", ""]
    else:
        sig_rows = []
        for _, row in pw_all.iterrows():
            sc_rate = row.get("sign_concordance_rate", np.nan)
            sc_p    = row.get("sign_concordance_p",    np.nan)
            sc_fdr  = row.get("sign_concordance_FDR",  np.nan)
            sc_gfdr = row.get("sign_concordance_globalFDR", np.nan)
            ccc     = row.get("Lin_CCC",     np.nan)
            tau     = row.get("Kendall_tau", np.nan)
            tau_p   = row.get("Kendall_p",   np.nan)
            tau_fdr = row.get("Kendall_FDR", np.nan)
            tau_gfdr= row.get("Kendall_globalFDR", np.nan)

            sc_gate   = sc_gfdr if pd.notna(sc_gfdr) else (sc_fdr if (use_fdr and pd.notna(sc_fdr)) else sc_p)
            sc_cutoff = (fdr_threshold
                         if pd.notna(sc_gfdr) or (use_fdr and pd.notna(sc_fdr))
                         else p_threshold)

            tau_gate   = tau_gfdr if pd.notna(tau_gfdr) else (tau_fdr if (use_fdr and pd.notna(tau_fdr)) else tau_p)
            tau_cutoff = (fdr_threshold
                          if pd.notna(tau_gfdr) or (use_fdr and pd.notna(tau_fdr))
                          else p_threshold)

            reasons = []
            # Sign concordance: significant departure from 0.5 AND |rate-0.5| meaningful
            if (pd.notna(sc_gate) and sc_gate < sc_cutoff
                    and pd.notna(sc_rate)
                    and (sc_rate > sign_concordance_threshold
                         or sc_rate < (1 - sign_concordance_threshold))):
                reasons.append(f"sign_rate={sc_rate:.2f} (p/FDR={_fmt(sc_gate)})")
            # CCC: high agreement
            if pd.notna(ccc) and abs(ccc) > ccc_threshold:
                reasons.append(f"|Lin_CCC|={abs(ccc):.2f}")
            # Kendall τ significant + |τ| above threshold
            if (pd.notna(tau_gate) and tau_gate < tau_cutoff
                    and pd.notna(tau) and abs(tau) > kendall_tau_threshold):
                reasons.append(f"|τ|={abs(tau):.2f} (p/FDR={_fmt(tau_gate)})")

            if not reasons:
                continue

            sig_rows.append({
                "GeneSet": row.get("GeneSet", "?"),
                "pair":    f"{row.get('Disease_1','?')} vs {row.get('Disease_2','?')}",
                "n_paired": int(row.get("n_genes_paired", 0) or 0),
                "sc_rate": sc_rate, "sc_p": sc_p,
                "ccc":     ccc,
                "tau":     tau, "tau_p": tau_p,
                "_sort":   min(
                    sc_p if pd.notna(sc_p) else 1.0,
                    tau_p if pd.notna(tau_p) else 1.0,
                    1.0 - abs(ccc) if pd.notna(ccc) else 1.0,
                ),
                "reasons": "; ".join(reasons),
            })

        sig_rows.sort(key=lambda r: r["_sort"])
        if top_n_per_section:
            sig_rows = sig_rows[:top_n_per_section]
        section_counts["concordance"] = len(sig_rows)

        if sig_rows:
            lines.append(f"  {len(sig_rows)} significant concordance pair(s):")
            lines.append("")
            for r in sig_rows:
                lines.append(
                    f"    • [{r['GeneSet']}]  {r['pair']}  n_paired={r['n_paired']}  "
                    f"sign_rate={_fmt(r['sc_rate'],'.3f')} (p={_fmt(r['sc_p'])})  "
                    f"CCC={_fmt(r['ccc'],'.3f')}  τ={_fmt(r['tau'],'.3f')} (p={_fmt(r['tau_p'])})"
                )
                lines.append(f"        → {r['reasons']}")
        else:
            lines.append("  (no significant concordance results at given thresholds)")
        lines.append("")

    # ==============================================================
    # SECTION 6 — Gene-level LOO influence
    # ==============================================================
    lines += [bar, "  [6] GENE-LEVEL LOO INFLUENCE (flagged genes)", bar]

    inf_files = []
    f_multi = results_dir / "multi_geneset_gene_influence.csv"
    if f_multi.exists():
        inf_files.append(f_multi)
    inf_files += [f for f in sorted(results_dir.rglob("gene_influence_*.csv"))]

    inf_dfs = []
    for f in inf_files:
        df = _safe_read(f)
        if df.empty:
            continue
        df = _ensure_geneset_col(df, _gs_from_filename(f, "gene_influence_"))
        inf_dfs.append(df)

    if not inf_dfs:
        lines += ["  (no gene_influence_*.csv files found)", ""]
    else:
        inf_all = pd.concat(inf_dfs, ignore_index=True, sort=False)
        dedup_cols = [c for c in ("GeneSet", "Disease", "Gene") if c in inf_all.columns]
        if dedup_cols:
            inf_all = inf_all.drop_duplicates(subset=dedup_cols, keep="first")

        flagged = inf_all[inf_all.get("Influential", False).astype(bool)].copy()
        if flagged.empty:
            lines.append("  (no genes flagged as influential)")
        else:
            flagged["_abs_pct"] = flagged["Pct_Change_Stouffer"].abs()
            flagged = flagged.sort_values(
                ["GeneSet", "Disease", "_abs_pct"], ascending=[True, True, False]
            )
            if top_n_per_section:
                flagged = flagged.head(top_n_per_section)

            section_counts["influence"] = len(flagged)
            lines.append(f"  {len(flagged)} influential gene-disease entries:")
            lines.append("")
            for (gs, dis), sub in flagged.groupby(["GeneSet", "Disease"], sort=False):
                lines.append(f"    [{gs}] {dis}:")
                for _, r in sub.iterrows():
                    sign_flag = " (SIGN-FLIP)" if bool(r.get("Sign_Change", False)) else ""
                    lines.append(
                        f"      • {r.get('Gene','?')}: "
                        f"Z={_fmt(r.get('Z_score'),'.2f')}, "
                        f"ΔStouffer={_fmt(r.get('Delta_Stouffer'),'+.2f')} "
                        f"({_fmt(r.get('Pct_Change_Stouffer'),'.1f')}%)"
                        f"{sign_flag}"
                    )
        lines.append("")

    # ==============================================================
    # SECTION 7 — Family-wise FDR pool
    # ==============================================================
    lines += [bar, "  [7] FAMILY-WISE FDR (pooled across all in-run p-values)", bar]

    fw_files = sorted(results_dir.rglob("family_wise_fdr_*.csv"))
    if not fw_files:
        lines += ["  (no family_wise_fdr_*.csv files found — was the run launched with family_wise_fdr=True?)", ""]
    else:
        sig_rows = []
        for f in fw_files:
            df = _safe_read(f)
            if df.empty:
                continue
            df = _ensure_geneset_col(df, _gs_from_filename(f, "family_wise_fdr_"))
            for _, row in df.iterrows():
                fdr = row.get("family_wise_FDR", np.nan)
                p   = row.get("p_value", np.nan)
                if pd.notna(fdr) and fdr < fdr_threshold:
                    sig_rows.append({
                        "GeneSet":    row.get("GeneSet", "?"),
                        "Source":     row.get("Source", "?"),
                        "Test":       row.get("Test", "?"),
                        "Comparison": row.get("Comparison", "?"),
                        "p":   p,
                        "fdr": fdr,
                        "_sort": fdr,
                    })

        sig_rows.sort(key=lambda r: r["_sort"])
        if top_n_per_section:
            sig_rows = sig_rows[:top_n_per_section]
        section_counts["family_wise"] = len(sig_rows)

        if sig_rows:
            lines.append(f"  {len(sig_rows)} family-wise FDR-significant entries:")
            lines.append("")
            for r in sig_rows:
                lines.append(
                    f"    • [{r['GeneSet']}]  {r['Source']}/{r['Test']}  "
                    f"[{r['Comparison']}]  p={_fmt(r['p'])}  FDR={_fmt(r['fdr'])}"
                )
        else:
            lines.append("  (nothing survived family-wise FDR at given threshold)")
        lines.append("")

    # ==============================================================
    # FINAL TALLY
    # ==============================================================
    lines += [bar, "  TOTAL HITS BY SECTION", bar]
    section_titles = {
        "enrichment":      "1. Enrichment            ",
        "differential":    "2. Differential          ",
        "proximity":       "3. Proximity             ",
        "pairwise_stats":  "4. Pairwise stat tests   ",
        "concordance":     "5. Concordance           ",
        "influence":       "6. Influential genes     ",
        "family_wise":     "7. Family-wise FDR       ",
    }
    grand_total = 0
    for k, title in section_titles.items():
        n = section_counts.get(k, 0)
        grand_total += n
        lines.append(f"  {title}: {n}")
    lines.append(f"  ───────────────────────────────")
    lines.append(f"  GRAND TOTAL              : {grand_total}")
    lines.append("")
    lines += [bar]

    # ── write to disk ─────────────────────────────────────────────
    output_path.parent.mkdir(parents=True, exist_ok=True)
    output_path.write_text("\n".join(lines))
    _print(f"\n✓ Significant-results summary written → {output_path.resolve()}")
    _print(f"  Sections: " + ", ".join(
        f"{k}={section_counts.get(k,0)}" for k in section_titles))
    return output_path.resolve()

# Stage 1

In [ ]:
dirs_psy = [
    "/content/scz_cloz",
    "/content/scz2026_eur",
]

lab_psy = [
    "trs",
    "scz_eur",
]

### All-in-one


In [ ]:
multi_psy = run_multi_geneset_pipeline(
    dirs_psy,
    lab_psy,
    gene_sets=gene_sets,
    out="/content/091IvD_results",
    n_perm=10000,
    save_outputs=True,
    plot=True,
    verbose=True,
    family_wise_fdr=True,
    robust=True,
    bootstrap=True,
)

Streaming output truncated to the last 5000 lines.
  → /content/091IvD_results/EPITHELIAL_CELL_MIGRATION_GO_0010631/differential_pairwise_EPITHELIAL_CELL_MIGRATION_GO_0010631.csv
Disease_1 Disease_2  MWU_U    MWU_p  Rank_Biserial_r  Paired_Wilcoxon_p  Mean_Z_diff  MWU_FDR  Paired_FDR
      trs   scz_eur    194 0.704293          -0.0748           0.395462       0.2661 0.704293    0.395462

[5/7] Profile proximity & distance …
  → /content/091IvD_results/EPITHELIAL_CELL_MIGRATION_GO_0010631/proximity_EPITHELIAL_CELL_MIGRATION_GO_0010631.csv
Disease_1 Disease_2  Pearson_r  Pearson_p  Spearman_rho  Spearman_p  Cosine_sim  Euclidean_dist  Manhattan_dist  n_genes
      trs   scz_eur     0.8281   0.000012        0.7965    0.000045      0.8217          5.5906         20.8399       19

[6/7] Pairwise disease statistical tests (gene-set-restricted, robust=True) …
  1 disease pair(s) tested
    trs vs scz_eur: n_paired=19, mean_diff=0.2661, KS p=0.978078, Welch p=0.739873, Paired-t p=0.440496, Co

### Generate Summary

In [ ]:
# Filter on raw p-values rather than FDR
generate_sig_summary("/content/091IvD_results")


✓ Significant-results summary written → /content/091IvD_results/sig_summary.txt
  Sections: enrichment=92, differential=4, proximity=28, pairwise_stats=4, concordance=31, influence=897, family_wise=107


PosixPath('/content/091IvD_results/sig_summary.txt')

# Downstream

In [ ]:
# =====================================================================
# 091IvD DOWNSTREAM — CSV-ONLY ANALYSES + VERY DETAILED SUMMARY
# Consumes ONLY the tables the 091IvD pipeline already wrote.
# No S-PrediXcan folders, no permutations, no GTEx models re-run.
# Run in Colab. Everything in one cell. Ends with DOWNSTREAM/SUMMARY.txt
# =====================================================================

# ---------- 0. Imports ----------
from pathlib import Path
import os, re, glob, textwrap, datetime, warnings, traceback
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

# ---------- CONFIG ----------
RESULTS = Path("/content/091IvD_results")     # <-- CHANGE ME if needed
OUT     = RESULTS / "downstream"
FIG     = OUT / "figures"
OUT.mkdir(parents=True, exist_ok=True)
FIG.mkdir(parents=True, exist_ok=True)

FDR_SIG      = 0.05
PERM_SIG     = 0.05
Z_SIG        = 1.96
CCC_SHARED   = 0.5          # x-band threshold for "shared program"
TS           = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")

# structured results feeding the summary
R = {"meta": {}, "loads": {}, "leader": {}, "shift": {}, "tissue": {},
     "robust": {}, "drivers": {}, "twocohort": {}, "fdr": {}, "pub": {},
     "warnings": []}

def warn(msg):
    print("  [warn]", msg); R["warnings"].append(msg)

# =====================================================================
# 1. ROBUST LOADERS
# =====================================================================
def read_first(patterns):
    """Concatenate every file matching the first pattern that hits."""
    for pat in patterns:
        hits = sorted(RESULTS.rglob(pat))
        if hits:
            frames = []
            for h in hits:
                try:
                    df = pd.read_csv(h)
                    df["__src_file"] = h.name
                    # try to carry a GeneSet label from the filename
                    m = re.search(r"(?:enrichment|gene_detail|gene_influence|"
                                  r"proximity|pairwise_disease_stats|"
                                  r"family_wise_fdr)_(.+)\.csv$", h.name)
                    if m and "GeneSet" not in df.columns and "label" not in df.columns:
                        df["GeneSet"] = m.group(1)
                    frames.append(df)
                except Exception as ex:
                    warn(f"could not read {h.name}: {ex}")
            if frames:
                return pd.concat(frames, ignore_index=True, sort=False)
    return pd.DataFrame()

enrich = read_first(["multi_geneset_enrichment_summary.csv", "enrichment_*.csv"])
pw     = read_first(["multi_geneset_pairwise_disease_stats.csv", "pairwise_disease_stats_*.csv"])
infl   = read_first(["multi_geneset_gene_influence.csv", "gene_influence_*.csv"])
fw     = read_first(["family_wise_fdr*.csv"])
prox   = read_first(["proximity_*.csv"])

# normalise label -> GeneSet
for df in (enrich, pw, infl, fw, prox):
    if not df.empty and "GeneSet" not in df.columns and "label" in df.columns:
        df.rename(columns={"label": "GeneSet"}, inplace=True)

shapes = {"enrichment": enrich.shape, "pairwise": pw.shape,
          "gene_influence": infl.shape, "family_wise_fdr": fw.shape,
          "proximity": prox.shape}
R["loads"] = {k: (v[0], v[1]) for k, v in shapes.items()}
print(">>> Loaded tables:", shapes)

def has(df, *cols):
    return (not df.empty) and all(c in df.columns for c in cols)

def firstcol(df, *cands):
    for c in cands:
        if c in df.columns:
            return c
    return None

# =====================================================================
# §1 CONSERVATIVE ENRICHMENT LEADERBOARD  (CI excludes 0 + FDR)
# =====================================================================
print("\n### §1 Enrichment leaderboard ###")
leader = pd.DataFrame()
try:
    e = enrich.copy()
    if has(e, "Tissue"):
        meta = e[e["Tissue"].fillna("meta").astype(str)
                 .str.contains("meta|across", case=False, na=True)].copy()
        if meta.empty:
            meta = e.copy()
    else:
        meta = e.copy()

    zcol   = firstcol(meta, "stouffer_z", "Stouffer_Z", "stouffer")
    lowcol = firstcol(meta, "stouffer_ci_low", "ci_low", "CI_low")
    hicol  = firstcol(meta, "stouffer_ci_high", "ci_high", "CI_high")
    pcol   = firstcol(meta, "permutation_p", "perm_p", "p_perm")
    wcol   = firstcol(meta, "wilcoxon_p", "wilcox_p")
    ncol   = firstcol(meta, "n_genes_found", "n_found", "n_genes")
    dcol   = firstcol(meta, "Disease", "disease", "Comparison")

    if lowcol and hicol:
        meta["ci_excludes0"] = (
            meta[lowcol].notna() & meta[hicol].notna()
            & (np.sign(meta[lowcol]) == np.sign(meta[hicol]))
            & (meta[lowcol] != 0))
    else:
        meta["ci_excludes0"] = False
        warn("no CI columns; ci_excludes0 set False")

    if zcol:
        meta["direction"] = np.where(meta[zcol] > 0, "up_in_cases", "down_in_cases")

    # attach permutation/wilcoxon FDR from family-wise table
    if not fw.empty and firstcol(fw, "Test"):
        tcol = firstcol(fw, "Test")
        enr_fw = fw[fw[tcol].astype(str)
                    .str.contains("permutation_p|wilcoxon_p", case=False, na=False)].copy()
        fwd = firstcol(enr_fw, "family_wise_FDR", "FDR", "fdr")
        comp = firstcol(enr_fw, "Comparison", "Disease")
        if fwd:
            keep = [c for c in ["GeneSet", comp, tcol, fwd] if c and c in enr_fw.columns]
            enr_fw = enr_fw[keep].rename(columns={comp: dcol or "Disease",
                                                  fwd: "enrich_FDR"})
            merge_on = [c for c in ["GeneSet", dcol or "Disease"]
                        if c in meta.columns and c in enr_fw.columns]
            if merge_on:
                meta = meta.merge(enr_fw.drop_duplicates(merge_on),
                                  on=merge_on, how="left")

    # sort: permutation_p asc, then |z| desc
    if pcol and zcol:
        meta["_absz"] = meta[zcol].abs()
        leader = meta.sort_values([pcol, "_absz"], ascending=[True, False]).drop(columns="_absz")
    elif zcol:
        leader = meta.reindex(meta[zcol].abs().sort_values(ascending=False).index)
    else:
        leader = meta.copy()

    leader.to_csv(OUT / "01_enrichment_leaderboard.csv", index=False)

    # survivors: CI excludes 0 AND perm p < 0.05
    survivors = pd.DataFrame()
    if pcol:
        survivors = leader[leader["ci_excludes0"] & (pd.to_numeric(leader[pcol],
                    errors="coerce") < PERM_SIG)].copy()
    R["leader"] = {
        "n_rows": int(len(leader)),
        "zcol": zcol, "pcol": pcol, "dcol": dcol, "ncol": ncol,
        "n_ci_excludes0": int(leader["ci_excludes0"].sum()) if "ci_excludes0" in leader else 0,
        "survivors": survivors,
        "cols_present": {"low": lowcol, "high": hicol, "wilcox": wcol},
    }
    print(f"    leaderboard rows={len(leader)}  survivors(CI+perm)={len(survivors)}")
except Exception:
    warn("§1 failed:\n" + traceback.format_exc())

# =====================================================================
# §2 SHARED-PROFILE vs MEAN-SHIFT MAP
# =====================================================================
print("\n### §2 Concordance vs mean-shift ###")
try:
    p = pw.copy()
    if p.empty or "GeneSet" not in p.columns:
        warn("§2 skipped: pairwise table missing GeneSet")
    else:
        # merge proximity Pearson if absent
        if "Pearson_r" not in p.columns and not prox.empty and "Pearson_r" in prox.columns:
            pcols = [c for c in ["GeneSet", "Pearson_r", "Spearman_rho", "n_genes"]
                     if c in prox.columns]
            p = p.merge(prox[pcols].drop_duplicates("GeneSet"), on="GeneSet", how="left")

        xcol = firstcol(p, "Lin_CCC", "Pearson_r", "Spearman_rho")
        ycol = firstcol(p, "mean_diff", "paired_mean_diff", "Cohens_d_paired")
        scol = firstcol(p, "n_genes_paired", "n_genes_D1", "n_genes")

        if xcol and ycol:
            x = pd.to_numeric(p[xcol], errors="coerce")
            y = pd.to_numeric(p[ycol], errors="coerce")
            s = (pd.to_numeric(p[scol], errors="coerce").fillna(10)
                 if scol else pd.Series(10, index=p.index))
            fig, ax = plt.subplots(figsize=(8, 6))
            ax.axhline(0, ls="--", c="grey", lw=0.8)
            ax.axvline(CCC_SHARED, ls=":", c="grey", lw=0.8)
            ax.scatter(x, y, s=np.clip(s, 8, 80) * 2, alpha=0.75, edgecolor="k", lw=0.3)
            top = y.abs().sort_values(ascending=False).head(8).index
            for idx in top:
                ax.annotate(str(p.loc[idx, "GeneSet"])[:40], (x.loc[idx], y.loc[idx]),
                            fontsize=7)
            ax.set_xlabel(f"{xcol}  (TRS vs SCZ-EUR)")
            ax.set_ylabel(f"{ycol} (TRS − SCZ-EUR)")
            ax.set_title("Shared profile vs mean shift")
            fig.tight_layout(); fig.savefig(FIG / "02_concordance_vs_shift.png", dpi=200)
            plt.close(fig)
            p.assign(_x=x, _y=y).to_csv(OUT / "02_concordance_vs_shift.csv", index=False)

            shared = p.loc[(x >= CCC_SHARED) & (y.abs() < 0.5), "GeneSet"].tolist()
            shifts = p.reindex(top)[["GeneSet"]].copy()
            shifts["x"] = x.reindex(top).values; shifts["y"] = y.reindex(top).values
            R["shift"] = {"xcol": xcol, "ycol": ycol,
                          "n_shared": len(shared), "shared_examples": shared[:15],
                          "top_shifts": shifts}
            print(f"    shared-program sets={len(shared)}  labelled shifts={len(shifts)}")
        else:
            warn("§2 skipped: no x/y columns for scatter")
except Exception:
    warn("§2 failed:\n" + traceback.format_exc())

# =====================================================================
# §3 TISSUE CONSISTENCY (FDR-surviving focus sets)
# =====================================================================
print("\n### §3 Tissue consistency ###")
try:
    # focus sets = survivors from §1 if available, else user defaults
    focus = []
    if isinstance(R["leader"].get("survivors"), pd.DataFrame) and len(R["leader"]["survivors"]):
        focus = R["leader"]["survivors"]["GeneSet"].dropna().unique().tolist()
    if not focus:
        focus = ["REGULATION_OF_CELL_MOTILITY_GO_2000145",
                 "POSITIVE_REGULATION_OF_PROTEIN_SERINE_THREONINE_KINASE_ACTIVITY_GO_0071902"]
    focus = focus[:6]

    if not has(enrich, "Tissue"):
        warn("§3 skipped: no Tissue column")
    else:
        zcol = firstcol(enrich, "stouffer_z", "Stouffer_Z", "stouffer")
        dcol = firstcol(enrich, "Disease", "disease", "Comparison")
        t = enrich[enrich["GeneSet"].isin(focus)].copy()
        t = t[~t["Tissue"].fillna("").astype(str)
              .str.contains("meta|across", case=False, na=False)]
        if len(t) and zcol and dcol:
            pivot = t.pivot_table(index=["GeneSet", "Tissue"], columns=dcol,
                                  values=zcol, aggfunc="first")
            pivot.to_csv(OUT / "03_tissue_stouffer_focus_sets.csv")

            def sign_consistency(sub):
                z = pd.to_numeric(sub[zcol], errors="coerce").dropna()
                if z.empty:
                    return np.nan
                return float((np.sign(z) == np.sign(z.median())).mean())

            cons = (t.groupby(["GeneSet", dcol]).apply(sign_consistency)
                     .rename("frac_tissues_same_sign").reset_index())
            cons["n_tissues"] = (t.groupby(["GeneSet", dcol])["Tissue"]
                                 .nunique().reset_index(drop=True))
            cons.to_csv(OUT / "03_tissue_sign_consistency.csv", index=False)
            R["tissue"] = {"focus": focus, "consistency": cons}
            print(f"    focus sets={len(focus)}  consistency rows={len(cons)}")
        else:
            warn("§3 skipped: no tissue-level rows for focus sets")
except Exception:
    warn("§3 failed:\n" + traceback.format_exc())

# =====================================================================
# §4 INFLUENCE-ROBUST STOUFFER (jackknife influential genes)
# =====================================================================
print("\n### §4 Influence-robust Stouffer ###")
try:
    g = infl.copy()
    if g.empty:
        warn("§4 skipped: gene_influence empty")
    else:
        zcol = firstcol(g, "Z_score", "Z", "z")
        dcol = firstcol(g, "Disease", "disease", "Comparison")
        icol = firstcol(g, "Influential", "influential", "is_influential")
        if not (zcol and dcol and "GeneSet" in g.columns):
            warn("§4 skipped: missing Z/Disease/GeneSet columns")
        else:
            if icol:
                g["_infl"] = g[icol].astype(str).str.lower().isin(
                    ["true", "1", "yes", "t"])
            else:
                g["_infl"] = False
                warn("no Influential column; treating none as influential")

            def stouffer(z):
                z = np.asarray(z, float); z = z[np.isfinite(z)]
                return float(np.sum(z) / np.sqrt(len(z))) if len(z) >= 3 else np.nan

            rows = []
            for (gs, dis), sub in g.groupby(["GeneSet", dcol]):
                z_all = pd.to_numeric(sub[zcol], errors="coerce").to_numpy()
                z_rob = pd.to_numeric(sub.loc[~sub["_infl"], zcol], errors="coerce").to_numpy()
                s_all, s_rob = stouffer(z_all), stouffer(z_rob)
                rows.append({
                    "GeneSet": gs, "Disease": dis,
                    "n_all": int(np.isfinite(z_all).sum()),
                    "n_after_drop": int(np.isfinite(z_rob).sum()),
                    "n_influential": int(sub["_infl"].sum()),
                    "stouffer_all": s_all, "stouffer_robust": s_rob,
                    "sign_flip_after_drop": (np.isfinite(s_all) and np.isfinite(s_rob)
                                             and np.sign(s_all) != np.sign(s_rob)),
                })
            rob = pd.DataFrame(rows)
            if len(rob):
                rob["delta"] = rob["stouffer_all"] - rob["stouffer_robust"]
                rob["survives"] = (rob["stouffer_robust"].abs() >= Z_SIG) & (~rob["sign_flip_after_drop"])
                rob.to_csv(OUT / "04_influence_robust_stouffer.csv", index=False)
                fragile = rob[rob["sign_flip_after_drop"] |
                              rob["stouffer_robust"].abs().lt(Z_SIG)]
                R["robust"] = {"table": rob,
                               "n_fragile": int(len(fragile)),
                               "n_survive": int(rob["survives"].sum()),
                               "fragile": fragile.sort_values("n_influential",
                                                              ascending=False)}
                print(f"    sets tested={len(rob)}  survive drop={int(rob['survives'].sum())}  fragile={len(fragile)}")
except Exception:
    warn("§4 failed:\n" + traceback.format_exc())

# =====================================================================
# §5 CROSS-SET DRIVER CENSUS
# =====================================================================
print("\n### §5 Driver census ###")
try:
    g = infl.copy()
    zcol = firstcol(g, "Z_score", "Z", "z")
    dcol = firstcol(g, "Disease", "disease", "Comparison")
    gcol = firstcol(g, "Gene", "gene", "GeneName")
    icol = firstcol(g, "Influential", "influential")
    if g.empty or not (zcol and dcol and gcol and "GeneSet" in g.columns):
        warn("§5 skipped: gene_influence missing needed columns")
    else:
        g["_infl"] = (g[icol].astype(str).str.lower().isin(["true", "1", "yes", "t"])
                      if icol else False)
        drv_src = g[g["_infl"]] if g["_infl"].any() else g
        drv = (drv_src.groupby([gcol, dcol])
               .agg(n_sets=("GeneSet", "nunique"),
                    mean_abs_z=(zcol, lambda s: float(np.nanmean(np.abs(
                        pd.to_numeric(s, errors="coerce"))))),
                    sets=("GeneSet", lambda s: "; ".join(sorted(set(map(str, s)))[:8])))
               .reset_index()
               .sort_values([dcol, "n_sets"], ascending=[True, False]))
        drv.to_csv(OUT / "05_driver_census.csv", index=False)

        # incidence matrix
        try:
            inc = (drv_src.assign(hit=1)
                   .pivot_table(index=gcol, columns="GeneSet", values="hit",
                                aggfunc="max", fill_value=0))
            inc.to_csv(OUT / "05_driver_incidence.csv")
        except Exception:
            pass

        top_drivers = drv.sort_values("n_sets", ascending=False).head(20)
        R["drivers"] = {"table": drv, "top": top_drivers,
                        "used_influential": bool(g["_infl"].any())}
        print(f"    driver rows={len(drv)}  top recycled genes noted")
except Exception:
    warn("§5 failed:\n" + traceback.format_exc())

# =====================================================================
# §6 SIGNED TWO-COHORT STOUFFER SCATTER
# =====================================================================
print("\n### §6 Two-cohort Stouffer scatter ###")
try:
    src = leader if len(leader) else enrich
    zcol = firstcol(src, "stouffer_z", "Stouffer_Z", "stouffer")
    dcol = firstcol(src, "Disease", "disease", "Comparison")
    ncol = firstcol(src, "n_genes_found", "n_found", "n_genes")
    if not (zcol and dcol and "GeneSet" in src.columns):
        warn("§6 skipped: need Stouffer/Disease/GeneSet")
    else:
        wide = src.pivot_table(index="GeneSet", columns=dcol, values=zcol, aggfunc="first")
        if ncol:
            nwide = src.pivot_table(index="GeneSet", columns=dcol, values=ncol, aggfunc="first")
            wide = wide.join(nwide.add_prefix("n_"))
        wide.to_csv(OUT / "06_stouffer_trs_vs_scz.csv")

        # identify the two disease columns
        dcols = [c for c in wide.columns if not str(c).startswith("n_")]
        trs = next((c for c in dcols if re.search("trs", str(c), re.I)), None)
        scz = next((c for c in dcols if re.search("scz|eur", str(c), re.I)), None)
        if trs and scz:
            fig, ax = plt.subplots(figsize=(6.5, 6.5))
            ax.axhline(0, c="grey", lw=0.6); ax.axvline(0, c="grey", lw=0.6)
            lim = np.nanmax(np.abs(pd.concat([wide[trs], wide[scz]]))) if len(wide) else 6
            lim = float(lim) if np.isfinite(lim) else 6
            ax.plot([-lim, lim], [-lim, lim], ls="--", c="grey", lw=0.7)
            ax.scatter(wide[scz], wide[trs], alpha=0.8, edgecolor="k", lw=0.3)
            for gs, r in wide.iterrows():
                if abs(pd.to_numeric(r[scz], errors="coerce")) > 4 or \
                   abs(pd.to_numeric(r[trs], errors="coerce")) > 4:
                    ax.annotate(str(gs)[:28], (r[scz], r[trs]), fontsize=7)
            ax.set_xlabel(f"Stouffer Z  {scz}")
            ax.set_ylabel(f"Stouffer Z  {trs}")
            ax.set_title("Set-level TWAS shift, two cohorts")
            fig.tight_layout(); fig.savefig(FIG / "06_stouffer_trs_vs_scz.png", dpi=200)
            plt.close(fig)

            q = wide.dropna(subset=[trs, scz]).copy()
            down_both = q[(q[trs] < 0) & (q[scz] < 0)].index.tolist()
            up_both   = q[(q[trs] > 0) & (q[scz] > 0)].index.tolist()
            offdiag   = q[(np.sign(q[trs]) != np.sign(q[scz]))].index.tolist()
            R["twocohort"] = {"trs_col": trs, "scz_col": scz,
                              "down_both": down_both[:20], "up_both": up_both[:20],
                              "offdiag": offdiag[:20],
                              "n_down": len(down_both), "n_up": len(up_both),
                              "n_off": len(offdiag)}
            print(f"    down_both={len(down_both)}  up_both={len(up_both)}  off-diagonal={len(offdiag)}")
        else:
            warn(f"§6: could not resolve TRS/SCZ columns from {list(dcols)}")
except Exception:
    warn("§6 failed:\n" + traceback.format_exc())

# =====================================================================
# §7 FAMILY-WISE FDR SLICED BY TEST FAMILY
# =====================================================================
print("\n### §7 FDR by test family ###")
try:
    if fw.empty:
        warn("§7 skipped: no family_wise_fdr files")
    else:
        scol = firstcol(fw, "Source", "source")
        tcol = firstcol(fw, "Test", "test")
        fcol = firstcol(fw, "family_wise_FDR", "FDR", "fdr")
        if fcol is None:
            warn("§7 skipped: no FDR column")
        else:
            fw["_fdr"] = pd.to_numeric(fw[fcol], errors="coerce")
            fw["family"] = ((fw[scol].astype(str) if scol else "NA") + "/" +
                            (fw[tcol].astype(str) if tcol else "NA"))
            tally = (fw[fw["_fdr"] < FDR_SIG].groupby("family").size()
                     .sort_values(ascending=False).rename("n_hits").reset_index())
            tally.to_csv(OUT / "07_fdr_by_test_family.csv", index=False)

            enr_hits = pd.DataFrame()
            if tcol:
                enr_hits = fw[fw[tcol].astype(str)
                              .str.contains("permutation_p|wilcoxon_p", case=False, na=False)
                              & fw["_fdr"].lt(FDR_SIG)]
                enr_hits.to_csv(OUT / "07_fdr_enrichment_only.csv", index=False)
            R["fdr"] = {"tally": tally,
                        "n_total_hits": int((fw["_fdr"] < FDR_SIG).sum()),
                        "n_enrichment_hits": int(len(enr_hits))}
            print(f"    total FDR<{FDR_SIG} hits={int((fw['_fdr']<FDR_SIG).sum())}  "
                  f"enrichment-only={len(enr_hits)}")
except Exception:
    warn("§7 failed:\n" + traceback.format_exc())

# =====================================================================
# §8 PUBLICATION-SCALE COMPACT TABLE
# =====================================================================
print("\n### §8 Publication table ###")
try:
    src = leader if len(leader) else enrich
    zcol = firstcol(src, "stouffer_z", "Stouffer_Z", "stouffer")
    dcol = firstcol(src, "Disease", "disease", "Comparison")
    pcol = firstcol(src, "permutation_p", "perm_p")
    ncol = firstcol(src, "n_genes_found", "n_found", "n_genes")
    if not (zcol and dcol and "GeneSet" in src.columns):
        warn("§8 skipped: need Stouffer/Disease/GeneSet")
    else:
        vals = [v for v in [zcol, pcol, ncol] if v]
        base = src.pivot_table(index="GeneSet", columns=dcol, values=vals, aggfunc="first")
        base.columns = [f"{a}_{b}" for a, b in base.columns]
        base = base.reset_index()

        extra = pd.DataFrame({"GeneSet": base["GeneSet"]})
        if not pw.empty and "GeneSet" in pw.columns:
            ecols = [c for c in ["GeneSet", "n_genes_paired", "mean_diff",
                                 "Cohens_d_paired", "Lin_CCC",
                                 "sign_concordance_rate", "Kendall_tau"]
                     if c in pw.columns]
            extra = pw[ecols].drop_duplicates("GeneSet")
        if not prox.empty and "GeneSet" in prox.columns:
            pcols = [c for c in ["GeneSet", "Pearson_r", "Spearman_rho"]
                     if c in prox.columns]
            if len(pcols) > 1:
                extra = extra.merge(prox[pcols].drop_duplicates("GeneSet"),
                                    on="GeneSet", how="left")
        pub = base.merge(extra, on="GeneSet", how="left")
        pub.to_csv(OUT / "08_publication_table.csv", index=False)
        R["pub"] = {"n_rows": int(len(pub)), "columns": list(pub.columns)}
        print(f"    publication table rows={len(pub)}  cols={len(pub.columns)}")
except Exception:
    warn("§8 failed:\n" + traceback.format_exc())

# =====================================================================
# VERY DETAILED SUMMARY.txt
# =====================================================================
print("\n### Writing SUMMARY.txt ###")
S = []
def w(*a): S.append(" ".join(str(x) for x in a))
def wrap(items, width=90, indent="        "):
    txt = ", ".join(str(i) for i in items)
    return "\n".join(textwrap.wrap(txt, width=width,
                     initial_indent=indent, subsequent_indent=indent)) or (indent + "(none)")
def fmt(x, nd=3):
    try:
        xf = float(x)
        return f"{xf:.{nd}g}" if abs(xf) < 1e-3 or abs(xf) >= 1e4 else f"{xf:.{nd}f}"
    except Exception:
        return str(x)
bar = "=" * 80

w(bar)
w("091IvD DOWNSTREAM — CONSOLIDATED DETAILED SUMMARY")
w("CSV-only re-analysis of persistent-enhancer TWAS results (TRS vs SCZ-EUR)")
w(bar)
w(f"Generated        : {TS}")
w(f"Results dir      : {RESULTS}")
w(f"Output dir       : {OUT}")
w("")
w("Input tables loaded (rows x cols):")
for k, (r, c) in R["loads"].items():
    w(f"  {k:20s} {r:6d} x {c}")
w("")

# --- Section 1 ---
w(bar); w("1. CONSERVATIVE ENRICHMENT LEADERBOARD (CI excludes 0 + permutation FDR)"); w(bar)
L = R["leader"]
if L:
    w(f"  Leaderboard rows                : {L.get('n_rows', 0)}")
    w(f"  Rows with CI excluding 0        : {L.get('n_ci_excludes0', 0)}")
    sv = L.get("survivors")
    if isinstance(sv, pd.DataFrame) and len(sv):
        w(f"  SURVIVORS (CI≠0 AND perm p<{PERM_SIG}) : {len(sv)}")
        w("")
        zc, pc, dc, nc = L["zcol"], L["pcol"], L["dcol"], L["ncol"]
        lo = L["cols_present"]["low"]; hi = L["cols_present"]["high"]; wl = L["cols_present"]["wilcox"]
        for _, r in sv.head(20).iterrows():
            line = f"  • {str(r['GeneSet'])[:55]}"
            if dc and dc in r: line += f"  [{r[dc]}]"
            w(line)
            det = "        "
            if nc and nc in r: det += f"n={r[nc]}  "
            if zc: det += f"Stouffer_Z={fmt(r[zc])}  "
            if lo and hi: det += f"CI=[{fmt(r[lo])}, {fmt(r[hi])}]  "
            if pc: det += f"perm_p={fmt(r[pc])}  "
            if wl and wl in r: det += f"wilcox_p={fmt(r[wl])}"
            w(det)
    else:
        w("  SURVIVORS (CI≠0 AND perm p<0.05) : 0  (none passed the conservative gate)")
else:
    w("  (§1 not run)")
w("")

# --- Section 2 ---
w(bar); w("2. SHARED-PROFILE vs MEAN-SHIFT MAP"); w(bar)
SH = R["shift"]
if SH:
    w(f"  X axis (concordance)   : {SH['xcol']}")
    w(f"  Y axis (mean shift)    : {SH['ycol']}")
    w(f"  Shared-program sets (x>= {CCC_SHARED}, |y|<0.5) : {SH['n_shared']}")
    w("  Examples of shared-program sets:")
    w(wrap(SH["shared_examples"]))
    w("")
    w("  Largest TRS−SCZ mean shifts (labelled on figure):")
    ts = SH.get("top_shifts")
    if isinstance(ts, pd.DataFrame):
        for _, r in ts.iterrows():
            w(f"    {str(r['GeneSet'])[:55]:55s} x={fmt(r['x'])}  y(shift)={fmt(r['y'])}")
    w("  Figure: figures/02_concordance_vs_shift.png")
else:
    w("  (§2 not run — pairwise columns unavailable)")
w("")

# --- Section 3 ---
w(bar); w("3. TISSUE CONSISTENCY OF FOCUS (FDR-SURVIVING) SETS"); w(bar)
T = R["tissue"]
if T:
    w("  Focus sets:")
    w(wrap(T["focus"]))
    w("")
    cons = T.get("consistency")
    if isinstance(cons, pd.DataFrame) and len(cons):
        w("  Fraction of tissues sharing the sign of the median Stouffer:")
        for _, r in cons.iterrows():
            nt = r.get("n_tissues", "?")
            w(f"    {str(r['GeneSet'])[:45]:45s} {str(r.iloc[1]):10s} "
              f"frac_same_sign={fmt(r['frac_tissues_same_sign'])}  (n_tissues={nt})")
    w("  Tables: 03_tissue_stouffer_focus_sets.csv, 03_tissue_sign_consistency.csv")
else:
    w("  (§3 not run — no Tissue column / no matching rows)")
w("")

# --- Section 4 ---
w(bar); w("4. INFLUENCE-ROBUST STOUFFER (jackknife of Influential genes)"); w(bar)
RB = R["robust"]
if RB:
    tab = RB["table"]
    w(f"  Set×disease pairs tested        : {len(tab)}")
    w(f"  Survive drop (|Z|>={Z_SIG}, no flip) : {RB['n_survive']}")
    w(f"  Fragile (flip or |Z|<{Z_SIG})       : {RB['n_fragile']}")
    w("")
    w("  ROBUST SURVIVORS (headline holds after dropping influential genes):")
    surv = tab[tab["survives"]].sort_values("stouffer_robust", key=lambda s: s.abs(),
                                            ascending=False)
    if len(surv):
        for _, r in surv.head(20).iterrows():
            w(f"    {str(r['GeneSet'])[:45]:45s} [{r['Disease']}] "
              f"Z_all={fmt(r['stouffer_all'])} -> Z_robust={fmt(r['stouffer_robust'])} "
              f"(dropped {int(r['n_influential'])})")
    else:
        w("    (none)")
    w("")
    w("  MOST FRAGILE (claim dies once influential genes removed):")
    fr = RB.get("fragile")
    if isinstance(fr, pd.DataFrame) and len(fr):
        for _, r in fr.head(15).iterrows():
            flip = "SIGN FLIP" if r["sign_flip_after_drop"] else "drops<1.96"
            w(f"    {str(r['GeneSet'])[:45]:45s} [{r['Disease']}] "
              f"Z_all={fmt(r['stouffer_all'])} -> Z_robust={fmt(r['stouffer_robust'])} "
              f"n_infl={int(r['n_influential'])} [{flip}]")
    w("  Table: 04_influence_robust_stouffer.csv")
else:
    w("  (§4 not run — gene_influence unavailable)")
w("")

# --- Section 5 ---
w(bar); w("5. CROSS-SET DRIVER CENSUS (genes recycled across GO labels)"); w(bar)
DR = R["drivers"]
if DR:
    if not DR["used_influential"]:
        w("  NOTE: no Influential flag present — census uses all genes.")
    top = DR.get("top")
    if isinstance(top, pd.DataFrame) and len(top):
        w("  Top drivers by number of gene sets they appear in:")
        gcol = [c for c in top.columns if c.lower() in ("gene", "genename")][0]
        dcol = [c for c in top.columns if c.lower() in ("disease", "comparison")]
        dcol = dcol[0] if dcol else None
        for _, r in top.iterrows():
            dis = f" [{r[dcol]}]" if dcol else ""
            w(f"    {str(r[gcol])[:18]:18s}{dis:12s} n_sets={int(r['n_sets']):3d}  "
              f"mean|Z|={fmt(r['mean_abs_z'])}")
            w(f"        sets: {str(r['sets'])[:120]}")
    w("  Tables: 05_driver_census.csv, 05_driver_incidence.csv")
else:
    w("  (§5 not run)")
w("")

# --- Section 6 ---
w(bar); w("6. SIGNED TWO-COHORT STOUFFER SCATTER (set-level)"); w(bar)
TC = R["twocohort"]
if TC:
    w(f"  Axes: x={TC['scz_col']}  y={TC['trs_col']}")
    w(f"  Down in BOTH cohorts : {TC['n_down']}")
    w(wrap(TC["down_both"]))
    w(f"  Up in BOTH cohorts   : {TC['n_up']}")
    w(wrap(TC["up_both"]))
    w(f"  Off-diagonal (differential TRS vs SCZ) : {TC['n_off']}")
    w(wrap(TC["offdiag"]))
    w("  Figure: figures/06_stouffer_trs_vs_scz.png  |  Table: 06_stouffer_trs_vs_scz.csv")
else:
    w("  (§6 not run — could not resolve TRS/SCZ columns)")
w("")

# --- Section 7 ---
w(bar); w("7. FAMILY-WISE FDR SLICED BY TEST FAMILY"); w(bar)
FD = R["fdr"]
if FD:
    w(f"  Total family-wise FDR<{FDR_SIG} hits : {FD['n_total_hits']}")
    w(f"  Of which enrichment (perm/wilcox): {FD['n_enrichment_hits']}")
    w("")
    tally = FD.get("tally")
    if isinstance(tally, pd.DataFrame) and len(tally):
        w("  Hits by test family (Source/Test):")
        for _, r in tally.iterrows():
            w(f"    {str(r['family'])[:55]:55s} {int(r['n_hits']):5d}")
    w("  Tables: 07_fdr_by_test_family.csv, 07_fdr_enrichment_only.csv")
    w("  CAUTION: most FDR hits are proximity/concordance, not enrichment —")
    w("           do not report the total as N independent enrichment findings.")
else:
    w("  (§7 not run — no family_wise_fdr files)")
w("")

# --- Section 8 ---
w(bar); w("8. PUBLICATION-SCALE COMPACT TABLE"); w(bar)
PB = R["pub"]
if PB:
    w(f"  Rows (gene sets) : {PB['n_rows']}")
    w(f"  Columns          : {len(PB['columns'])}")
    w(wrap(PB["columns"]))
    w("  Table: 08_publication_table.csv")
else:
    w("  (§8 not run)")
w("")

# --- Interpretation gate ---
w(bar); w("9. RECOMMENDED REPORTING GATE"); w(bar)
w("  Report a GO-level claim only if the set:")
w("    (a) survives the conservative leaderboard (§1: CI excludes 0 AND perm p<0.05),")
w("    (b) stays signed with |Stouffer|>=1.96 after dropping Influential genes (§4),")
w("    (c) is not carried by a single recycled driver in §5.")
w("  Otherwise treat it as a two-gene / driver-level observation, not a pathway result.")
w("")

# --- Warnings ---
if R["warnings"]:
    w(bar); w("10. RUN WARNINGS / SKIPPED STEPS"); w(bar)
    for m in R["warnings"]:
        for ln in str(m).splitlines():
            w("  " + ln)
    w("")

# --- File inventory ---
w(bar); w("11. OUTPUT FILES"); w(bar)
for f in sorted(glob.glob(str(OUT / "**" / "*"), recursive=True)):
    if os.path.isfile(f):
        w(f"  {f}  ({os.path.getsize(f)//1024} KB)")
w(""); w(bar); w("END OF SUMMARY"); w(bar)

with open(OUT / "SUMMARY.txt", "w") as fh:
    fh.write("\n".join(S) + "\n")

print("\n=== DONE ===")
print("Detailed summary:", (OUT / "SUMMARY.txt").resolve())
print("\n----- SUMMARY.txt preview (first 70 lines) -----")
print("\n".join(S[:70]))

>>> Loaded tables: {'enrichment': (462, 22), 'pairwise': (33, 62), 'gene_influence': (1909, 14), 'family_wise_fdr': (616, 7), 'proximity': (33, 12)}

### §1 Enrichment leaderboard ###
    leaderboard rows=66  survivors(CI+perm)=5

### §2 Concordance vs mean-shift ###
    shared-program sets=26  labelled shifts=8

### §3 Tissue consistency ###
    focus sets=3  consistency rows=6

### §4 Influence-robust Stouffer ###
    sets tested=66  survive drop=14  fragile=39

### §5 Driver census ###
    driver rows=586  top recycled genes noted

### §6 Two-cohort Stouffer scatter ###
    down_both=8  up_both=9  off-diagonal=16

### §7 FDR by test family ###
    total FDR<0.05 hits=107  enrichment-only=9

### §8 Publication table ###
    publication table rows=33  cols=15

### Writing SUMMARY.txt ###

=== DONE ===
Detailed summary: /content/091IvD_results/downstream/SUMMARY.txt

----- SUMMARY.txt preview (first 70 lines) -----
091IvD DOWNSTREAM — CONSOLIDATED DETAILED SUMMARY
CSV-only re-analysis 

# Stage 3

In [11]:
# =====================================================================
# 091IvD  STAGE 3  —  GENE-LEVEL CONVERGENCE / DIVERGENCE (CSV-ONLY)
# The unit of discovery is the GENE, not the GO label.
# Consumes ONLY gene_detail_*.csv + gene_influence_*.csv from the pipeline.
# One cell. Ends with stage3/SUMMARY.txt (very detailed).
# =====================================================================

from pathlib import Path
import os, re, glob, textwrap, datetime, warnings, traceback
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

# ---------------------------- CONFIG ---------------------------------
RESULTS = Path("/content/091IvD_results")     # <-- change if needed
OUT     = RESULTS / "stage3"
FIG     = OUT / "figures"
OUT.mkdir(parents=True, exist_ok=True)
FIG.mkdir(parents=True, exist_ok=True)

Z_SIG        = 1.96      # |Z| significance gate
CONVERGE_Z   = 1.96      # both cohorts must exceed this (same sign) for a core call
DIVERGE_MIN  = 1.0       # min |Z| in weaker cohort to call a sign divergence
DELTA_SHIFT  = 2.0       # |Z_trs - Z_scz| to call a shift divergence
CCC_SHARED   = 0.5
TS           = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")

R = {"loads": {}, "cohorts": {}, "matrix": {}, "classes": {}, "core_down": {},
     "core_up": {}, "diverge": {}, "hubs": {}, "modstats": {}, "anchor": {},
     "roster": {}, "warnings": []}

def warn(msg):
    print("  [warn]", msg); R["warnings"].append(str(msg))

def firstcol(df, *names):
    if df is None or df.empty:
        return None
    lower = {c.lower(): c for c in df.columns}
    for n in names:
        if n.lower() in lower:
            return lower[n.lower()]
    return None

# ============================ 1. LOAD =================================
def stack(globpat, strip_prefix):
    frames = []
    for p in sorted(RESULTS.rglob(globpat)):
        if p.name.startswith("multi_"):
            continue
        try:
            df = pd.read_csv(p)
        except Exception as ex:
            warn(f"could not read {p.name}: {ex}"); continue
        if "GeneSet" not in df.columns and "label" not in df.columns:
            df.insert(0, "GeneSet", re.sub(strip_prefix, "", p.stem))
        frames.append(df)
    if frames:
        out = pd.concat(frames, ignore_index=True, sort=False)
        if "GeneSet" not in out.columns and "label" in out.columns:
            out.rename(columns={"label": "GeneSet"}, inplace=True)
        return out
    return pd.DataFrame()

detail = stack("gene_detail_*.csv", r"^gene_detail_")
if detail.empty:
    hits = list(RESULTS.rglob("all_genes_all_genesets.csv"))
    if hits:
        detail = pd.read_csv(hits[0]); warn("used all_genes_all_genesets.csv as detail fallback")

infl = stack("gene_influence_*.csv", r"^gene_influence_")
if infl.empty:
    hits = list(RESULTS.rglob("multi_geneset_gene_influence.csv"))
    if hits:
        infl = pd.read_csv(hits[0]); warn("used multi_geneset_gene_influence.csv as influence fallback")

R["loads"] = {"gene_detail": tuple(detail.shape), "gene_influence": tuple(infl.shape)}
print(">>> Loaded:", R["loads"])
if detail.empty:
    raise SystemExit("FATAL: no gene_detail_*.csv found — cannot run Stage 3.")

# ------------------- detect gene + cohort Z columns ------------------
gcol = firstcol(detail, "Gene", "gene", "GeneName")
z_all = [c for c in detail.columns if re.match(r"^Z_", str(c)) and "__" not in str(c)
         and c not in ("Z_range", "Z_mean")]
z_trs = next((c for c in z_all if re.search(r"trs", c, re.I)), None)
z_scz = next((c for c in z_all if re.search(r"scz|eur", c, re.I)), None)
# fall back to first two Z_ columns if naming differs
if (z_trs is None or z_scz is None) and len(z_all) >= 2:
    z_trs = z_trs or z_all[0]
    z_scz = z_scz or next(c for c in z_all if c != z_trs)
if not (gcol and z_trs and z_scz):
    raise SystemExit(f"FATAL: could not resolve Gene/TRS/SCZ columns. Found Z cols={z_all}")

TRS_LAB = re.sub(r"^Z_", "", z_trs)
SCZ_LAB = re.sub(r"^Z_", "", z_scz)
R["cohorts"] = {"gene_col": gcol, "trs_col": z_trs, "scz_col": z_scz,
                "trs_label": TRS_LAB, "scz_label": SCZ_LAB, "all_Z_cols": z_all}
print(f">>> Cohorts: TRS='{TRS_LAB}' ({z_trs})   SCZ='{SCZ_LAB}' ({z_scz})")

# ======================= 2. MODULE MAPPING ===========================
MODULE = {
    "migration_cytoskeleton": [
        "REGULATION_OF_CELL_MOTILITY_GO_2000145",
        "REGULATION_OF_CELL_MIGRATION_GO_0030334",
        "REGULATION_OF_FILOPODIUM_ASSEMBLY_GO_0051489",
        "POSITIVE_REGULATION_OF_FILOPODIUM_ASSEMBLY_GO_0051491",
        "EPITHELIAL_CELL_MIGRATION_GO_0010631",
    ],
    "neurite_development": [
        "AXONOGENESIS_GO_0007409",
        "NEURON_PROJECTION_MORPHOGENESIS_GO_0048812",
        "POSITIVE_REGULATION_OF_NEURON_PROJECTION_DEVELOPMENT_GO_0010976",
        "POSITIVE_REGULATION_OF_CELL_PROJECTION_ORGANIZATION_GO_0031346",
        "CELL_MORPHOGENESIS_INVOLVED_IN_NEURON_DIFFERENTIATION_GO_0048667",
        "NERVOUS_SYSTEM_DEVELOPMENT_GO_0007399",
        "POSITIVE_REGULATION_OF_NERVOUS_SYSTEM_DEVELOPMENT_GO_0051962",
        "LIMB_DEVELOPMENT_GO_0060173",
    ],
    "adhesion_synapse": [
        "CELL_CELL_ADHESION_VIA_PLASMA_MEMBRANE_ADHESION_MOLECULES_GO_0098742",
        "HOMOPHILIC_CELL_ADHESION_VIA_PLASMA_MEMBRANE_ADHESION_MOLECULES_GO_0007156",
        "CELL_JUNCTION_ASSEMBLY_GO_0034329",
        "SYNAPTIC_MEMBRANE_ADHESION_GO_0099560",
        "SYNAPSE_ASSEMBLY_GO_0007416",
        "SYNAPSE_ORGANIZATION_GO_0050808",
    ],
    "signaling_plasticity": [
        "POSITIVE_REGULATION_OF_PROTEIN_SERINE_THREONINE_KINASE_ACTIVITY_GO_0071902",
        "TRANSMEMBRANE_RECEPTOR_PROTEIN_TYROSINE_KINASE_SIGNALING_PATHWAY_GO_0007169",
        "ADENYLATE_CYCLASE_MODULATING_G_PROTEIN_COUPLED_RECEPTOR_SIGNALING_PATHWAY_GO_0007188",
        "CELLULAR_RESPONSE_TO_ORGANIC_CYCLIC_COMPOUND_GO_0071407",
        "POSITIVE_REGULATION_OF_CELL_DIFFERENTIATION_GO_0045597",
    ],
    "excitability": [
        "POTASSIUM_ION_TRANSPORT_GO_0006813",
        "REGULATION_OF_CARDIAC_MUSCLE_CELL_ACTION_POTENTIAL_GO_0098901",
        "SYNAPTIC_TRANSMISSION_GABAERGIC_GO_0051932",
    ],
    "metabolism": [
        "PTERIDINE_CONTAINING_COMPOUND_METABOLIC_PROCESS_GO_0042558",
    ],
}
SET2MOD = {s: m for m, sets in MODULE.items() for s in sets}
ANCHOR_SETS = {
    "REGULATION_OF_CELL_MOTILITY_GO_2000145",
    "POSITIVE_REGULATION_OF_PROTEIN_SERINE_THREONINE_KINASE_ACTIVITY_GO_0071902",
}
LAYER = {
    "L2_cytoskeleton_guidance": "migration_cytoskeleton",
    "L2L3_neurite":             "neurite_development",
    "L3_adhesion_synapse":      "adhesion_synapse",
    "L4_signaling":             "signaling_plasticity",
    "L4_excitability":          "excitability",
    "L_meta_pteridine":         "metabolism",
}

d = detail.copy()
d["module"] = d["GeneSet"].map(SET2MOD).fillna("other")
d["_trs"] = pd.to_numeric(d[z_trs], errors="coerce")
d["_scz"] = pd.to_numeric(d[z_scz], errors="coerce")
n_mapped = int((d["module"] != "other").sum())
warn(f"{n_mapped}/{len(d)} gene_detail rows mapped to a named module "
     f"({d['GeneSet'].map(SET2MOD).notna().sum()} rows via SET2MOD)")

# ============ 3.1 GENE × COHORT MATRIX (one row per gene) ============
print("\n### 3.1 Gene × cohort matrix ###")
try:
    def _maxabs(s):
        s = pd.to_numeric(s, errors="coerce")
        return float(np.nanmax(np.abs(s))) if s.notna().any() else np.nan

    gene = (d.groupby(gcol)
              .agg(n_sets=("GeneSet", "nunique"),
                   modules=("module", lambda s: ";".join(sorted(set(s) - {"other"})
                                                          or ["other"])),
                   n_modules=("module", lambda s: len(set(s) - {"other"}) or 0),
                   Z_trs=("_trs", "median"),
                   Z_scz=("_scz", "median"),
                   max_abs_trs=("_trs", _maxabs),
                   max_abs_scz=("_scz", _maxabs))
              .reset_index())

    gene["delta"]        = gene["Z_trs"] - gene["Z_scz"]      # TRS - SCZ
    gene["mean_Z"]       = gene[["Z_trs", "Z_scz"]].mean(axis=1)
    gene["min_abs"]      = gene[["Z_trs", "Z_scz"]].abs().min(axis=1)
    gene["same_sign"]    = np.sign(gene["Z_trs"]) == np.sign(gene["Z_scz"])
    gene["both_nonzero"] = gene["Z_trs"].ne(0) & gene["Z_scz"].ne(0)

    gene["class"] = np.select(
        [
            gene["same_sign"] & gene["Z_scz"].lt(-CONVERGE_Z) & gene["Z_trs"].lt(-CONVERGE_Z),
            gene["same_sign"] & gene["Z_scz"].gt( CONVERGE_Z) & gene["Z_trs"].gt( CONVERGE_Z),
            (~gene["same_sign"]) & gene["min_abs"].gt(DIVERGE_MIN),
            gene["delta"].abs().gt(DELTA_SHIFT),
        ],
        ["converge_down", "converge_up", "diverge_sign", "diverge_shift"],
        default="weak",
    )
    gene["layers"] = gene["modules"].map(
        lambda ms: ";".join(k for k, v in LAYER.items()
                            if v in set(str(ms).split(";"))) or "unassigned")

    gene.to_csv(OUT / "31_gene_cohort_matrix.csv", index=False)
    vc = gene["class"].value_counts().to_dict()
    R["matrix"] = {"n_genes": int(len(gene))}
    R["classes"] = vc
    print("    class counts:", vc)
except Exception:
    warn("3.1 failed:\n" + traceback.format_exc())
    gene = pd.DataFrame()

# =================== 3.2 CONVERGENCE / DIVERGENCE PLOTS ==============
print("\n### 3.2 Scatter plots ###")
try:
    if not gene.empty:
        colors = {"converge_down": "#1f4e79", "converge_up": "#8b1e1e",
                  "diverge_sign": "#c47b00", "diverge_shift": "#6a3d9a",
                  "weak": "#bbbbbb"}
        fig, ax = plt.subplots(figsize=(7, 7))
        ax.axhline(0, c="grey", lw=0.6); ax.axvline(0, c="grey", lw=0.6)
        lim = float(np.nanmax(np.abs(gene[["Z_trs", "Z_scz"]].values))) if len(gene) else 6
        lim = lim if np.isfinite(lim) else 6
        ax.plot([-lim, lim], [-lim, lim], ls="--", c="grey", lw=0.7)
        for cls, sub in gene.groupby("class"):
            ax.scatter(sub["Z_scz"], sub["Z_trs"],
                       s=16 if cls == "weak" else 34,
                       c=colors.get(cls, "#888"),
                       alpha=0.35 if cls == "weak" else 0.85,
                       label=f"{cls} n={len(sub)}", edgecolor="k", lw=0.2)
        lab = gene[gene["class"].ne("weak") &
                   ((gene[["Z_trs", "Z_scz"]].abs().max(axis=1) > 3) |
                    (gene["delta"].abs() > 2.5))]
        for _, r in lab.iterrows():
            ax.annotate(str(r[gcol]), (r["Z_scz"], r["Z_trs"]), fontsize=7)
        ax.set_xlabel(f"TWAS Z  {SCZ_LAB}")
        ax.set_ylabel(f"TWAS Z  {TRS_LAB}")
        ax.set_title("Gene-level convergence / divergence")
        ax.legend(fontsize=8, loc="best")
        fig.tight_layout(); fig.savefig(FIG / "32_gene_scatter.png", dpi=200); plt.close(fig)

        # anchor sets only
        anchor = d[d["GeneSet"].isin(ANCHOR_SETS)].copy()
        if not anchor.empty:
            groups = list(anchor.groupby("GeneSet"))
            fig, axes = plt.subplots(1, max(len(groups), 1),
                                     figsize=(5.5 * max(len(groups), 1), 5),
                                     sharex=True, sharey=True, squeeze=False)
            axes = axes[0]
            for ax, (gs, sub) in zip(axes, groups):
                ax.axhline(0, c="grey", lw=0.6); ax.axvline(0, c="grey", lw=0.6)
                ax.scatter(sub["_scz"], sub["_trs"], c="#1f4e79", edgecolor="k", lw=0.2)
                for _, r in sub.iterrows():
                    if abs(r["_scz"]) > 2 or abs(r["_trs"]) > 2:
                        ax.annotate(str(r[gcol]), (r["_scz"], r["_trs"]), fontsize=7)
                ax.set_title(str(gs)[:40]); ax.set_xlabel(f"Z {SCZ_LAB}")
                ax.set_ylabel(f"Z {TRS_LAB}")
            fig.suptitle("Anchor sets — gene Z in both cohorts")
            fig.tight_layout(); fig.savefig(FIG / "32_anchor_gene_scatter.png", dpi=200)
            plt.close(fig)
        print("    figures written")
except Exception:
    warn("3.2 failed:\n" + traceback.format_exc())

# =========== 3.3 CONVERGENT CORE & DIVERGENT TAIL ====================
print("\n### 3.3 Ranked lists ###")
core_down = core_up = div = pd.DataFrame()
try:
    if not gene.empty:
        core_down = (gene[gene["class"].eq("converge_down")]
                     .assign(score=lambda x: -x["mean_Z"])
                     .sort_values("score", ascending=False))
        core_up   = (gene[gene["class"].eq("converge_up")]
                     .assign(score=lambda x: x["mean_Z"])
                     .sort_values("score", ascending=False))
        div       = (gene[gene["class"].isin(["diverge_sign", "diverge_shift"])]
                     .assign(abs_delta=lambda x: x["delta"].abs())
                     .sort_values("abs_delta", ascending=False))
        core_down.to_csv(OUT / "33_converge_down.csv", index=False)
        core_up.to_csv(OUT / "33_converge_up.csv", index=False)
        div.to_csv(OUT / "33_diverge.csv", index=False)
        R["core_down"] = {"n": int(len(core_down))}
        R["core_up"]   = {"n": int(len(core_up))}
        R["diverge"]   = {"n": int(len(div))}
        print(f"    converge_down={len(core_down)}  converge_up={len(core_up)}  diverge={len(div)}")
except Exception:
    warn("3.3 failed:\n" + traceback.format_exc())

# ================= 3.4 LAYER HUBS (multi-module genes) ===============
print("\n### 3.4 Layer hubs ###")
hubs = pd.DataFrame()
try:
    if not gene.empty:
        hubs = gene[gene["n_modules"] >= 3].sort_values(["n_modules", "n_sets"],
                                                        ascending=False)
        hubs.to_csv(OUT / "34_layer_hubs.csv", index=False)

        inc = (d[d["module"] != "other"]
               .dropna(subset=["_scz", "_trs"], how="all")
               .assign(hit=1)
               .pivot_table(index=gcol, columns="module", values="hit",
                            aggfunc="max", fill_value=0))
        if not inc.empty:
            inc["n_modules"] = inc.sum(axis=1)
            inc = inc.join(gene.set_index(gcol)[["Z_scz", "Z_trs", "class"]], how="left")
            inc.sort_values("n_modules", ascending=False).to_csv(OUT / "34_gene_by_module.csv")
        R["hubs"] = {"n_hubs": int(len(hubs))}
        print(f"    hubs (>=3 modules)={len(hubs)}")
except Exception:
    warn("3.4 failed:\n" + traceback.format_exc())

# ========== 3.5 WITHIN-MODULE CONCORDANCE vs MEAN SHIFT ==============
print("\n### 3.5 Module concordance ###")
modstats = pd.DataFrame()
try:
    rows = []
    for mod, sub in d[d["module"] != "other"].groupby("module"):
        x, y = sub["_scz"], sub["_trs"]
        m = x.notna() & y.notna()
        if m.sum() < 8:
            continue
        r = float(np.corrcoef(x[m], y[m])[0, 1]) if m.sum() > 2 else np.nan
        rows.append({
            "module": mod, "n_gene_rows": int(m.sum()),
            "n_unique_genes": int(sub.loc[m, gcol].nunique()),
            "pearson_r": r,
            "sign_concordance": float((np.sign(x[m]) == np.sign(y[m])).mean()),
            "mean_delta_TRS_minus_SCZ": float((y[m] - x[m]).mean()),
            "mean_Z": float(pd.concat([x[m], y[m]]).mean()),
        })
    modstats = pd.DataFrame(rows).sort_values("pearson_r", ascending=False)
    if not modstats.empty:
        modstats.to_csv(OUT / "35_module_concordance.csv", index=False)
        R["modstats"] = {"n_modules": int(len(modstats))}
        print(modstats.to_string(index=False))
except Exception:
    warn("3.5 failed:\n" + traceback.format_exc())

# =============== 3.6 ANCHOR-SET GENE TABLE ===========================
print("\n### 3.6 Anchor-set gene table ###")
anchor_tab = pd.DataFrame()
try:
    if not gene.empty:
        focus_genes = d[d["GeneSet"].isin(ANCHOR_SETS)][gcol].unique()
        anchor_tab = gene[gene[gcol].isin(focus_genes)].copy()

        icol = firstcol(infl, "Influential", "influential", "is_influential")
        g_inf = firstcol(infl, "Gene", "gene", "GeneName")
        d_inf = firstcol(infl, "Disease", "disease", "Comparison")
        if (not infl.empty) and icol and g_inf and d_inf:
            infl2 = infl.copy()
            infl2["_infl"] = infl2[icol].astype(str).str.lower().isin(
                ["true", "1", "yes", "t"])
            if "GeneSet" in infl2.columns:
                infl_f = infl2[infl2["GeneSet"].isin(ANCHOR_SETS)]
            else:
                infl_f = infl2
            if not infl_f.empty:
                flags = (infl_f.groupby([g_inf, d_inf])["_infl"].max()
                         .unstack(d_inf).add_prefix("influential_"))
                anchor_tab = anchor_tab.merge(flags, left_on=gcol,
                                              right_index=True, how="left")
        anchor_tab = anchor_tab.sort_values("mean_Z")
        anchor_tab.to_csv(OUT / "36_anchor_set_genes.csv", index=False)
        R["anchor"] = {"n_genes": int(len(anchor_tab))}
        print(f"    anchor-set genes={len(anchor_tab)}")
except Exception:
    warn("3.6 failed:\n" + traceback.format_exc())

# ============ 3.7 MECHANISTIC LAYER ROSTER ===========================
print("\n### 3.7 Mechanistic roster ###")
roster = pd.DataFrame()
try:
    if not gene.empty:
        roster = (gene[gene["class"].ne("weak")]
                  [[gcol, "class", "Z_scz", "Z_trs", "delta", "layers",
                    "modules", "n_sets", "n_modules"]]
                  .sort_values(["class", "mean_Z"] if "mean_Z" in gene else ["class"]))
        # re-add mean_Z ordering safely
        roster = roster.merge(gene[[gcol, "mean_Z"]], on=gcol, how="left") \
                       .sort_values(["class", "mean_Z"])
        roster.to_csv(OUT / "37_mechanistic_gene_roster.csv", index=False)
        R["roster"] = {"n": int(len(roster)),
                       "by_class_layer": roster.groupby(["class", "layers"]).size()
                                         .to_dict()}
        print(f"    roster rows={len(roster)}")
except Exception:
    warn("3.7 failed:\n" + traceback.format_exc())

# =====================================================================
#                    VERY DETAILED SUMMARY.txt
# =====================================================================
print("\n### Writing SUMMARY.txt ###")
S = []
def w(*a): S.append(" ".join(str(x) for x in a))
def wrap(items, width=90, indent="        "):
    txt = ", ".join(str(i) for i in items)
    return "\n".join(textwrap.wrap(txt, width=width, initial_indent=indent,
                                   subsequent_indent=indent)) or (indent + "(none)")
def fmt(x, nd=3):
    try:
        xf = float(x)
        if np.isnan(xf): return "NA"
        return f"{xf:.{nd}g}" if (abs(xf) < 1e-3 or abs(xf) >= 1e4) else f"{xf:.{nd}f}"
    except Exception:
        return "NA"
bar = "=" * 80

w(bar)
w("091IvD  STAGE 3  —  GENE-LEVEL CONVERGENCE / DIVERGENCE SUMMARY")
w("The unit of discovery is the GENE, not the GO label.")
w(bar)
w(f"Generated   : {TS}")
w(f"Results dir : {RESULTS}")
w(f"Output dir  : {OUT}")
w(f"Cohorts     : TRS='{TRS_LAB}'  vs  SCZ='{SCZ_LAB}'")
w(f"             (columns {z_trs} / {z_scz})")
w("")
w("Input tables (rows x cols):")
for k, v in R["loads"].items():
    w(f"  {k:16s} {v[0]:6d} x {v[1]}")
w("")
w("Classification thresholds:")
w(f"  converge : same sign AND both |Z| > {CONVERGE_Z}")
w(f"  diverge_sign  : opposite sign AND min|Z| > {DIVERGE_MIN}")
w(f"  diverge_shift : |Z_TRS - Z_SCZ| > {DELTA_SHIFT}")
w("")

# ---- Section 1: class overview ----
w(bar); w("1. GENE CLASS OVERVIEW"); w(bar)
if R.get("classes"):
    total = sum(R["classes"].values())
    w(f"  Total unique genes: {R['matrix'].get('n_genes', total)}")
    for cls in ["converge_down", "converge_up", "diverge_sign", "diverge_shift", "weak"]:
        n = R["classes"].get(cls, 0)
        pct = 100 * n / total if total else 0
        w(f"    {cls:16s}: {n:5d}  ({pct:4.1f}%)")
    w("")
    w("  Interpretation:")
    w("    converge_down = gene-level content of the motility / kinase GO claims")
    w("                    (report as NEGATIVE aggregate TWAS association).")
    w("    converge_up   = weaker shared developmental / synaptic tail.")
    w("    diverge_*     = the ONLY genes that can support a TRS-vs-SCZ sentence.")
else:
    w("  (matrix not built)")
w("")

# ---- Section 2: convergent DOWN core ----
w(bar); w("2. CONVERGENT DOWN CORE  (shared negative TWAS Z, both cohorts)"); w(bar)
if not core_down.empty:
    w(f"  {len(core_down)} genes. Top 25 by shared negative Z:")
    w("")
    w(f"    {'GENE':16s} {'Z_'+SCZ_LAB[:6]:>10s} {'Z_'+TRS_LAB[:6]:>10s} "
      f"{'delta':>8s}  modules")
    for _, r in core_down.head(25).iterrows():
        w(f"    {str(r[gcol])[:16]:16s} {fmt(r['Z_scz']):>10s} {fmt(r['Z_trs']):>10s} "
          f"{fmt(r['delta']):>8s}  {str(r['modules'])[:44]}")
    w("  Table: 33_converge_down.csv")
else:
    w("  (none reached the double-cohort |Z|>1.96 gate)")
w("")

# ---- Section 3: convergent UP core ----
w(bar); w("3. CONVERGENT UP CORE  (shared positive TWAS Z, both cohorts)"); w(bar)
if not core_up.empty:
    w(f"  {len(core_up)} genes. Top 15:")
    w("")
    for _, r in core_up.head(15).iterrows():
        w(f"    {str(r[gcol])[:16]:16s} Z_{SCZ_LAB[:5]}={fmt(r['Z_scz'])} "
          f"Z_{TRS_LAB[:5]}={fmt(r['Z_trs'])}  modules={str(r['modules'])[:40]}")
    w("  Table: 33_converge_up.csv")
else:
    w("  (none)")
w("")

# ---- Section 4: divergent tail ----
w(bar); w("4. DIVERGENT TAIL  (TRS-specific hypotheses ONLY)"); w(bar)
if not div.empty:
    w(f"  {len(div)} genes (sign flip or |delta Z| > {DELTA_SHIFT}). Top 20 by |delta|:")
    w("")
    w(f"    {'GENE':16s} {'class':14s} {'Z_'+SCZ_LAB[:5]:>9s} {'Z_'+TRS_LAB[:5]:>9s} "
      f"{'delta':>8s}  modules")
    for _, r in div.head(20).iterrows():
        w(f"    {str(r[gcol])[:16]:16s} {str(r['class'])[:14]:14s} "
          f"{fmt(r['Z_scz']):>9s} {fmt(r['Z_trs']):>9s} {fmt(r['delta']):>8s}  "
          f"{str(r['modules'])[:34]}")
    w("")
    w("  CAUTION: keep the three confounders in every TRS sentence —")
    w("           chronicity, cumulative drug exposure, ascertainment.")
    w("           These are TRS HYPOTHESES, not TRS mechanisms.")
    w("  Table: 33_diverge.csv")
else:
    w("  (no divergent genes — TRS and SCZ-EUR profiles are concordant)")
w("")

# ---- Section 5: cross-layer hubs ----
w(bar); w("5. CROSS-LAYER HUB GENES  (>=3 modules — wiring-diagram nodes)"); w(bar)
if not hubs.empty:
    w(f"  {len(hubs)} hub genes. These belong in the DIAGRAM, not as extra GO hits:")
    w("")
    for _, r in hubs.head(20).iterrows():
        w(f"    {str(r[gcol])[:16]:16s} n_modules={int(r['n_modules'])} "
          f"n_sets={int(r['n_sets'])}  Z_{SCZ_LAB[:5]}={fmt(r['Z_scz'])} "
          f"Z_{TRS_LAB[:5]}={fmt(r['Z_trs'])} [{r['class']}]")
        w(f"        modules: {str(r['modules'])[:100]}")
    w("  Tables: 34_layer_hubs.csv, 34_gene_by_module.csv")
else:
    w("  (no genes span >=3 modules)")
w("")

# ---- Section 6: module concordance ----
w(bar); w("6. WITHIN-MODULE CONCORDANCE vs MEAN SHIFT"); w(bar)
if not modstats.empty:
    w("  Gene-level 'shared profile, rare mean shift', per module:")
    w("")
    w(f"    {'module':26s} {'n_gene':>7s} {'r':>7s} {'sign_conc':>10s} "
      f"{'mean_dZ(TRS-SCZ)':>18s} {'mean_Z':>8s}")
    for _, r in modstats.iterrows():
        w(f"    {str(r['module'])[:26]:26s} {int(r['n_gene_rows']):>7d} "
          f"{fmt(r['pearson_r']):>7s} {fmt(r['sign_concordance']):>10s} "
          f"{fmt(r['mean_delta_TRS_minus_SCZ']):>18s} {fmt(r['mean_Z']):>8s}")
    w("")
    w("  Read: high r + high sign_conc = shared architecture (not a 2nd enrichment).")
    w("        the module with the most negative mean_dZ carries the TRS shift.")
    w("  Table: 35_module_concordance.csv")
else:
    w("  (insufficient module-level rows)")
w("")

# ---- Section 7: anchor-set gene table ----
w(bar); w("7. ANCHOR-SET GENE TABLE  (the only gene table for the GO-level claim)"); w(bar)
if not anchor_tab.empty:
    infl_cols = [c for c in anchor_tab.columns if c.startswith("influential_")]
    w(f"  {len(anchor_tab)} genes across the two anchor sets "
      f"(motility + Ser/Thr kinase).")
    if infl_cols:
        w(f"  Influence flags present: {infl_cols}")
    w("")
    for _, r in anchor_tab.iterrows():
        fl = ""
        for c in infl_cols:
            if str(r.get(c)).lower() in ("true", "1", "1.0", "yes"):
                fl += f" [{c.replace('influential_','INFL:')}]"
        w(f"    {str(r[gcol])[:16]:16s} Z_{SCZ_LAB[:5]}={fmt(r['Z_scz'])} "
          f"Z_{TRS_LAB[:5]}={fmt(r['Z_trs'])} [{r['class']}]{fl}")
    w("")
    w("  If motility has NO influential flags -> Stage-2 robustness restated at")
    w("  gene resolution. If a single gene flags in the kinase set and the rest")
    w("  stays converge_down, keep the set sentence + name the gene in a footnote.")
    w("  Table: 36_anchor_set_genes.csv")
else:
    w("  (anchor sets not present in gene_detail)")
w("")

# ---- Section 8: mechanistic roster ----
w(bar); w("8. MECHANISTIC LAYER ROSTER  (non-weak genes -> layer candidates)"); w(bar)
if not roster.empty:
    w(f"  {len(roster)} classified genes assigned to layer candidates.")
    w("  Counts by (class, layers):")
    for (cls, lay), n in sorted(R["roster"].get("by_class_layer", {}).items()):
        w(f"    {str(cls):16s} | {str(lay):40s} : {n}")
    w("")
    w("  Suggested diagram scaffold (populate from 37_mechanistic_gene_roster.csv):")
    w("    enhancer-derived sets")
    w("        v")
    w("    L2 cytoskeleton/guidance  <- converge_down in migration_cytoskeleton")
    w("        v")
    w("    L3 adhesion/synapse hubs  <- high concordance, recycled hub genes")
    w("        v")
    w("    L4 signaling nodes        <- converge_down, footnote influential genes")
    w("        v")
    w("    L4 excitability (K+/HCN)  <- secondary; TRS-tilted only if in diverge")
    w("  Table: 37_mechanistic_gene_roster.csv")
else:
    w("  (roster empty)")
w("")

# ---- Section 9: reporting gate ----
w(bar); w("9. RECOMMENDED REPORTING GATE"); w(bar)
w("  Report a GO-level claim only if the set:")
w("    (a) survived the Stage-2 conservative leaderboard (CI!=0 AND perm p<0.05),")
w("    (b) its genes sit in converge_down/up with |Z|>1.96 in BOTH cohorts here,")
w("    (c) is not carried by a single recycled hub/driver gene (Section 5 + hubs).")
w("  A TRS-vs-SCZ claim needs genes in Section 4 (diverge_*), with confounders stated.")
w("  Everything else is a driver-level / hub observation, not a pathway result.")
w("")
w("  Psychedelic sentence that still matches the data:")
w("    Gene-level convergence on motility/cytoskeleton + kinase nodes that a")
w("    persistent-enhancer experiment also tagged supports a SHARED plasticity-")
w("    related regulatory substrate. It does NOT choose among therapeutic")
w("    recruitment, shared vulnerability, or generic constrained-gene overlap.")
w("")

# ---- Section 10: warnings ----
if R["warnings"]:
    w(bar); w("10. RUN WARNINGS / NOTES"); w(bar)
    for m in R["warnings"]:
        for ln in str(m).splitlines():
            w("  " + ln)
    w("")

# ---- Section 11: file inventory ----
w(bar); w("11. OUTPUT FILES"); w(bar)
for f in sorted(glob.glob(str(OUT / "**" / "*"), recursive=True)):
    if os.path.isfile(f):
        w(f"  {f}  ({os.path.getsize(f)//1024} KB)")
w(""); w(bar); w("END OF STAGE 3 SUMMARY"); w(bar)

with open(OUT / "SUMMARY.txt", "w") as fh:
    fh.write("\n".join(S) + "\n")

print("\n=== STAGE 3 DONE ===")
print("Summary:", (OUT / "SUMMARY.txt").resolve())
print("\n----- SUMMARY.txt preview (first 80 lines) -----")
print("\n".join(S[:80]))

>>> Loaded: {'gene_detail': (1301, 7), 'gene_influence': (1909, 13)}
>>> Cohorts: TRS='trs' (Z_trs)   SCZ='scz_eur' (Z_scz_eur)
  [warn] 1242/1301 gene_detail rows mapped to a named module (1242 rows via SET2MOD)

### 3.1 Gene × cohort matrix ###
    class counts: {'weak': 498, 'diverge_shift': 60, 'converge_up': 54, 'converge_down': 53, 'diverge_sign': 37}

### 3.2 Scatter plots ###
    figures written

### 3.3 Ranked lists ###
    converge_down=53  converge_up=54  diverge=97

### 3.4 Layer hubs ###
    hubs (>=3 modules)=30

### 3.5 Module concordance ###
                module  n_gene_rows  n_unique_genes  pearson_r  sign_concordance  mean_delta_TRS_minus_SCZ    mean_Z
          excitability           49              48   0.854183          0.755102                  0.088137  0.488111
      adhesion_synapse          145              79   0.839033          0.917241                 -0.461945 -0.060481
migration_cytoskeleton          172             130   0.819696          0.872093     

# Stage 4

In [12]:
# =====================================================================
# 091IvD  STAGE 4  —  ROBUSTNESS STRESS TEST (CSV-ONLY)
# Not a discovery pass. Each block moves a robustness score for a claim
# you are willing to print. Consumes ONLY Stage-3 + pipeline CSVs.
# One cell. Ends with stage4/SUMMARY.txt (very detailed) + ledger.
# =====================================================================

from pathlib import Path
import os, re, glob, textwrap, datetime, warnings, traceback
from itertools import combinations
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

# ---------------------------- CONFIG ---------------------------------
RESULTS = Path("/content/091IvD_results")     # <-- change if needed
S3      = RESULTS / "stage3"
OUT     = RESULTS / "stage4"
FIG     = OUT / "figures"
OUT.mkdir(parents=True, exist_ok=True)
FIG.mkdir(parents=True, exist_ok=True)

Z_SIG        = 1.96
DELTA_SHIFT  = 2.0
DIVERGE_MIN  = 1.0
BOOT_N       = 2000
RNG_SEED     = 1
CONC_MIN_OK  = 0.75      # pass rule for "same map" concordance after ablation
TS           = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")

# Claim-anchor gene lists (validated against data below; missing genes dropped)
DRIVERS        = {"CD40", "GIGYF1", "GCH1", "CACNB3"}
MOTILITY_CORE  = {"TNR","CCDC25","RHOA","NEXN","TENM2","ERBB4",
                  "ADRA2A","SPATA13","PLEKHG3","LDB2"}
TRS_CAND       = {"AMIGO1","KCNQ2","CNTN1","ST8SIA2","CDH10","ITM2B","GLI2"}
ADHESION_LIKE  = {"AMIGO1","CNTN1","CDH10","ST8SIA2","ITM2B"}

R = {"loads": {}, "cohorts": {}, "validate": {}, "sens": {}, "trs_sens": {},
     "loo": {}, "kinase": {}, "ablation": {}, "conc": {}, "adh": {},
     "signstab": {}, "boot": {}, "overlap": {}, "ledger": {}, "warnings": []}

def warn(msg):
    print("  [warn]", msg); R["warnings"].append(str(msg))

def firstcol(df, *names):
    if df is None or df.empty:
        return None
    lower = {c.lower(): c for c in df.columns}
    for n in names:
        if n.lower() in lower:
            return lower[n.lower()]
    return None

def stouffer(z):
    z = np.asarray(z, float); z = z[np.isfinite(z)]
    return float(np.sum(z) / np.sqrt(len(z))) if len(z) >= 3 else np.nan

# ============================ 1. LOAD =================================
def rd(p):
    try:
        return pd.read_csv(p)
    except Exception as ex:
        warn(f"could not read {getattr(p,'name',p)}: {ex}")
        return pd.DataFrame()

def stack(globpat, strip_prefix):
    frames = []
    for p in sorted(RESULTS.rglob(globpat)):
        if p.name.startswith("multi_"):
            continue
        df = rd(p)
        if df.empty:
            continue
        if "GeneSet" not in df.columns and "label" not in df.columns:
            df.insert(0, "GeneSet", re.sub(strip_prefix, "", p.stem))
        frames.append(df)
    if frames:
        out = pd.concat(frames, ignore_index=True, sort=False)
        if "GeneSet" not in out.columns and "label" in out.columns:
            out.rename(columns={"label": "GeneSet"}, inplace=True)
        return out
    return pd.DataFrame()

g    = rd(S3 / "31_gene_cohort_matrix.csv")
down = rd(S3 / "33_converge_down.csv")
up   = rd(S3 / "33_converge_up.csv")
div  = rd(S3 / "33_diverge.csv")
anc  = rd(S3 / "36_anchor_set_genes.csv")
mod  = rd(S3 / "35_module_concordance.csv")
hubs = rd(S3 / "34_layer_hubs.csv")
infl = stack("gene_influence_*.csv", r"^gene_influence_")
if infl.empty:
    infl = rd(next(iter(RESULTS.rglob("multi_geneset_gene_influence.csv")), Path("___")))

R["loads"] = {"31_matrix": tuple(g.shape), "33_down": tuple(down.shape),
              "33_up": tuple(up.shape), "33_div": tuple(div.shape),
              "36_anchor": tuple(anc.shape), "35_modconc": tuple(mod.shape),
              "34_hubs": tuple(hubs.shape), "gene_influence": tuple(infl.shape)}
print(">>> Loaded:", R["loads"])

if g.empty:
    raise SystemExit("FATAL: stage3/31_gene_cohort_matrix.csv not found — run Stage 3 first.")

# ---- resolve columns on the master matrix ----
gene_col = firstcol(g, "Gene", "gene", "GeneName") or g.columns[0]
ztrs = firstcol(g, "Z_trs", "Z_TRS")
zscz = firstcol(g, "Z_scz", "Z_SCZ", "Z_scz_eur")
if ztrs is None:
    ztrs = next((c for c in g.columns if re.search(r"^Z_.*trs", c, re.I)), None) or "Z_trs"
if zscz is None:
    zscz = next((c for c in g.columns if re.search(r"^Z_.*(scz|eur)", c, re.I)), None) or "Z_scz"
# standardise to Z_trs / Z_scz internally
if ztrs != "Z_trs" and ztrs in g.columns:
    g = g.rename(columns={ztrs: "Z_trs"})
if zscz != "Z_scz" and zscz in g.columns:
    g = g.rename(columns={zscz: "Z_scz"})
for c in ("Z_trs", "Z_scz"):
    if c not in g.columns:
        raise SystemExit(f"FATAL: matrix missing {c}. Columns={list(g.columns)}")
    g[c] = pd.to_numeric(g[c], errors="coerce")

# rebuild derived cols if absent
if "delta" not in g.columns:
    g["delta"] = g["Z_trs"] - g["Z_scz"]
if "class" not in g.columns:
    same = np.sign(g["Z_trs"]) == np.sign(g["Z_scz"])
    minabs = g[["Z_trs", "Z_scz"]].abs().min(axis=1)
    g["class"] = np.select(
        [same & g["Z_scz"].lt(-Z_SIG) & g["Z_trs"].lt(-Z_SIG),
         same & g["Z_scz"].gt(Z_SIG) & g["Z_trs"].gt(Z_SIG),
         (~same) & minabs.gt(DIVERGE_MIN),
         g["delta"].abs().gt(DELTA_SHIFT)],
        ["converge_down", "converge_up", "diverge_sign", "diverge_shift"], "weak")
if "modules" not in g.columns:
    g["modules"] = "other"
    warn("matrix had no 'modules' column; module-based blocks limited")

TRS_LAB = "TRS"; SCZ_LAB = "SCZ"
R["cohorts"] = {"gene_col": gene_col, "z_trs": "Z_trs", "z_scz": "Z_scz"}
print(f">>> gene col='{gene_col}'  cohorts: Z_trs / Z_scz")

# ---- validate configured gene lists against data ----
present = set(g[gene_col].astype(str))
def keep_present(s, name):
    ok = {x for x in s if x in present}
    miss = s - ok
    if miss:
        warn(f"{name}: {len(miss)} not in matrix -> {sorted(miss)}")
    return ok
DRIVERS_D       = keep_present(DRIVERS, "DRIVERS")
MOTILITY_CORE_D = keep_present(MOTILITY_CORE, "MOTILITY_CORE")
TRS_CAND_D      = keep_present(TRS_CAND, "TRS_CAND")
ADHESION_LIKE_D = keep_present(ADHESION_LIKE, "ADHESION_LIKE")
R["validate"] = {"drivers_present": sorted(DRIVERS_D),
                 "motility_present": sorted(MOTILITY_CORE_D),
                 "trs_cand_present": sorted(TRS_CAND_D),
                 "adhesion_present": sorted(ADHESION_LIKE_D)}

hub_names  = set(hubs[firstcol(hubs, "Gene", "gene") or "Gene"].astype(str)) if not hubs.empty else set()
tail_names = set(div[firstcol(div, "Gene", "gene") or "Gene"].astype(str)) if not div.empty else \
             set(g.loc[g["class"].isin(["diverge_sign", "diverge_shift"]), gene_col].astype(str))

# ============ 4.1 THRESHOLD SENSITIVITY (down/up split) ==============
print("\n### 4.1 Threshold sensitivity ###")
try:
    rows = []
    for zcut in (1.0, 1.64, 1.96, 2.58, 3.0):
        same = np.sign(g["Z_trs"]) == np.sign(g["Z_scz"])
        cd = same & g["Z_trs"].lt(-zcut) & g["Z_scz"].lt(-zcut)
        cu = same & g["Z_trs"].gt(zcut) & g["Z_scz"].gt(zcut)
        rows.append({
            "zcut": zcut, "n_genes": len(g),
            "converge_down": int(cd.sum()), "converge_up": int(cu.sum()),
            "ratio_down_over_up": round(cd.sum() / max(cu.sum(), 1), 3),
            "n_motility_core_in_down": int(g.loc[cd, gene_col].isin(MOTILITY_CORE_D).sum()),
        })
    sens = pd.DataFrame(rows)
    sens.to_csv(OUT / "41_threshold_sensitivity.csv", index=False)
    # pass rule: ratio stays within 0.5..2 across 1.64-2.58, motility core present
    band = sens[sens["zcut"].isin([1.64, 1.96, 2.58])]
    ratio_ok = bool(band["ratio_down_over_up"].between(0.5, 2.0).all())
    mot_ok = bool((band["n_motility_core_in_down"] >= max(1, len(MOTILITY_CORE_D)//2)).all())
    R["sens"] = {"table": sens, "ratio_stable": ratio_ok, "motility_stable": mot_ok,
                 "pass": ratio_ok and mot_ok}
    print(sens.to_string(index=False))
    print(f"    bidirectional-split PASS={R['sens']['pass']}")
except Exception:
    warn("4.1 failed:\n" + traceback.format_exc())

# ---- TRS tail sensitivity ----
try:
    rows = []
    for dcut in (1.0, 1.5, 2.0, 2.5, 3.0):
        tail = g["delta"].abs().gt(dcut) | (
            (np.sign(g["Z_trs"]) != np.sign(g["Z_scz"])) &
            g[["Z_trs", "Z_scz"]].abs().min(axis=1).gt(1.0))
        rows.append({"delta_cut": dcut, "n_tail": int(tail.sum()),
                     "frac_of_genes": round(float(tail.mean()), 4),
                     "n_trs_cand_kept": int(g.loc[tail, gene_col].isin(TRS_CAND_D).sum())})
    tsens = pd.DataFrame(rows)
    tsens.to_csv(OUT / "41_trs_tail_sensitivity.csv", index=False)
    mid = tsens[tsens["delta_cut"].isin([1.5, 2.0, 2.5])]
    tail_minor = bool((mid["frac_of_genes"] < 0.5).all())
    cand_kept = bool((mid["n_trs_cand_kept"] >= max(1, len(TRS_CAND_D)//2)).all())
    R["trs_sens"] = {"table": tsens, "tail_minority": tail_minor,
                     "cand_kept": cand_kept, "pass": tail_minor and cand_kept}
    print(tsens.to_string(index=False))
    print(f"    TRS-tail-minority PASS={R['trs_sens']['pass']}")
except Exception:
    warn("4.1 TRS tail failed:\n" + traceback.format_exc())

# ============ 4.2 LEAVE-k-OUT ON MOTILITY CORE ======================
print("\n### 4.2 Motility leave-k-out ###")
try:
    core = g[g[gene_col].isin(MOTILITY_CORE_D)].copy()
    base_scz = stouffer(core["Z_scz"]); base_trs = stouffer(core["Z_trs"])
    combos = [set()] + [{x} for x in MOTILITY_CORE_D] + \
             [set(p) for p in combinations(sorted(MOTILITY_CORE_D), 2)]
    rows = []
    for drop in combos:
        keep = core[~core[gene_col].isin(drop)]
        rows.append({
            "dropped": ",".join(sorted(drop)) or "NONE",
            "k_dropped": len(drop), "n_left": len(keep),
            "stouffer_scz": round(stouffer(keep["Z_scz"]), 4),
            "stouffer_trs": round(stouffer(keep["Z_trs"]), 4),
            "n_both_lt_-1.96": int((keep["Z_scz"].lt(-1.96) & keep["Z_trs"].lt(-1.96)).sum()),
        })
    loo = pd.DataFrame(rows)
    if np.isfinite(base_scz) and base_scz != 0:
        loo["flip_scz"] = np.sign(loo["stouffer_scz"]) != np.sign(base_scz)
    loo.to_csv(OUT / "42_motility_leavek.csv", index=False)
    # pass: after any leave-2, both cohort stouffer < -1.96 and >=6 double-sig genes
    k2 = loo[loo["k_dropped"] == 2]
    surv_stouffer = bool((k2["stouffer_scz"] < -1.96).all() and (k2["stouffer_trs"] < -1.96).all()) if len(k2) else False
    min_double = int(k2["n_both_lt_-1.96"].min()) if len(k2) else 0
    R["loo"] = {"base_scz": base_scz, "base_trs": base_trs, "n_core": len(core),
                "k2_min_double": min_double, "k2_stouffer_survives": surv_stouffer,
                "pass": surv_stouffer and min_double >= 6, "table": loo}
    print(f"    base Stouffer SCZ={base_scz:.2f} TRS={base_trs:.2f}  n_core={len(core)}")
    print(f"    leave-2 min double-sig={min_double}  stouffer survives={surv_stouffer}  "
          f"PASS={R['loo']['pass']}")
except Exception:
    warn("4.2 failed:\n" + traceback.format_exc())

# ---- kinase residual after removing CD40 ----
try:
    # anchor table preferred, else module tag on matrix
    if not anc.empty and "modules" in anc.columns:
        ag = firstcol(anc, "Gene", "gene") or "Gene"
        for c in ("Z_scz", "Z_trs"):
            if c not in anc.columns:
                anc[c] = anc.get(c, np.nan)
        kin = anc[anc["modules"].astype(str).str.contains("signaling", na=False)].copy()
        kin_gene = ag
    else:
        kin = g[g["modules"].astype(str).str.contains("signaling_plasticity", na=False)].copy()
        kin_gene = gene_col
    if kin.empty:
        warn("kinase residual: no signaling-module genes found")
    else:
        kin_nocd40 = kin[~kin[kin_gene].astype(str).eq("CD40")]
        res_scz = stouffer(pd.to_numeric(kin_nocd40.get("Z_scz"), errors="coerce"))
        res_trs = stouffer(pd.to_numeric(kin_nocd40.get("Z_trs"), errors="coerce"))
        n_cd = int((kin_nocd40.get("class", pd.Series(dtype=str)) == "converge_down").sum()) \
               if "class" in kin_nocd40.columns else np.nan
        R["kinase"] = {"n_kinase": len(kin), "n_after_cd40": len(kin_nocd40),
                       "resid_stouffer_scz": res_scz, "resid_stouffer_trs": res_trs,
                       "n_converge_down": n_cd,
                       "multigene_ok": bool(np.isfinite(res_scz) and abs(res_scz) >= 2.0)}
        print(f"    kinase residual (no CD40): Stouffer SCZ={res_scz}  TRS={res_trs}  "
              f"n_down={n_cd}  multigene_ok={R['kinase']['multigene_ok']}")
except Exception:
    warn("4.2 kinase residual failed:\n" + traceback.format_exc())

# ============ 4.3 DRIVER ABLATION ===================================
print("\n### 4.3 Driver ablation ###")
try:
    def recount(df):
        return {
            "n": len(df),
            "down": int((df["class"] == "converge_down").sum()),
            "up": int((df["class"] == "converge_up").sum()),
            "div": int(df["class"].isin(["diverge_sign", "diverge_shift"]).sum()),
            "motility_core_left": int(df.loc[df["class"] == "converge_down", gene_col]
                                      .isin(MOTILITY_CORE_D).sum()),
        }
    recs = {"all": recount(g)}
    for dgene in sorted(DRIVERS_D):
        recs[f"drop_{dgene}"] = recount(g[g[gene_col].ne(dgene)])
    recs["drop_all_drivers"] = recount(g[~g[gene_col].isin(DRIVERS_D)])
    abl = pd.DataFrame(recs).T
    abl.to_csv(OUT / "43_driver_ablation.csv")
    base = recs["all"]; alld = recs["drop_all_drivers"]
    motility_robust = bool(alld["motility_core_left"] >= max(1, base["motility_core_left"] - 1))
    up_fragile = bool(base["up"] > 0 and alld["up"] <= base["up"] * 0.6)
    R["ablation"] = {"table": abl, "base": base, "drop_all": alld,
                     "motility_robust": motility_robust, "up_fragile": up_fragile}
    print(abl.to_string())
    print(f"    motility survives driver ablation={motility_robust}  up-core fragile={up_fragile}")
except Exception:
    warn("4.3 failed:\n" + traceback.format_exc())

# ============ 4.4 CONCORDANCE ABLATION ==============================
print("\n### 4.4 Concordance ablation ###")
try:
    def corr_on(sub):
        m = sub["Z_scz"].notna() & sub["Z_trs"].notna()
        if m.sum() < 10:
            return np.nan, np.nan, int(m.sum())
        r = float(np.corrcoef(sub.loc[m, "Z_scz"], sub.loc[m, "Z_trs"])[0, 1])
        c = float((np.sign(sub.loc[m, "Z_scz"]) == np.sign(sub.loc[m, "Z_trs"])).mean())
        return r, c, int(m.sum())

    rows = []
    for label, mask in [
        ("all", pd.Series(True, index=g.index)),
        ("no_hubs", ~g[gene_col].isin(hub_names)),
        ("no_tail", ~g[gene_col].isin(tail_names)),
        ("no_hubs_no_tail", ~g[gene_col].isin(hub_names | tail_names)),
        ("no_drivers", ~g[gene_col].isin(DRIVERS_D)),
    ]:
        r, c, n = corr_on(g.loc[mask])
        rows.append({"subset": label, "n": n, "pearson_r": r, "sign_conc": c})
    # per-module without tail
    if (g["modules"] != "other").any():
        for modname, sub in g.groupby(g["modules"].astype(str).str.split(";").str[0]):
            r, c, n = corr_on(sub[~sub[gene_col].isin(tail_names)])
            rows.append({"subset": f"{modname}_no_tail", "n": n, "pearson_r": r, "sign_conc": c})
    conc = pd.DataFrame(rows)
    conc.to_csv(OUT / "44_concordance_ablation.csv", index=False)
    r_all = conc.loc[conc["subset"] == "all", "pearson_r"].iloc[0] if len(conc) else np.nan
    r_nht = conc.loc[conc["subset"] == "no_hubs_no_tail", "pearson_r"]
    r_nht = float(r_nht.iloc[0]) if len(r_nht) else np.nan
    same_map = bool(np.isfinite(r_nht) and r_nht >= CONC_MIN_OK)
    R["conc"] = {"table": conc, "r_all": r_all, "r_no_hubs_no_tail": r_nht,
                 "same_map_pass": same_map}
    print(conc.to_string(index=False))
    print(f"    r(all)={r_all}  r(no hubs/tail)={r_nht}  same-map PASS={same_map}")
except Exception:
    warn("4.4 failed:\n" + traceback.format_exc())

# ---- adhesion-candidate ablation of module ΔZ ----
try:
    rows = []
    for theme, gs in [("adhesion_like", ADHESION_LIKE_D), ("kcnq2_only", {"KCNQ2"})]:
        gs = {x for x in gs if x in present}
        sub = g[g[gene_col].isin(gs)]
        rest = g[g["modules"].astype(str).str.contains("adhesion|neurite", na=False)
                 & ~g[gene_col].isin(gs)]
        rows.append({"theme": theme, "n_cand": len(sub),
                     "cand_mean_delta": round(float(sub["delta"].mean()), 4) if len(sub) else np.nan,
                     "n_rest_module": len(rest),
                     "rest_mean_delta": round(float(rest["delta"].mean()), 4) if len(rest) else np.nan})
    adh = pd.DataFrame(rows)
    adh.to_csv(OUT / "44_adhesion_ablation.csv", index=False)
    ar = adh[adh["theme"] == "adhesion_like"]
    shift_is_candidates = bool(len(ar) and pd.notna(ar["rest_mean_delta"].iloc[0])
                               and abs(ar["rest_mean_delta"].iloc[0]) < 0.5
                               and abs(ar["cand_mean_delta"].iloc[0]) >= 0.5)
    R["adh"] = {"table": adh, "shift_is_candidates": shift_is_candidates}
    print(adh.to_string(index=False))
    print(f"    adhesion TRS shift = named list (not module)? {shift_is_candidates}")
except Exception:
    warn("4.4 adhesion ablation failed:\n" + traceback.format_exc())

# ============ 4.5 SIGN STABILITY + BOOTSTRAP OF CORE r ==============
print("\n### 4.5 Motility-core sign stability + bootstrap ###")
try:
    mc = g[g[gene_col].isin(MOTILITY_CORE_D)].copy()
    all_same = bool((np.sign(mc["Z_scz"]) == np.sign(mc["Z_trs"])).all()) if len(mc) else False
    min_abs = float(mc[["Z_scz", "Z_trs"]].abs().min().min()) if len(mc) else np.nan
    vals = mc[["Z_scz", "Z_trs"]].dropna().to_numpy()
    ci = [np.nan, np.nan, np.nan]
    if len(vals) >= 5:
        rng = np.random.default_rng(RNG_SEED)
        boots = []
        for _ in range(BOOT_N):
            i = rng.integers(0, len(vals), len(vals))
            if np.std(vals[i, 0]) > 0 and np.std(vals[i, 1]) > 0:
                boots.append(np.corrcoef(vals[i, 0], vals[i, 1])[0, 1])
        if boots:
            ci = list(np.quantile(boots, [0.025, 0.5, 0.975]))
    mc[[gene_col, "Z_scz", "Z_trs", "class"]].to_csv(OUT / "45_motility_sign_stability.csv", index=False)
    ci_excludes0 = bool(np.isfinite(ci[0]) and np.isfinite(ci[2]) and np.sign(ci[0]) == np.sign(ci[2]))
    R["signstab"] = {"n_core": len(mc), "all_same_sign": all_same, "min_abs": min_abs,
                     "boot_ci_low": ci[0], "boot_med": ci[1], "boot_ci_high": ci[2],
                     "core_ranking_tight": ci_excludes0}
    print(f"    n_core={len(mc)} all_same_sign={all_same} min|Z|={min_abs}")
    print(f"    core r bootstrap 95% = [{ci[0]:.3f}, {ci[2]:.3f}] (med {ci[1]:.3f})"
          if np.isfinite(ci[0]) else "    core r bootstrap unavailable")
except Exception:
    warn("4.5 failed:\n" + traceback.format_exc())

# ============ 4.6 SET-COMPOSITION CONTROL (motility vs kinase) ======
print("\n### 4.6 Motility/kinase overlap ###")
try:
    if (g["modules"] != "other").any():
        mot_genes = set(g.loc[g["modules"].astype(str)
                        .str.contains("migration_cytoskeleton", na=False), gene_col])
        kin_genes = set(g.loc[g["modules"].astype(str)
                        .str.contains("signaling_plasticity", na=False), gene_col])
        both = mot_genes & kin_genes
        mot_only = g[g[gene_col].isin(mot_genes - kin_genes)]
        mo_scz, mo_trs = stouffer(mot_only["Z_scz"]), stouffer(mot_only["Z_trs"])
        overlap_down = sorted(g.loc[g[gene_col].isin(both) & (g["class"] == "converge_down"), gene_col])
        pd.DataFrame({"overlap_gene": sorted(both)}).to_csv(OUT / "46_motility_kinase_overlap.csv", index=False)
        indep = bool(np.isfinite(mo_scz) and mo_scz < -1.5)
        R["overlap"] = {"n_motility": len(mot_genes), "n_kinase": len(kin_genes),
                        "n_overlap": len(both), "overlap_down": overlap_down,
                        "motility_only_scz": mo_scz, "motility_only_trs": mo_trs,
                        "motility_independent": indep}
        print(f"    n_mot={len(mot_genes)} n_kin={len(kin_genes)} overlap={len(both)} "
              f"mot-only Stouffer SCZ={mo_scz}  independent={indep}")
    else:
        warn("4.6 skipped: no module tags on matrix")
except Exception:
    warn("4.6 failed:\n" + traceback.format_exc())

# ============ 4.7 ROBUSTNESS LEDGER =================================
print("\n### 4.7 Robustness ledger ###")
try:
    def verdict(flag):
        return "PASS" if flag is True else ("FAIL" if flag is False else "NA")
    ledger = pd.DataFrame([
        ["motility multi-gene core", "4.2 leave-2 + 4.6 mot-only + 4.3 ablation",
         verdict(R.get("loo", {}).get("pass")),
         "core double-sig after leave-2, mot-only Stouffer<-1.5, survives driver ablation"],
        ["kinase set minus CD40", "4.2 residual",
         verdict(R.get("kinase", {}).get("multigene_ok")),
         "residual |Stouffer|>=2 without CD40 else set+footnote only"],
        ["bidirectional down~=up split", "4.1 zcut grid",
         verdict(R.get("sens", {}).get("pass")),
         "down/up ratio in 0.5-2 across 1.64-2.58"],
        ["TRS = same map + minority tail", "4.1 tail grid + 4.4",
         verdict(bool(R.get("trs_sens", {}).get("pass")) and bool(R.get("conc", {}).get("same_map_pass"))),
         "tail<50% & r>=0.75 after dropping hubs+tail"],
        ["adhesion/neurite TRS shift", "4.4 adhesion ablation",
         verdict(R.get("adh", {}).get("shift_is_candidates")),
         "if shift=named list, report genes not module"],
        ["motility-core tight ranking", "4.5 bootstrap",
         verdict(R.get("signstab", {}).get("core_ranking_tight")),
         "core r CI excludes 0 (else negative block, not shared ranking)"],
        ["motility != kinase re-annotation", "4.6 overlap",
         verdict(R.get("overlap", {}).get("motility_independent")),
         "motility-only Stouffer stays negative"],
        ["KCNQ2/KCNN3 split", "single-gene; untestable here", "NA",
         "remains hypothesis; robustness cap ~60"],
        ["GCH1 / pteridine", "4.3 drop GCH1", "NA",
         "metabolism class should vanish without GCH1"],
        ["cascade / psychedelic reversal", "untestable on these CSVs", "NA",
         "leave robustness 15-32; do not add a figure"],
    ], columns=["claim", "test", "verdict", "pass_rule"])
    ledger.to_csv(OUT / "47_robustness_ledger.csv", index=False)
    R["ledger"] = {"table": ledger}
    print(ledger.to_string(index=False))
except Exception:
    warn("4.7 failed:\n" + traceback.format_exc())

# =====================================================================
#                    VERY DETAILED SUMMARY.txt
# =====================================================================
print("\n### Writing SUMMARY.txt ###")
S = []
def w(*a): S.append(" ".join(str(x) for x in a))
def wrap(items, width=90, indent="        "):
    txt = ", ".join(str(i) for i in items)
    return "\n".join(textwrap.wrap(txt, width=width, initial_indent=indent,
                                   subsequent_indent=indent)) or (indent + "(none)")
def fmt(x, nd=3):
    try:
        xf = float(x)
        if np.isnan(xf): return "NA"
        return f"{xf:.{nd}g}" if (abs(xf) < 1e-3 or abs(xf) >= 1e4) else f"{xf:.{nd}f}"
    except Exception:
        return "NA"
def yn(flag):
    return "PASS ✓" if flag is True else ("FAIL ✗" if flag is False else "NA")
bar = "=" * 80

w(bar)
w("091IvD  STAGE 4  —  ROBUSTNESS STRESS TEST SUMMARY")
w("Not a discovery pass. Each block moves a robustness score for a printable claim.")
w(bar)
w(f"Generated   : {TS}")
w(f"Results dir : {RESULTS}")
w(f"Output dir  : {OUT}")
w(f"Cohorts     : Z_trs (TRS)  vs  Z_scz (SCZ-EUR)")
w("")
w("Input tables (rows x cols):")
for k, v in R["loads"].items():
    w(f"  {k:18s} {v[0]:6d} x {v[1]}")
w("")
w("Anchor gene lists validated against the matrix:")
w(f"  motility core present : {len(R['validate']['motility_present'])}/{len(MOTILITY_CORE)}")
w(wrap(R["validate"]["motility_present"]))
w(f"  drivers present       : {len(R['validate']['drivers_present'])}/{len(DRIVERS)}")
w(wrap(R["validate"]["drivers_present"]))
w(f"  TRS candidates present: {len(R['validate']['trs_cand_present'])}/{len(TRS_CAND)}")
w(wrap(R["validate"]["trs_cand_present"]))
w("")

# ---- Section 1 ----
w(bar); w("1. THRESHOLD SENSITIVITY  (does the down≈up bidirectional split survive?)"); w(bar)
SN = R.get("sens", {})
if SN.get("table") is not None:
    t = SN["table"]
    w("  Down/up convergent counts across |Z| cutoffs:")
    w(f"    {'zcut':>6s} {'down':>6s} {'up':>6s} {'ratio':>7s} {'mot_core_in_down':>18s}")
    for _, r in t.iterrows():
        w(f"    {r['zcut']:>6.2f} {int(r['converge_down']):>6d} {int(r['converge_up']):>6d} "
          f"{r['ratio_down_over_up']:>7.2f} {int(r['n_motility_core_in_down']):>18d}")
    w("")
    w(f"  ratio stable across 1.64-2.58 : {SN.get('ratio_stable')}")
    w(f"  motility core stays in down   : {SN.get('motility_stable')}")
    w(f"  VERDICT bidirectional split   : {yn(SN.get('pass'))}")
    w("  Read: if PASS, GO-concordance vs GO-enrichment disagreement is real, not a cutoff artefact.")
else:
    w("  (4.1 not run)")
w("")
TS_ = R.get("trs_sens", {})
if TS_.get("table") is not None:
    t = TS_["table"]
    w("  TRS divergent-tail size across delta cutoffs:")
    w(f"    {'dcut':>6s} {'n_tail':>7s} {'frac':>7s} {'trs_cand_kept':>15s}")
    for _, r in t.iterrows():
        w(f"    {r['delta_cut']:>6.2f} {int(r['n_tail']):>7d} {r['frac_of_genes']:>7.3f} "
          f"{int(r['n_trs_cand_kept']):>15d}")
    w(f"  tail stays minority           : {TS_.get('tail_minority')}")
    w(f"  TRS candidates retained       : {TS_.get('cand_kept')}")
    w(f"  VERDICT 'same map + tail'     : {yn(TS_.get('pass'))}")
w("  Tables: 41_threshold_sensitivity.csv, 41_trs_tail_sensitivity.csv")
w("")

# ---- Section 2 ----
w(bar); w("2. LEAVE-k-OUT ON MOTILITY CORE  (is the 88-robustness claim a 2-gene object?)"); w(bar)
LO = R.get("loo", {})
if LO:
    w(f"  Core genes tested       : {LO.get('n_core')}")
    w(f"  Baseline Stouffer SCZ   : {fmt(LO.get('base_scz'))}")
    w(f"  Baseline Stouffer TRS   : {fmt(LO.get('base_trs'))}")
    w(f"  Leave-2 min double-sig  : {LO.get('k2_min_double')}")
    w(f"  Leave-2 Stouffer<-1.96  : {LO.get('k2_stouffer_survives')}")
    w(f"  VERDICT motility core   : {yn(LO.get('pass'))}")
    w("  Read: PASS => genuine multi-gene negative block. FAIL => write it like GCH1 (single/2-gene).")
else:
    w("  (4.2 not run)")
w("")
KN = R.get("kinase", {})
if KN:
    w("  Kinase (Ser/Thr) set with CD40 removed — the honest robustness of the kinase GO:")
    w(f"    n_kinase={KN.get('n_kinase')}  n_after_CD40={KN.get('n_after_cd40')}")
    w(f"    residual Stouffer SCZ={fmt(KN.get('resid_stouffer_scz'))}  "
      f"TRS={fmt(KN.get('resid_stouffer_trs'))}  n_converge_down={KN.get('n_converge_down')}")
    w(f"    VERDICT multi-gene kinase mechanism : {yn(KN.get('multigene_ok'))}")
    w("    If FAIL: keep the set + CD40 footnote; do NOT call it a multi-gene kinase mechanism.")
w("  Tables: 42_motility_leavek.csv")
w("")

# ---- Section 3 ----
w(bar); w("3. DRIVER ABLATION  (what remains if CD40/GIGYF1/GCH1/CACNB3 vanish)"); w(bar)
AB = R.get("ablation", {})
if AB.get("table") is not None:
    t = AB["table"]
    w(f"  {'subset':>20s} {'n':>5s} {'down':>5s} {'up':>4s} {'div':>4s} {'mot_core_left':>14s}")
    for idx, r in t.iterrows():
        w(f"  {str(idx):>20s} {int(r['n']):>5d} {int(r['down']):>5d} {int(r['up']):>4d} "
          f"{int(r['div']):>4d} {int(r['motility_core_left']):>14d}")
    w("")
    w(f"  motility survives full driver ablation : {AB.get('motility_robust')}")
    w(f"  up-core fragile to driver removal      : {AB.get('up_fragile')}")
    w("  Read: motility_core_left barely moving = robust. up collapsing = 'RTK-as-pathway' is not real.")
    w("  Table: 43_driver_ablation.csv")
else:
    w("  (4.3 not run)")
w("")

# ---- Section 4 ----
w(bar); w("4. CONCORDANCE ABLATION  ('same map, r~0.80' without the genes we interpret)"); w(bar)
CC = R.get("conc", {})
if CC.get("table") is not None:
    t = CC["table"]
    w(f"    {'subset':>22s} {'n':>5s} {'pearson_r':>10s} {'sign_conc':>10s}")
    for _, r in t.iterrows():
        w(f"    {str(r['subset']):>22s} {int(r['n']):>5d} {fmt(r['pearson_r']):>10s} "
          f"{fmt(r['sign_conc']):>10s}")
    w("")
    w(f"  r(all)={fmt(CC.get('r_all'))}  r(no hubs+no tail)={fmt(CC.get('r_no_hubs_no_tail'))}")
    w(f"  VERDICT 'TRS is the same architecture' : {yn(CC.get('same_map_pass'))}")
    w("  Read: PASS => concordance is not a hub artefact and not defined by TRS-specific genes.")
    w("  Table: 44_concordance_ablation.csv")
else:
    w("  (4.4 not run)")
w("")
AD = R.get("adh", {})
if AD.get("table") is not None:
    t = AD["table"]
    w("  Adhesion/neurite ΔZ carried by named candidates?")
    for _, r in t.iterrows():
        w(f"    {str(r['theme']):16s} cand_mean_delta={fmt(r['cand_mean_delta'])} "
          f"(n={int(r['n_cand'])})  rest_module_mean_delta={fmt(r['rest_mean_delta'])} "
          f"(n={int(r['n_rest_module'])})")
    w(f"  VERDICT adhesion shift = named gene list (not module) : {yn(AD.get('shift_is_candidates'))}")
    w("  If PASS: report AMIGO1/CNTN1/CDH10/ST8SIA2/ITM2B as a list, not an adhesion-module TRS effect.")
    w("  Table: 44_adhesion_ablation.csv")
w("")

# ---- Section 5 ----
w(bar); w("5. MOTILITY-CORE SIGN STABILITY + BOOTSTRAP r (cohort-choice surrogate)"); w(bar)
SS = R.get("signstab", {})
if SS:
    w(f"  n_core                 : {SS.get('n_core')}")
    w(f"  all genes same sign    : {SS.get('all_same_sign')}")
    w(f"  min |Z| in either cohort: {fmt(SS.get('min_abs'))}")
    w(f"  core r bootstrap 95%   : [{fmt(SS.get('boot_ci_low'))}, {fmt(SS.get('boot_ci_high'))}] "
      f"(median {fmt(SS.get('boot_med'))})")
    w(f"  VERDICT tight shared ranking inside core : {yn(SS.get('core_ranking_tight'))}")
    w("  Read: CI excluding 0 => shared ranking. CI spanning 0 => negative block but not tight ranking")
    w("        (still supports the GO claim; weakens 'shared architecture' INSIDE motility).")
    w("  Table: 45_motility_sign_stability.csv")
else:
    w("  (4.5 not run)")
w("")

# ---- Section 6 ----
w(bar); w("6. SET-COMPOSITION CONTROL  (motility != kinase genes re-annotated)"); w(bar)
OV = R.get("overlap", {})
if OV:
    w(f"  n motility={OV.get('n_motility')}  n kinase={OV.get('n_kinase')}  overlap={OV.get('n_overlap')}")
    w(f"  motility-ONLY Stouffer SCZ={fmt(OV.get('motility_only_scz'))}  "
      f"TRS={fmt(OV.get('motility_only_trs'))}")
    w(f"  VERDICT motility independent of kinase set : {yn(OV.get('motility_independent'))}")
    w("  Overlap genes that are converge_down (belong in diagram once, not double-counted):")
    w(wrap(OV.get("overlap_down", [])))
    w("  Table: 46_motility_kinase_overlap.csv")
else:
    w("  (4.6 not run — module tags unavailable)")
w("")

# ---- Section 7 ----
w(bar); w("7. ROBUSTNESS LEDGER  (supplement one-pager)"); w(bar)
LD = R.get("ledger", {})
if LD.get("table") is not None:
    for _, r in LD["table"].iterrows():
        w(f"  • {r['claim']}")
        w(f"      test    : {r['test']}")
        w(f"      verdict : {r['verdict']}")
        w(f"      rule    : {r['pass_rule']}")
    w("  Table: 47_robustness_ledger.csv")
else:
    w("  (4.7 not run)")
w("")

# ---- Section 8 ----
w(bar); w("8. WHAT STAGE 4 CANNOT MOVE (leave scores low on purpose)"); w(bar)
w("  These are NOT testable on the pipeline CSVs and must NOT get a new figure here:")
w("    - enhancer-to-gene linkage (needs enhancer-promoter maps)")
w("    - cell-type attribution (needs single-cell / sorted expression)")
w("    - psychedelic direction / therapeutic reversal (needs directional expression)")
w("    - 'TRS mechanism' vs chronicity/drug-exposure/ascertainment confounds")
w("  Anything requiring a 3rd GWAS, MAGMA gene-size/constraint covariates, or")
w("  enhancer-promoter maps is STAGE 5, not Stage 4.")
w("")

# ---- Section 9: manuscript sentences ----
w(bar); w("9. FOUR SENTENCES TO WRITE (each tied to a table)"); w(bar)
w("  1) Motility core remains a double-cohort negative block after leave-2-out (42),")
w("     after removing kinase-overlapping genes (46), and after deleting drivers (43).")
w("  2) Down-core and up-core stay co-equal across Z cutoffs (41) — why GO concordance")
w("     and GO enrichment disagree.")
w("  3) Cross-cohort r stays high after removing hubs and the divergent tail (44);")
w("     TRS is not defined by the genes that differ.")
w("  4) The adhesion/neurite TRS shift shrinks to a named gene list if it vanishes")
w("     after dropping AMIGO1/CNTN1/CDH10/ST8SIA2/ITM2B (44). KCNQ2 vs KCNN3 not promoted.")
w("")

# ---- warnings ----
if R["warnings"]:
    w(bar); w("10. RUN WARNINGS / NOTES"); w(bar)
    for m in R["warnings"]:
        for ln in str(m).splitlines():
            w("  " + ln)
    w("")

# ---- file inventory ----
w(bar); w("11. OUTPUT FILES"); w(bar)
for f in sorted(glob.glob(str(OUT / "**" / "*"), recursive=True)):
    if os.path.isfile(f):
        w(f"  {f}  ({os.path.getsize(f)//1024} KB)")
w(""); w(bar); w("END OF STAGE 4 SUMMARY"); w(bar)

with open(OUT / "SUMMARY.txt", "w") as fh:
    fh.write("\n".join(S) + "\n")

print("\n=== STAGE 4 DONE ===")
print("Summary:", (OUT / "SUMMARY.txt").resolve())
print("\n----- SUMMARY.txt preview (first 90 lines) -----")
print("\n".join(S[:90]))

>>> Loaded: {'31_matrix': (702, 15), '33_down': (53, 16), '33_up': (54, 16), '33_div': (97, 16), '36_anchor': (81, 17), '35_modconc': (5, 7), '34_hubs': (30, 15), 'gene_influence': (1909, 13)}
>>> gene col='Gene'  cohorts: Z_trs / Z_scz

### 4.1 Threshold sensitivity ###
 zcut  n_genes  converge_down  converge_up  ratio_down_over_up  n_motility_core_in_down
 1.00      702            101          109               0.927                       10
 1.64      702             65           65               1.000                       10
 1.96      702             53           54               0.981                       10
 2.58      702             36           37               0.973                        7
 3.00      702             32           22               1.455                        7
    bidirectional-split PASS=True
 delta_cut  n_tail  frac_of_genes  n_trs_cand_kept
       1.0     247         0.3519                7
       1.5     171         0.2436                7
       2.0   

In [ ]:
!cp "/content/091IvD_results" "/content/drive/MyDrive/Dr Uccello/00_Studies/091_Psychedelics" -r

# The End